# E-HEDO + X-HVSC on native Mamba-2: reproducible Flickr8k pipeline

Implements the proposal *Hamiltonian-Inspired Energy Dissipation and Chunk-wise Variational State Coupling for
Efficient Multimodal State Space Models*. `hedo_native/METHOD.md` has the formulation, **Theorem 1 (energy
non-increase)** with proof, the complexity analysis, and the **H1-H4 criteria fixed before any run**.

This notebook only orchestrates. All methodology is in version-controlled files under `Native mamba notebook/hedo_native/`.
Every artifact records the code hash, git commit, config hash, and dataset/feature checksums. A run counts only if
`run_summary.json` says `COMPLETED`: finite training, checkpoint reload reproduces validation, deterministic inference
(including exchange scores), and an independent evaluator reproduces the test metrics (including re-ranking).

| Step | Cell | Pass condition |
|---|---|---|
| Environment, code checkout, unit tests | 0-2 | `NATIVE_MAMBA2_CHECK: PASS`, tests ok |
| Data + frozen features (+ ViT saliency) | 3-4 | `DATASET_READY`, `FEATURE_CACHE` |
| Gate: Theorem 1 numerics, exchange no-op at init, all variants | 5 | `GATE: PASS` |
| Sanity: the proposed model learns (2 epochs) | 6 | val MR well above chance (~0.53) |
| Proposed 2x2 factorial x seeds | 7 | all `COMPLETED` |
| Ablations and controls | 8 | all `COMPLETED` |
| H1/H2 probes | 9 | `PROBES: PASS` |
| H3 corruption benchmark | 10 | `ROBUSTNESS_EVAL: PASS` |
| H4 scaling + end-to-end efficiency | 11 | `SCALING`, `EFFICIENCY_PROFILE` |
| Tables, statistics, hypothesis verdicts | 12 | `ANALYSIS`, `HYPOTHESES` |
| Bundle | 13 | |

Verdicts come from the fixed criteria. "Not supported" is a valid, reportable outcome.

In [ ]:
# CELL 0: CONFIGURATION (the only cell you should edit)
import os, subprocess, sys, time, json
import pandas as pd
from pathlib import Path

USE_DRIVE = True                     # keep data/features/runs on Google Drive so a disconnect loses nothing
QUICK_TEST = False                   # True: every suite with 1 seed and 2 epochs (pipeline check, NOT paper results)
PKG = Path("/content/hedo_native")   # the method code is embedded in cell 2 of this notebook
NATIVE_PYTHON = "/content/mamba312/bin/python"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/hedo_work")
else:
    WORK = Path("/content/hedo_work")

CORE_SEEDS = [42, 43, 44, 45, 46]    # 5 seeds: n=3 gives df=2 and no usable Wilcoxon test
ABLATION_SEEDS = [42, 43, 44, 45, 46]
EPOCHS = 10
if QUICK_TEST:
    CORE_SEEDS, ABLATION_SEEDS, EPOCHS = [42], [42], 2

os.environ["HEDO_WORK"] = str(WORK)
os.environ["PYTHONUNBUFFERED"] = "1"
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "logs").mkdir(exist_ok=True)


def run(cmd, log_name, cwd=None, check=True):
    # Stream a native subprocess live and tee it to WORK/logs/<log_name>.log
    log_path = WORK / "logs" / f"{log_name}.log"
    print("$", " ".join(map(str, cmd)), flush=True)
    with open(log_path, "w", encoding="utf-8") as log:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=os.environ.copy())
        for line in p.stdout:
            print(line, end="", flush=True)
            log.write(line)
        rc = p.wait()
    print(f"[exit {rc}] log: {log_path}", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"{log_name} failed with exit code {rc}; see {log_path}")
    return rc


print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# CELL 1: NATIVE PYTHON 3.12 ENVIRONMENT (pinned)
import shutil

TORCH = ["torch==2.9.0", "torchvision==0.24.0"]
PINNED = ["numpy==2.1.3", "pandas==2.2.3", "scipy==1.14.1", "matplotlib==3.9.2", "Pillow==11.0.0",
          "transformers==4.56.2", "einops==0.8.1", "packaging", "ninja", "pytest"]
WHEELS = [
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/"
    "causal_conv1d-1.6.2.post1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    "https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/"
    "mamba_ssm-2.3.2.post1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
]

if not Path(NATIVE_PYTHON).is_file():
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], "install_uv")
    run(["uv", "venv", "--python", "3.12", "/content/mamba312"], "venv")
    run(["uv", "pip", "install", "--python", NATIVE_PYTHON, "pip"], "install_pip")
    run(["uv", "pip", "install", "--python", NATIVE_PYTHON, *TORCH,
         "--index-url", "https://download.pytorch.org/whl/cu128"], "install_torch")
    run(["uv", "pip", "install", "--python", NATIVE_PYTHON, *PINNED], "install_pinned")
    run(["uv", "pip", "install", "--python", NATIVE_PYTHON, "--no-deps", *WHEELS], "install_mamba")

check = r'''
import json, sys, torch
from mamba_ssm import Mamba2
assert torch.cuda.is_available()
assert Mamba2.__module__ == "mamba_ssm.modules.mamba2", Mamba2.__module__
m = Mamba2(d_model=128, d_state=64, d_conv=4, expand=2, headdim=64).cuda()
x = torch.randn(2, 64, 128, device="cuda", requires_grad=True)
y = m(x); y.sum().backward()
assert y.shape == x.shape and torch.isfinite(y).all() and torch.isfinite(x.grad).all()
print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.get_device_name(0),
      torch.cuda.get_device_capability(0))
print("NATIVE_MAMBA2_CHECK: PASS")
'''
run([NATIVE_PYTHON, "-c", check], "native_check")
freeze = subprocess.run([NATIVE_PYTHON, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
json.dump({"pip_freeze": freeze, "pinned": TORCH + PINNED, "wheels": WHEELS},
          open(WORK / "environment_install.json", "w"), indent=2)

In [ ]:
# CELL 2: WRITE THE EMBEDDED METHOD CODE (verified by SHA256) + CPU UNIT TESTS
# Files are written verbatim; any byte difference aborts. Nothing is patched at run time.
import base64, hashlib, shutil
BUNDLE_SHA256 = "51cf7284ed94f6476b29edbb321ef99c19e8fad742dec0ad43919da3a9e7f05e"
EMBEDDED = {"analyze.py": ["26cc4a158edba46e05fa4ffa429b133afd021ff0a9650d41118b971716c981ee", "IiIiQWdncmVnYXRlIGNvbXBsZXRlZCBydW5zIGludG8gdGFibGVzLCBmaWd1cmVzIGFuZCBzdGF0aXN0aWNzLgoKVXNlcyBvbmx5IHJ1bnMgdGhhdCBhcmUgQ09NUExFVEVELCBwcm9kdWNlZCBieSB0aGUgY3VycmVudCBjb2RlLCBhbmQgd2hvc2UKaW5kZXBlbmRlbnQgZXZhbHVhdGlvbiBwYXNzZWQuIEV2ZXJ5dGhpbmcgaXMgcmVnZW5lcmF0ZWQgZnJvbSByYXcgcnVuCmFydGlmYWN0czsgbm90aGluZyBpcyB0eXBlZCBpbiBieSBoYW5kLgoKU3RhdGlzdGljcyAoc2VlIHN0YXRpc3RpY2FsX2FuYWx5c2lzLmpzb24pOgogICogcGVyLXNlZWQgdmFsdWVzLCBtZWFuICstIFNEIGFjcm9zcyBzZWVkcwogICogcGFpcmVkIGRpZmZlcmVuY2VzIG9uIGNvbW1vbiBzZWVkczogbWVhbiwgU0QsIGRfeiAoPSBtZWFuIGRpZmYgLyBTRCBkaWZmKQogICogcGFpcmVkIHQtdGVzdCBvbmx5IGZvciBuID49IDMgYW5kIGxhYmVsbGVkIGluZGljYXRpdmU7IEhvbG0tYWRqdXN0ZWQgYWNyb3NzIHRoZSBmYW1pbHkKICAqIFdpbGNveG9uIHNpZ25lZC1yYW5rIG9ubHkgZm9yIG4gPj0gNiAod2l0aCBuIDw9IDUgaXQgY2Fubm90IHJlYWNoIHAgPCAwLjA1KQogICogcGFpcmVkIGJvb3RzdHJhcCBvdmVyIHRlc3QgKmltYWdlcyogKHRoZWlyIDUgY2FwdGlvbnMgbW92ZSB3aXRoIHRoZW0pOiB0ZXN0LXNldAogICAgc2FtcGxpbmcgdW5jZXJ0YWludHksIGNvbXBsZW1lbnRhcnkgdG8gc2VlZCB2YXJpYW5jZQoKVXNhZ2U6IHB5dGhvbiBhbmFseXplLnB5IFstLWJvb3RzdHJhcCAyMDAwXQoiIiIKCmltcG9ydCBhcmdwYXJzZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBtYXRwbG90bGliCm1hdHBsb3RsaWIudXNlKCJBZ2ciKQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2NpcHkgaW1wb3J0IHN0YXRzCgpmcm9tIGNvbW1vbiBpbXBvcnQgUkVQT1JUX0RJUiwgUlVOU19ESVIsIGJhbm5lciwgY29kZV9zaGEyNTYsIHByb3ZlbmFuY2UsIHJlYWRfanNvbiwgd3JpdGVfanNvbgpmcm9tIG1ldHJpY3MgaW1wb3J0IENBUFRJT05TX1BFUl9JTUFHRSwgUkVDQUxMX0tTLCByZXJhbmtlZF9yYW5rcywgcmV0cmlldmFsX3JhbmtzCgpNRVRSSUNTID0gWyJpMnRfcjEiLCAiaTJ0X3I1IiwgImkydF9yMTAiLCAidDJpX3IxIiwgInQyaV9yNSIsICJ0MmlfcjEwIiwgIm1lYW5fcmVjYWxsIiwgImR1YWxfbWVhbl9yZWNhbGwiXQpDT1JFID0gWyJiYXNlbGluZSIsICJoZWRvX2VuZXJneSIsICJodnNjX3giLCAiZnVsbF94Il0KTEVHQUNZX0NPUkUgPSBbImJhc2VsaW5lIiwgImhlZG8iLCAiaHZzYyIsICJmdWxsIl0KTEFCRUxTID0gewogICAgImJhc2VsaW5lIjogIk1hbWJhLTIiLCAiaGVkbyI6ICJNYW1iYS0yICsgSEVETyIsICJodnNjIjogIk1hbWJhLTIgKyBIVlNDIiwgImZ1bGwiOiAiTWFtYmEtMiArIEhFRE8gKyBIVlNDIiwKICAgICJmdWxsX2xpbmVhcl9vcGVyYXRvciI6ICJGdWxsLCBIRURPICRcXHJpZ2h0YXJyb3ckIGxpbmVhciBzdGFjayIsCiAgICAiZnVsbF9odnNjX2RldGVybWluaXN0aWMiOiAiRnVsbCwgSFZTQyB3L28gc2FtcGxpbmcvS0wiLAogICAgImJhc2VsaW5lX3BhcmFtX21hdGNoZWQiOiAiTWFtYmEtMiwgcGFyYW0tbWF0Y2hlZCBoZWFkIiwKICAgICJub19taXhlcl9tZWFucG9vbCI6ICJObyBtaXhlciAobWVhbi1wb29sKSIsCiAgICAidHJhbnNmb3JtZXJfcGFyYW1fbWF0Y2hlZCI6ICJUcmFuc2Zvcm1lciwgcGFyYW0tbWF0Y2hlZCIsCiAgICAiaGVkb19lbmVyZ3kiOiAiTWFtYmEtMiArIEUtSEVETyIsICJodnNjX3giOiAiTWFtYmEtMiArIFgtSFZTQyIsICJmdWxsX3giOiAiTWFtYmEtMiArIEUtSEVETyArIFgtSFZTQyAob3VycykiLAogICAgImZ1bGxfeF9hZmZpbmUiOiAiT3VycywgRS1IRURPICRcXHJpZ2h0YXJyb3ckIGFmZmluZSBIRURPIiwgImZ1bGxfeF9jb25zdGRhbXAiOiAiT3VycywgY29uc3RhbnQgZGFtcGluZyIsCiAgICAiZnVsbF94X25vZXhjaGFuZ2UiOiAiT3Vycywgbm8gZXhjaGFuZ2UgKGJvdHRsZW5lY2sgb25seSkiLCAiZnVsbF94X25vcHJpb3IiOiAiT3Vycywgbm8gcHJpb3IgS0wiLAogICAgImJhc2VsaW5lX3BhcmFtX21hdGNoZWRfeCI6ICJNYW1iYS0yLCBwYXJhbS1tYXRjaGVkIHRvIG91cnMiLCAidHJhbnNmb3JtZXJfeCI6ICJPdXJzIHdpdGggVHJhbnNmb3JtZXIgbWl4ZXIiLAp9CkNPTVBBUklTT05TID0gWwogICAgKCJmdWxsX3giLCAiYmFzZWxpbmUiKSwgKCJoZWRvX2VuZXJneSIsICJiYXNlbGluZSIpLCAoImh2c2NfeCIsICJiYXNlbGluZSIpLCAoImZ1bGxfeCIsICJoZWRvX2VuZXJneSIpLAogICAgKCJmdWxsX3giLCAiaHZzY194IiksICgiZnVsbF94IiwgImZ1bGxfeF9hZmZpbmUiKSwgKCJmdWxsX3giLCAiZnVsbF94X2NvbnN0ZGFtcCIpLCAoImZ1bGxfeCIsICJmdWxsX3hfbm9leGNoYW5nZSIpLAogICAgKCJmdWxsX3giLCAiZnVsbF94X25vcHJpb3IiKSwgKCJmdWxsX3giLCAiYmFzZWxpbmVfcGFyYW1fbWF0Y2hlZF94IiksICgiZnVsbF94IiwgInRyYW5zZm9ybWVyX3giKSwKICAgICgiZnVsbF94IiwgImZ1bGwiKSwKICAgICgiZnVsbCIsICJiYXNlbGluZSIpLCAoImhlZG8iLCAiYmFzZWxpbmUiKSwgKCJodnNjIiwgImJhc2VsaW5lIiksICgiZnVsbCIsICJoZWRvIiksICgiZnVsbCIsICJodnNjIiksCiAgICAoImZ1bGwiLCAiZnVsbF9saW5lYXJfb3BlcmF0b3IiKSwgKCJmdWxsIiwgImZ1bGxfaHZzY19kZXRlcm1pbmlzdGljIiksICgiZnVsbCIsICJiYXNlbGluZV9wYXJhbV9tYXRjaGVkIiksCiAgICAoImJhc2VsaW5lIiwgIm5vX21peGVyX21lYW5wb29sIiksICgiYmFzZWxpbmUiLCAidHJhbnNmb3JtZXJfcGFyYW1fbWF0Y2hlZCIpLApdCgoKZGVmIGtleShyb3cpOgogICAgcmV0dXJuIGYie3Jvd1sndmFyaWFudCddfXxrbHtyb3dbJ2tsX3dlaWdodCddOmd9fGN7aW50KHJvd1snaHZzY19jaHVua19zaXplJ10pfSIKCgpkZWYgbG9hZF9ydW5zKGFsbG93X3N0YWxlKToKICAgIGRmID0gcGQucmVhZF9jc3YoUlVOU19ESVIucGFyZW50IC8gInJlc3VsdHMiIC8gImFsbF9ydW5zLmNzdiIpCiAgICBkZiA9IGRmW2RmWyJzdGF0dXMiXSA9PSAiQ09NUExFVEVEIl0uY29weSgpCiAgICBpZiBub3QgYWxsb3dfc3RhbGU6CiAgICAgICAgZGYgPSBkZltkZlsiY3VycmVudF9jb2RlIl0gPT0gVHJ1ZV0gICMgbm9xYTogRTcxMgogICAga2VlcCA9IFtdCiAgICBmb3IgXywgciBpbiBkZi5pdGVycm93cygpOgogICAgICAgIGluZCA9IFJVTlNfRElSIC8gclsicnVuIl0gLyAiaW5kZXBlbmRlbnRfZXZhbC5qc29uIgogICAgICAgIGtlZXAuYXBwZW5kKGluZC5pc19maWxlKCkgYW5kIHJlYWRfanNvbihpbmQpWyJwYXNzIl0pCiAgICBkZiA9IGRmW2tlZXBdLmNvcHkoKQogICAgZGZbImdyb3VwIl0gPSBkZi5hcHBseShrZXksIGF4aXM9MSkKICAgIHJldHVybiBkZgoKCmRlZiBkZWZhdWx0X2dyb3VwKHZhcmlhbnQpOgogICAgcmV0dXJuIGYie3ZhcmlhbnR9fGtsMC4wMDAxfGMxNiIKCgpkZWYgaG9sbShwdmFscyk6CiAgICBvcmRlciA9IG5wLmFyZ3NvcnQocHZhbHMpCiAgICBhZGogPSBucC5lbXB0eShsZW4ocHZhbHMpKQogICAgcnVubmluZyA9IDAuMAogICAgZm9yIHJhbmssIGlkeCBpbiBlbnVtZXJhdGUob3JkZXIpOgogICAgICAgIHJ1bm5pbmcgPSBtYXgocnVubmluZywgbWluKDEuMCwgKGxlbihwdmFscykgLSByYW5rKSAqIHB2YWxzW2lkeF0pKQogICAgICAgIGFkaltpZHhdID0gcnVubmluZwogICAgcmV0dXJuIGFkagoKCmRlZiBtZWFuX3JlY2FsbF9mcm9tX3JhbmtzKGkydCwgdDJpKToKICAgIHJldHVybiBucC5tZWFuKFtucC5tZWFuKHIgPCBrKSAqIDEwMCBmb3IgciBpbiAoaTJ0LCB0MmkpIGZvciBrIGluIFJFQ0FMTF9LU10pCgoKZGVmIHBhaXJlZF9zdGF0cyhkZiwgYSwgYiwgbl9ib290LCBybmdfc2VlZD0wKToKICAgIEEgPSBkZltkZlsiZ3JvdXAiXSA9PSBhXS5zZXRfaW5kZXgoInNlZWQiKQogICAgQiA9IGRmW2RmWyJncm91cCJdID09IGJdLnNldF9pbmRleCgic2VlZCIpCiAgICBzZWVkcyA9IHNvcnRlZChzZXQoQS5pbmRleCkgJiBzZXQoQi5pbmRleCkpCiAgICByZXMgPSB7ImEiOiBhLCAiYiI6IGIsICJzZWVkcyI6IHNlZWRzLCAibiI6IGxlbihzZWVkcyl9CiAgICBpZiBub3Qgc2VlZHM6CiAgICAgICAgcmV0dXJuIHJlcwogICAgZCA9IEEubG9jW3NlZWRzLCAibWVhbl9yZWNhbGwiXS52YWx1ZXMgLSBCLmxvY1tzZWVkcywgIm1lYW5fcmVjYWxsIl0udmFsdWVzCiAgICByZXMudXBkYXRlKG1lYW5fZGlmZj1mbG9hdChkLm1lYW4oKSksIHNkX2RpZmY9ZmxvYXQoZC5zdGQoZGRvZj0xKSkgaWYgbGVuKGQpID4gMSBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgIHBlcl9zZWVkX2RpZmY9ZGljdCh6aXAobWFwKGludCwgc2VlZHMpLCBtYXAoZmxvYXQsIGQpKSkpCiAgICBpZiBsZW4oZCkgPiAxIGFuZCBkLnN0ZChkZG9mPTEpID4gMDoKICAgICAgICByZXNbImRfeiJdID0gZmxvYXQoZC5tZWFuKCkgLyBkLnN0ZChkZG9mPTEpKQogICAgaWYgbGVuKGQpID49IDM6CiAgICAgICAgcmVzWyJ0X3Rlc3RfcCJdID0gZmxvYXQoc3RhdHMudHRlc3RfcmVsKEEubG9jW3NlZWRzLCAibWVhbl9yZWNhbGwiXSwgQi5sb2Nbc2VlZHMsICJtZWFuX3JlY2FsbCJdKS5wdmFsdWUpCiAgICAgICAgcmVzWyJ0X3Rlc3Rfbm90ZSJdID0gZiJpbmRpY2F0aXZlIG9ubHk6IG49e2xlbihkKX0gc2VlZHMsIGRmPXtsZW4oZCkgLSAxfSIKICAgIGlmIGxlbihkKSA+PSA2OgogICAgICAgIHJlc1sid2lsY294b25fcCJdID0gZmxvYXQoc3RhdHMud2lsY294b24oZCkucHZhbHVlKQoKICAgIGlmIG5fYm9vdDoKICAgICAgICAjIHBhaXJlZCBib290c3RyYXAgb3ZlciB0ZXN0IGltYWdlcywgc2FtZSByZXNhbXBsZXMgZm9yIGV2ZXJ5IHNlZWQgYW5kIGJvdGggZ3JvdXBzCiAgICAgICAgcmFua3MgPSB7fQogICAgICAgIGZvciBzIGluIHNlZWRzOgogICAgICAgICAgICBmb3IgZywgZnJhbWUgaW4gKCgiYSIsIEEpLCAoImIiLCBCKSk6CiAgICAgICAgICAgICAgICBydW4gPSBSVU5TX0RJUiAvIGZyYW1lLmxvY1tzLCAicnVuIl0KICAgICAgICAgICAgICAgIGVtYiA9IChucC5sb2FkKHJ1biAvICJ0ZXN0X2ltYWdlX2VtYmVkZGluZ3MubnB5IiksIG5wLmxvYWQocnVuIC8gInRlc3RfdGV4dF9lbWJlZGRpbmdzLm5weSIpKQogICAgICAgICAgICAgICAgcnJfcGF0aCA9IHJ1biAvICJ0ZXN0X3JlcmFuay5ucHoiCiAgICAgICAgICAgICAgICByYW5rc1soZywgcyldID0gKHJlcmFua2VkX3JhbmtzKCplbWIsIGRpY3QobnAubG9hZChycl9wYXRoKSkpIGlmIHJyX3BhdGguaXNfZmlsZSgpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVsc2UgcmV0cmlldmFsX3JhbmtzKCplbWIpKQogICAgICAgIG5faW1nID0gbGVuKHJhbmtzWygiYSIsIHNlZWRzWzBdKV1bMF0pCiAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHJuZ19zZWVkKQogICAgICAgIGRpZmZzID0gbnAuZW1wdHkobl9ib290KQogICAgICAgIGZvciBpIGluIHJhbmdlKG5fYm9vdCk6CiAgICAgICAgICAgIGlkeCA9IHJuZy5pbnRlZ2VycygwLCBuX2ltZywgbl9pbWcpCiAgICAgICAgICAgIGNhcCA9IChpZHhbOiwgTm9uZV0gKiBDQVBUSU9OU19QRVJfSU1BR0UgKyBucC5hcmFuZ2UoQ0FQVElPTlNfUEVSX0lNQUdFKSkucmVzaGFwZSgtMSkKICAgICAgICAgICAgcGVyX3NlZWQgPSBbbWVhbl9yZWNhbGxfZnJvbV9yYW5rcyhyYW5rc1soImEiLCBzKV1bMF1baWR4XSwgcmFua3NbKCJhIiwgcyldWzFdW2NhcF0pCiAgICAgICAgICAgICAgICAgICAgICAgIC0gbWVhbl9yZWNhbGxfZnJvbV9yYW5rcyhyYW5rc1soImIiLCBzKV1bMF1baWR4XSwgcmFua3NbKCJiIiwgcyldWzFdW2NhcF0pIGZvciBzIGluIHNlZWRzXQogICAgICAgICAgICBkaWZmc1tpXSA9IG5wLm1lYW4ocGVyX3NlZWQpCiAgICAgICAgcmVzWyJib290c3RyYXBfdGVzdF9pbWFnZXMiXSA9IHsibl9yZXNhbXBsZXMiOiBuX2Jvb3QsICJtZWFuX2RpZmYiOiBmbG9hdChkaWZmcy5tZWFuKCkpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImNpOTUiOiBbZmxvYXQobnAucGVyY2VudGlsZShkaWZmcywgMi41KSksIGZsb2F0KG5wLnBlcmNlbnRpbGUoZGlmZnMsIDk3LjUpKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZnJhY19sZV8wIjogZmxvYXQobnAubWVhbihkaWZmcyA8PSAwKSl9CiAgICByZXR1cm4gcmVzCgoKZGVmIGZtdChtZWFuLCBzZCk6CiAgICByZXR1cm4gZiJ7bWVhbjouMmZ9ICRcXHBtJCB7c2Q6LjJmfSIgaWYgbm90IG5wLmlzbmFuKHNkKSBlbHNlIGYie21lYW46LjJmfSIKCgpkZWYgbGF0ZXhfdGFibGUoc3VtbWFyeSwgZ3JvdXBzLCBjYXB0aW9uLCBsYWJlbCk6CiAgICBsaW5lcyA9IFsiXFxiZWdpbnt0YWJsZX1bdF0iLCAiXFxjZW50ZXJpbmciLCBmIlxcY2FwdGlvbnt7e2NhcHRpb259fX0iLCBmIlxcbGFiZWx7e3tsYWJlbH19fSIsCiAgICAgICAgICAgICAiXFxyZXNpemVib3h7XFxsaW5ld2lkdGh9eyF9eyUiLCAiXFxiZWdpbnt0YWJ1bGFyfXtsY2NjY2NjY2NjfSIsICJcXGhsaW5lIiwKICAgICAgICAgICAgICJNb2RlbCAmICRuJCAmIFBhcmFtcyAmIEkyVCBSQDEgJiBJMlQgUkA1ICYgSTJUIFJAMTAgJiBUMkkgUkAxICYgVDJJIFJANSAmIFQySSBSQDEwICYgTVIgXFxcXCIsCiAgICAgICAgICAgICAiXFxobGluZSJdCiAgICBmb3IgZyBpbiBncm91cHM6CiAgICAgICAgaWYgZyBub3QgaW4gc3VtbWFyeS5pbmRleDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICByID0gc3VtbWFyeS5sb2NbZ10KICAgICAgICB2YXJpYW50LCBrbCwgY2h1bmsgPSBnLnNwbGl0KCJ8IikKICAgICAgICBuYW1lID0gTEFCRUxTLmdldCh2YXJpYW50LCB2YXJpYW50KQogICAgICAgIGlmIGtsICE9ICJrbDAuMDAwMSIgb3IgY2h1bmsgIT0gImMxNiI6CiAgICAgICAgICAgIG5hbWUgKz0gZiIgKHtrbC5yZXBsYWNlKCdrbCcsICdLTD0nKX0sIHtjaHVuay5yZXBsYWNlKCdjJywgJ2NodW5rPScpfSkiCiAgICAgICAgY2VsbHMgPSBbZm10KHJbKG0sICJtZWFuIildLCByWyhtLCAic3RkIildKSBmb3IgbSBpbiBNRVRSSUNTWzo3XV0KICAgICAgICBsaW5lcy5hcHBlbmQoZiJ7bmFtZX0gJiB7aW50KHJbKCdzZWVkJywgJ2NvdW50JyldKX0gJiB7aW50KHJbKCd0cmFpbmFibGVfcGFyYW1zJywgJ21lYW4nKV0pOix9ICYgIgogICAgICAgICAgICAgICAgICAgICArICIgJiAiLmpvaW4oY2VsbHMpICsgIiBcXFxcIikKICAgIGxpbmVzICs9IFsiXFxobGluZSIsICJcXGVuZHt0YWJ1bGFyfX0iLCAiXFxlbmR7dGFibGV9Il0KICAgIHJldHVybiAiXG4iLmpvaW4obGluZXMpICsgIlxuIgoKCmRlZiBmaWd1cmVzKGRmLCBzdW1tYXJ5LCBvdXQpOgogICAgZ3JvdXBzID0gW2RlZmF1bHRfZ3JvdXAodikgZm9yIHYgaW4gQ09SRSBpZiBkZWZhdWx0X2dyb3VwKHYpIGluIHN1bW1hcnkuaW5kZXhdCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDYsIDMuNikpCiAgICBmb3IgaSwgZyBpbiBlbnVtZXJhdGUoZ3JvdXBzKToKICAgICAgICB2YWxzID0gZGZbZGZbImdyb3VwIl0gPT0gZ11bIm1lYW5fcmVjYWxsIl0udmFsdWVzCiAgICAgICAgYXguYmFyKGksIHZhbHMubWVhbigpLCB5ZXJyPXZhbHMuc3RkKGRkb2Y9MSkgaWYgbGVuKHZhbHMpID4gMSBlbHNlIDAsIGNhcHNpemU9NCwgY29sb3I9IiM0YzcyYjAiLCBhbHBoYT0wLjYpCiAgICAgICAgYXguc2NhdHRlcihucC5mdWxsKGxlbih2YWxzKSwgaSkgKyBucC5saW5zcGFjZSgtMC4xNSwgMC4xNSwgbGVuKHZhbHMpKSwgdmFscywgY29sb3I9ImJsYWNrIiwgcz0xMiwgem9yZGVyPTMpCiAgICBheC5zZXRfeHRpY2tzKHJhbmdlKGxlbihncm91cHMpKSkKICAgIGF4LnNldF94dGlja2xhYmVscyhbTEFCRUxTW2cuc3BsaXQoInwiKVswXV0gZm9yIGcgaW4gZ3JvdXBzXSwgcm90YXRpb249MTUsIGhhPSJyaWdodCIsIGZvbnRzaXplPTgpCiAgICBheC5zZXRfeWxhYmVsKCJUZXN0IG1lYW4gcmVjYWxsICglKSIpCiAgICBheC5zZXRfdGl0bGUoIkZsaWNrcjhrIHRlc3QsIG1lYW4gJFxccG0kIFNEIG92ZXIgc2VlZHMgKGRvdHMgPSBzZWVkcykiLCBmb250c2l6ZT05KQogICAgZmlnLnRpZ2h0X2xheW91dCgpCiAgICBmb3IgZXh0IGluICgicG5nIiwgInBkZiIpOgogICAgICAgIGZpZy5zYXZlZmlnKG91dCAvIGYiZmlnX21lYW5fcmVjYWxsLntleHR9IiwgZHBpPTMwMCkKICAgIHBsdC5jbG9zZShmaWcpCgogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KDEwLCAzLjYpKQogICAgZm9yIGcgaW4gZ3JvdXBzOgogICAgICAgIGhpc3QgPSBbcGQucmVhZF9jc3YoUlVOU19ESVIgLyByIC8gInRyYWluaW5nX2hpc3RvcnkuY3N2IikgZm9yIHIgaW4gZGZbZGZbImdyb3VwIl0gPT0gZ11bInJ1biJdXQogICAgICAgIGZvciBheCwgY29sLCB5bGFiZWwgaW4gKChheGVzWzBdLCAidHJhaW5faW5mb25jZSIsICJUcmFpbiBJbmZvTkNFIChwYXNzIDEpIiksIChheGVzWzFdLCAidmFsX21lYW5fcmVjYWxsIiwgIlZhbCBtZWFuIHJlY2FsbCAoJSkiKSk6CiAgICAgICAgICAgIG0gPSBucC5hcnJheShbaFtjb2xdLnZhbHVlcyBmb3IgaCBpbiBoaXN0XSkKICAgICAgICAgICAgeCA9IGhpc3RbMF1bImVwb2NoIl0udmFsdWVzCiAgICAgICAgICAgIGF4LnBsb3QoeCwgbS5tZWFuKDApLCBsYWJlbD1MQUJFTFNbZy5zcGxpdCgifCIpWzBdXSkKICAgICAgICAgICAgaWYgbGVuKG0pID4gMToKICAgICAgICAgICAgICAgIGF4LmZpbGxfYmV0d2Vlbih4LCBtLm1lYW4oMCkgLSBtLnN0ZCgwLCBkZG9mPTEpLCBtLm1lYW4oMCkgKyBtLnN0ZCgwLCBkZG9mPTEpLCBhbHBoYT0wLjIpCiAgICAgICAgICAgIGF4LnNldF94bGFiZWwoIkVwb2NoIikKICAgICAgICAgICAgYXguc2V0X3lsYWJlbCh5bGFiZWwpCiAgICBheGVzWzFdLmxlZ2VuZChmb250c2l6ZT04KQogICAgZmlnLnRpZ2h0X2xheW91dCgpCiAgICBmb3IgZXh0IGluICgicG5nIiwgInBkZiIpOgogICAgICAgIGZpZy5zYXZlZmlnKG91dCAvIGYiZmlnX3RyYWluaW5nX2N1cnZlcy57ZXh0fSIsIGRwaT0zMDApCiAgICBwbHQuY2xvc2UoZmlnKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ib290c3RyYXAiLCB0eXBlPWludCwgZGVmYXVsdD0yMDAwKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWFsbG93LXN0YWxlLWNvZGUiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQogICAgUkVQT1JUX0RJUi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGYgPSBsb2FkX3J1bnMoYXJncy5hbGxvd19zdGFsZV9jb2RlKQogICAgYmFubmVyKGYiQU5BTFlTSVMgb3ZlciB7bGVuKGRmKX0gdmVyaWZpZWQgcnVucyAoY29kZSB7Y29kZV9zaGEyNTYoKVs6MTJdfSkiKQogICAgaWYgZGYuZW1wdHk6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgibm8gdmVyaWZpZWQgcnVucyB0byBhbmFseXNlIikKICAgIGRmLnNvcnRfdmFsdWVzKFsiZ3JvdXAiLCAic2VlZCJdKS50b19jc3YoUkVQT1JUX0RJUiAvICJwZXJfc2VlZF9yZXN1bHRzLmNzdiIsIGluZGV4PUZhbHNlKQoKICAgIHN1bW1hcnkgPSBkZi5ncm91cGJ5KCJncm91cCIpLmFnZyh7Kip7bTogWyJtZWFuIiwgInN0ZCJdIGZvciBtIGluIE1FVFJJQ1N9LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInNlZWQiOiAiY291bnQiLCAidHJhaW5hYmxlX3BhcmFtcyI6ICJtZWFuIn0pCiAgICBmbGF0ID0gc3VtbWFyeS5jb3B5KCkKICAgIGZsYXQuY29sdW1ucyA9IFsiXyIuam9pbihjKSBmb3IgYyBpbiBmbGF0LmNvbHVtbnNdCiAgICBmbGF0LnRvX2NzdihSRVBPUlRfRElSIC8gInN1bW1hcnlfbWVhbl9zZC5jc3YiKQogICAgcHJpbnQoZmxhdFtbIm1lYW5fcmVjYWxsX21lYW4iLCAibWVhbl9yZWNhbGxfc3RkIiwgInNlZWRfY291bnQiXV0udG9fc3RyaW5nKCkpCgogICAgY29tcGFyaXNvbnMgPSBbKGRlZmF1bHRfZ3JvdXAoYSksIGRlZmF1bHRfZ3JvdXAoYikpIGZvciBhLCBiIGluIENPTVBBUklTT05TXQogICAgY29tcGFyaXNvbnMgKz0gWyhnLCBkZWZhdWx0X2dyb3VwKCJmdWxsIikpIGZvciBnIGluIHN1bW1hcnkuaW5kZXgKICAgICAgICAgICAgICAgICAgICBpZiBnLnN0YXJ0c3dpdGgoImZ1bGx8IikgYW5kIGcgIT0gZGVmYXVsdF9ncm91cCgiZnVsbCIpXQogICAgcmVzdWx0cyA9IFtwYWlyZWRfc3RhdHMoZGYsIGEsIGIsIGFyZ3MuYm9vdHN0cmFwKSBmb3IgYSwgYiBpbiBjb21wYXJpc29ucwogICAgICAgICAgICAgICBpZiBhIGluIHN1bW1hcnkuaW5kZXggYW5kIGIgaW4gc3VtbWFyeS5pbmRleF0KICAgIHRlc3RlZCA9IFtyIGZvciByIGluIHJlc3VsdHMgaWYgInRfdGVzdF9wIiBpbiByXQogICAgaWYgdGVzdGVkOgogICAgICAgIGZvciByLCBwIGluIHppcCh0ZXN0ZWQsIGhvbG0oW3JbInRfdGVzdF9wIl0gZm9yIHIgaW4gdGVzdGVkXSkpOgogICAgICAgICAgICByWyJ0X3Rlc3RfcF9ob2xtIl0gPSBmbG9hdChwKQogICAgd3JpdGVfanNvbihSRVBPUlRfRElSIC8gInN0YXRpc3RpY2FsX2FuYWx5c2lzLmpzb24iLCB7CiAgICAgICAgImNvbXBhcmlzb25zIjogcmVzdWx0cywKICAgICAgICAibm90ZXMiOiBbImRfeiBpcyB0aGUgcGFpcmVkIGVmZmVjdCBzaXplIChtZWFuIGRpZmZlcmVuY2UgLyBTRCBvZiBkaWZmZXJlbmNlcyksIG5vdCBwb29sZWQgQ29oZW4ncyBkLiIsCiAgICAgICAgICAgICAgICAgICJXaXRoIGZldyBzZWVkcywgdC10ZXN0IHAtdmFsdWVzIGFyZSBpbmRpY2F0aXZlIG9ubHk7IGRvIG5vdCBjbGFpbSBzaWduaWZpY2FuY2UgZnJvbSBuPTMuIiwKICAgICAgICAgICAgICAgICAgIkJvb3RzdHJhcCBDSSByZWZsZWN0cyB0ZXN0LXNldCBzYW1wbGluZyBvbmx5LCBhdmVyYWdlZCBvdmVyIG1hdGNoZWQgc2VlZHMuIl0sCiAgICAgICAgInByb3ZlbmFuY2UiOiBwcm92ZW5hbmNlKCl9KQogICAgZm9yIHIgaW4gcmVzdWx0czoKICAgICAgICBib290ID0gci5nZXQoImJvb3RzdHJhcF90ZXN0X2ltYWdlcyIsIHt9KQogICAgICAgIHByaW50KGYie3JbJ2EnXToyOHN9IC0ge3JbJ2InXToyOHN9IG49e3JbJ24nXX0gZGlmZj17ci5nZXQoJ21lYW5fZGlmZicsIGZsb2F0KCduYW4nKSk6Ky4yZn0gIgogICAgICAgICAgICAgIGYiZF96PXtyLmdldCgnZF96JywgZmxvYXQoJ25hbicpKTouMmZ9IHA9e3IuZ2V0KCd0X3Rlc3RfcCcsIGZsb2F0KCduYW4nKSk6LjNmfSAiCiAgICAgICAgICAgICAgZiJwX2hvbG09e3IuZ2V0KCd0X3Rlc3RfcF9ob2xtJywgZmxvYXQoJ25hbicpKTouM2Z9IGJvb3Q5NT17Ym9vdC5nZXQoJ2NpOTUnKX0iKQoKICAgIGNvcmUgPSBbZGVmYXVsdF9ncm91cCh2KSBmb3IgdiBpbiBDT1JFXQogICAgYWJsYXRpb24gPSBbZGVmYXVsdF9ncm91cCh2KSBmb3IgdiBpbiAoImZ1bGxfeCIsICJmdWxsX3hfYWZmaW5lIiwgImZ1bGxfeF9jb25zdGRhbXAiLCAiZnVsbF94X25vZXhjaGFuZ2UiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImZ1bGxfeF9ub3ByaW9yIiwgImJhc2VsaW5lIiwgImJhc2VsaW5lX3BhcmFtX21hdGNoZWRfeCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAidHJhbnNmb3JtZXJfcGFyYW1fbWF0Y2hlZCIsICJ0cmFuc2Zvcm1lcl94IiwgIm5vX21peGVyX21lYW5wb29sIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJmdWxsIiwgImZ1bGxfbGluZWFyX29wZXJhdG9yIiwgImZ1bGxfaHZzY19kZXRlcm1pbmlzdGljIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJiYXNlbGluZV9wYXJhbV9tYXRjaGVkIiwgKkxFR0FDWV9DT1JFWzE6M10pXQogICAgYWJsYXRpb24gKz0gc29ydGVkKGcgZm9yIGcgaW4gc3VtbWFyeS5pbmRleCBpZiBnLnN0YXJ0c3dpdGgoImZ1bGx8IikgYW5kIGcgIT0gZGVmYXVsdF9ncm91cCgiZnVsbCIpKQogICAgKFJFUE9SVF9ESVIgLyAidGFibGVfbWFpbi50ZXgiKS53cml0ZV90ZXh0KGxhdGV4X3RhYmxlKAogICAgICAgIHN1bW1hcnksIGNvcmUsICJGbGlja3I4ayAxSyB0ZXN0IHJldHJpZXZhbCAobWVhbiAkXFxwbSQgU0Qgb3ZlciBzZWVkcykuIFBhcmFtczogdHJhaW5hYmxlIHBhcmFtZXRlcnMuIiwKICAgICAgICAidGFiOm1haW4iKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIChSRVBPUlRfRElSIC8gInRhYmxlX2FibGF0aW9ucy50ZXgiKS53cml0ZV90ZXh0KGxhdGV4X3RhYmxlKAogICAgICAgIHN1bW1hcnksIGFibGF0aW9uLCAiQWJsYXRpb25zIGFuZCBjb250cm9scyBvbiBGbGlja3I4ayAxSyB0ZXN0IChtZWFuICRcXHBtJCBTRCBvdmVyIHNlZWRzKS4iLAogICAgICAgICJ0YWI6YWJsYXRpb25zIiksIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgc3RhdF9saW5lcyA9IFsiXFxiZWdpbnt0YWJsZX1bdF0iLCAiXFxjZW50ZXJpbmciLAogICAgICAgICAgICAgICAgICAiXFxjYXB0aW9ue1BhaXJlZCBjb21wYXJpc29ucyBvZiB0ZXN0IG1lYW4gcmVjYWxsIG9uIG1hdGNoZWQgc2VlZHMuICRkX3okOiBwYWlyZWQgZWZmZWN0IHNpemUuICIKICAgICAgICAgICAgICAgICAgIiRwJDogcGFpcmVkICR0JC10ZXN0LCBIb2xtLWFkanVzdGVkLCBpbmRpY2F0aXZlIG9ubHkuIENJOiA5NVxcJSBib290c3RyYXAgb3ZlciB0ZXN0IGltYWdlcy59IiwKICAgICAgICAgICAgICAgICAgIlxcbGFiZWx7dGFiOnN0YXRzfSIsICJcXHJlc2l6ZWJveHtcXGxpbmV3aWR0aH17IX17JSIsICJcXGJlZ2lue3RhYnVsYXJ9e2xsY2NjY2N9IiwgIlxcaGxpbmUiLAogICAgICAgICAgICAgICAgICAiQSAmIEIgJiAkbiQgJiAkXFxEZWx0YSRNUiAmICRkX3okICYgJHBfe1xcdGV4dHtIb2xtfX0kICYgOTVcXCUgQ0kgXFxcXCIsICJcXGhsaW5lIl0KICAgIGZvciByIGluIHJlc3VsdHM6CiAgICAgICAgaWYgbm90IHJbIm4iXToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBjaSA9IHIuZ2V0KCJib290c3RyYXBfdGVzdF9pbWFnZXMiLCB7fSkuZ2V0KCJjaTk1IikKICAgICAgICBzdGF0X2xpbmVzLmFwcGVuZCgKICAgICAgICAgICAgZiJ7TEFCRUxTLmdldChyWydhJ10uc3BsaXQoJ3wnKVswXSwgclsnYSddKX0gKHtyWydhJ10uc3BsaXQoJ3wnLCAxKVsxXX0pICYgIgogICAgICAgICAgICBmIntMQUJFTFMuZ2V0KHJbJ2InXS5zcGxpdCgnfCcpWzBdLCByWydiJ10pfSAmIHtyWyduJ119ICYge3JbJ21lYW5fZGlmZiddOisuMmZ9ICYgIgogICAgICAgICAgICBmIntyLmdldCgnZF96JywgZmxvYXQoJ25hbicpKTouMmZ9ICYge3IuZ2V0KCd0X3Rlc3RfcF9ob2xtJywgZmxvYXQoJ25hbicpKTouM2Z9ICYgIgogICAgICAgICAgICArIChmIlt7Y2lbMF06Ky4yZn0sIHtjaVsxXTorLjJmfV0iIGlmIGNpIGVsc2UgIi0tIikgKyAiIFxcXFwiKQogICAgc3RhdF9saW5lcyArPSBbIlxcaGxpbmUiLCAiXFxlbmR7dGFidWxhcn19IiwgIlxcZW5ke3RhYmxlfSJdCiAgICAoUkVQT1JUX0RJUiAvICJ0YWJsZV9zdGF0cy50ZXgiKS53cml0ZV90ZXh0KCJcbiIuam9pbihzdGF0X2xpbmVzKS5yZXBsYWNlKCJ8IiwgIiwgIikgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQoKICAgIGZpZ3VyZXMoZGYsIHN1bW1hcnksIFJFUE9SVF9ESVIpCiAgICBwcmludChmInJlcG9ydCB3cml0dGVuIHRvIHtSRVBPUlRfRElSfSIpCiAgICBwcmludCgiQU5BTFlTSVM6IFBBU1MiKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"], "backbones.py": ["3ed49dec5e89a6702cb29f5016283c9eedf98313967e58a3441ac7400d49ae6e", "IiIiRnJvemVuIHByZXRyYWluZWQgYmFja2JvbmVzIChWaVQtQi8xNiBhbmQgUm9CRVJUYS1iYXNlKS4KCkJvdGggYXJlIGtlcHQgaW4gZXZhbCBtb2RlIHdpdGggcmVxdWlyZXNfZ3JhZD1GYWxzZS4gVGhlaXIgb3V0cHV0cyBhcmUgY2FjaGVkCm9uY2UgYnkgZmVhdHVyZXMucHk7IHRyYWluaW5nIG5ldmVyIHJlLXJ1bnMgdGhlbS4KIiIiCgppbXBvcnQgb3MKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNodmlzaW9uLm1vZGVscyBhcyB0dl9tb2RlbHMKClZJVF9XRUlHSFRTID0gdHZfbW9kZWxzLlZpVF9CXzE2X1dlaWdodHMuSU1BR0VORVQxS19WMQpST0JFUlRBX05BTUUgPSAicm9iZXJ0YS1iYXNlIgojIFBpbiBhIEh1Z2dpbmcgRmFjZSBjb21taXQgZm9yIGV4YWN0IHJlcHJvZHVjaWJpbGl0eS4gIm1haW4iIGlzIHJlc29sdmVkIGFuZCB0aGUKIyBjb25jcmV0ZSBjb21taXQgaGFzaCBpcyByZWNvcmRlZCBpbiBmZWF0dXJlX21hbmlmZXN0Lmpzb24uClJPQkVSVEFfUkVWSVNJT04gPSBvcy5lbnZpcm9uLmdldCgiSEVET19ST0JFUlRBX1JFVklTSU9OIiwgIm1haW4iKQpNQVhfVEVYVF9MRU4gPSA2NAoKCmRlZiB2aXRfdHJhbnNmb3JtKCk6CiAgICAiIiJPZmZpY2lhbCBwcmVwcm9jZXNzaW5nIGZvciB0aGUgY2hvc2VuIHdlaWdodHMgKFJlc2l6ZSAyNTYgLT4gQ2VudGVyQ3JvcCAyMjQgLT4gTm9ybWFsaXplKS4iIiIKICAgIHJldHVybiBWSVRfV0VJR0hUUy50cmFuc2Zvcm1zKCkKCgpjbGFzcyBGcm96ZW5WaVQobm4uTW9kdWxlKToKICAgICIiInRvcmNodmlzaW9uIFZpc2lvblRyYW5zZm9ybWVyLmZvcndhcmQgd2l0aG91dCB0aGUgY2xhc3MtdG9rZW4gc2VsZWN0aW9uIGFuZCBoZWFkLgoKICAgIFJldHVybnMgdGhlIDE5NiBwYXRjaCB0b2tlbnMgYWZ0ZXIgdGhlIGVuY29kZXIncyBmaW5hbCBMYXllck5vcm0uIFRoZSBlbmNvZGVyCiAgICBhZGRzIHBvc19lbWJlZGRpbmcgaW50ZXJuYWxseS4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnZpdCA9IHR2X21vZGVscy52aXRfYl8xNih3ZWlnaHRzPVZJVF9XRUlHSFRTKQogICAgICAgIGFzc2VydCBzZWxmLnZpdC5lbmNvZGVyLnBvc19lbWJlZGRpbmcuc2hhcGUgPT0gKDEsIDE5NywgNzY4KQogICAgICAgIGZvciBwIGluIHNlbGYudml0LnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgcC5yZXF1aXJlc19ncmFkID0gRmFsc2UKICAgICAgICBzZWxmLmV2YWwoKQoKICAgIGRlZiB0cmFpbihzZWxmLCBtb2RlPVRydWUpOgogICAgICAgIHJldHVybiBzdXBlcigpLnRyYWluKEZhbHNlKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHggPSBzZWxmLnZpdC5fcHJvY2Vzc19pbnB1dCh4KQogICAgICAgIGNscyA9IHNlbGYudml0LmNsYXNzX3Rva2VuLmV4cGFuZCh4LnNoYXBlWzBdLCAtMSwgLTEpCiAgICAgICAgeCA9IHNlbGYudml0LmVuY29kZXIodG9yY2guY2F0KFtjbHMsIHhdLCBkaW09MSkpCiAgICAgICAgcmV0dXJuIHhbOiwgMTosIDpdCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIGZvcndhcmRfd2l0aF9zYWxpZW5jeShzZWxmLCB4KToKICAgICAgICAiIiJTYW1lIG91dHB1dHMgYXMgZm9yd2FyZCgpLCBwbHVzIGxhc3QtYmxvY2sgY2xhc3MtdG9rZW4gYXR0ZW50aW9uIHRvIHRoZSAxOTYgcGF0Y2hlcyAoaGVhZC1hdmVyYWdlZCkuCgogICAgICAgIFVzZWQgb25seSBhcyBhIGZvcmVncm91bmQvYmFja2dyb3VuZCBwcm94eSBmb3IgdGhlIEgxIHByb2JlOyBuZXZlciBhcyBhIG1vZGVsIGlucHV0LgogICAgICAgICIiIgogICAgICAgIGVuYyA9IHNlbGYudml0LmVuY29kZXIKICAgICAgICB4ID0gc2VsZi52aXQuX3Byb2Nlc3NfaW5wdXQoeCkKICAgICAgICB4ID0gdG9yY2guY2F0KFtzZWxmLnZpdC5jbGFzc190b2tlbi5leHBhbmQoeC5zaGFwZVswXSwgLTEsIC0xKSwgeF0sIGRpbT0xKQogICAgICAgIHggPSBlbmMuZHJvcG91dCh4ICsgZW5jLnBvc19lbWJlZGRpbmcpCiAgICAgICAgZm9yIGxheWVyIGluIGVuYy5sYXllcnNbOi0xXToKICAgICAgICAgICAgeCA9IGxheWVyKHgpCiAgICAgICAgbGFzdCA9IGVuYy5sYXllcnNbLTFdCiAgICAgICAgeSA9IGxhc3QubG5fMSh4KQogICAgICAgIF8sIGF0dG4gPSBsYXN0LnNlbGZfYXR0ZW50aW9uKHksIHksIHksIG5lZWRfd2VpZ2h0cz1UcnVlLCBhdmVyYWdlX2F0dG5fd2VpZ2h0cz1UcnVlKQogICAgICAgIHggPSBlbmMubG4obGFzdCh4KSkKICAgICAgICByZXR1cm4geFs6LCAxOiwgOl0sIGF0dG5bOiwgMCwgMTpdCgoKY2xhc3MgRnJvemVuUm9CRVJUYShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwKICAgICAgICBzZWxmLnJvYmVydGEsIGluZm8gPSBBdXRvTW9kZWwuZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICBST0JFUlRBX05BTUUsIHJldmlzaW9uPVJPQkVSVEFfUkVWSVNJT04sIGFkZF9wb29saW5nX2xheWVyPUZhbHNlLCBvdXRwdXRfbG9hZGluZ19pbmZvPVRydWUpCiAgICAgICAgIyBsbV9oZWFkLiogYXBwZWFyIGFzIHVuZXhwZWN0ZWQga2V5czogdGhlIGNoZWNrcG9pbnQgaXMgYW4gTUxNIGNoZWNrcG9pbnQgYW5kCiAgICAgICAgIyB0aGUgTE0gaGVhZCBpcyBuZXZlciBpbnN0YW50aWF0ZWQuIEFueSAqbWlzc2luZyoga2V5IGlzIGEgcmVhbCBwcm9ibGVtLgogICAgICAgIHNlbGYubG9hZGluZ19pbmZvID0ge2s6IHNvcnRlZCh2KSBpZiBpc2luc3RhbmNlKHYsIChsaXN0LCBzZXQpKSBlbHNlIHYgZm9yIGssIHYgaW4gaW5mby5pdGVtcygpfQogICAgICAgIGlmIHNlbGYubG9hZGluZ19pbmZvLmdldCgibWlzc2luZ19rZXlzIik6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJvQkVSVGEgbWlzc2luZyB3ZWlnaHRzOiB7c2VsZi5sb2FkaW5nX2luZm9bJ21pc3Npbmdfa2V5cyddfSIpCiAgICAgICAgdW5leHBlY3RlZCA9IFtrIGZvciBrIGluIHNlbGYubG9hZGluZ19pbmZvLmdldCgidW5leHBlY3RlZF9rZXlzIiwgW10pIGlmIG5vdCBrLnN0YXJ0c3dpdGgoImxtX2hlYWQuIildCiAgICAgICAgaWYgdW5leHBlY3RlZDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUm9CRVJUYSB1bmV4cGVjdGVkIG5vbi1MTS1oZWFkIHdlaWdodHM6IHt1bmV4cGVjdGVkfSIpCiAgICAgICAgc2VsZi5jb21taXRfaGFzaCA9IGdldGF0dHIoc2VsZi5yb2JlcnRhLmNvbmZpZywgIl9jb21taXRfaGFzaCIsIE5vbmUpCiAgICAgICAgZm9yIHAgaW4gc2VsZi5yb2JlcnRhLnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgcC5yZXF1aXJlc19ncmFkID0gRmFsc2UKICAgICAgICBzZWxmLmV2YWwoKQoKICAgIGRlZiB0cmFpbihzZWxmLCBtb2RlPVRydWUpOgogICAgICAgIHJldHVybiBzdXBlcigpLnRyYWluKEZhbHNlKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGlucHV0X2lkcywgYXR0ZW50aW9uX21hc2spOgogICAgICAgIHJldHVybiBzZWxmLnJvYmVydGEoaW5wdXRfaWRzPWlucHV0X2lkcywgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2spLmxhc3RfaGlkZGVuX3N0YXRlCgoKZGVmIGxvYWRfdG9rZW5pemVyKCk6CiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Rva2VuaXplcgogICAgcmV0dXJuIEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKFJPQkVSVEFfTkFNRSwgcmV2aXNpb249Uk9CRVJUQV9SRVZJU0lPTikKCgpkZWYgdG9rZW5pemUodG9rZW5pemVyLCBjYXB0aW9ucyk6CiAgICByZXR1cm4gdG9rZW5pemVyKGxpc3QoY2FwdGlvbnMpLCBwYWRkaW5nPSJtYXhfbGVuZ3RoIiwgdHJ1bmNhdGlvbj1UcnVlLCBtYXhfbGVuZ3RoPU1BWF9URVhUX0xFTiwKICAgICAgICAgICAgICAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IikK"], "common.py": ["48ebdb01087dbbd46ddd92c9ea174aba5bf4d3d456f031281328d0f867749dfa", "IiIiU2hhcmVkIHByb3ZlbmFuY2UsIGhhc2hpbmcsIHNlZWRpbmcgYW5kIElPIGhlbHBlcnMuCgpFdmVyeSBhcnRpZmFjdCB3cml0dGVuIGJ5IHRoaXMgcGFja2FnZSBjYXJyaWVzIHRoZSBzYW1lIHByb3ZlbmFuY2UgYmxvY2sKKGBwcm92ZW5hbmNlKClgKSwgc28gYSByZXN1bHQgY2FuIGFsd2F5cyBiZSB0cmFjZWQgdG8gdGhlIGV4YWN0IGNvZGUsCmNvbmZpZ3VyYXRpb24sIGRhdGFzZXQgYW5kIGNhY2hlZCBmZWF0dXJlcyB0aGF0IHByb2R1Y2VkIGl0LgoiIiIKCmltcG9ydCBoYXNobGliCmltcG9ydCBpbXBvcnRsaWIubWV0YWRhdGEKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgcmFuZG9tCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKClBBQ0tBR0VfRElSID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudAoKIyBXb3JrIGRpcmVjdG9yeSBmb3IgZGF0YSwgZmVhdHVyZSBjYWNoZSBhbmQgcnVucy4gTmV2ZXIgaW5zaWRlIHRoZSBnaXQgY2hlY2tvdXQuCldPUktfRElSID0gUGF0aChvcy5lbnZpcm9uLmdldCgiSEVET19XT1JLIiwgIi9jb250ZW50L2hlZG9fd29yayIpKQpEQVRBX0RJUiA9IFdPUktfRElSIC8gImRhdGEiIC8gImZsaWNrcjhrIgpGRUFUVVJFX0RJUiA9IFdPUktfRElSIC8gImZlYXR1cmVzIgpSVU5TX0RJUiA9IFdPUktfRElSIC8gInJ1bnMiClJFUE9SVF9ESVIgPSBXT1JLX0RJUiAvICJyZXBvcnQiCgpUUkFDS0VEX1BBQ0tBR0VTID0gWwogICAgInRvcmNoIiwgInRvcmNodmlzaW9uIiwgIm1hbWJhX3NzbSIsICJjYXVzYWxfY29udjFkIiwgInRyaXRvbiIsICJ0cmFuc2Zvcm1lcnMiLAogICAgInRva2VuaXplcnMiLCAiaHVnZ2luZ2ZhY2VfaHViIiwgInNhZmV0ZW5zb3JzIiwgIm51bXB5IiwgInBhbmRhcyIsICJzY2lweSIsCiAgICAibWF0cGxvdGxpYiIsICJQaWxsb3ciLCAiZWlub3BzIiwKXQoKCmRlZiBzaGEyNTZfZmlsZShwYXRoLCBjaHVuaz0xIDw8IDIwKToKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZjoKICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6IGYucmVhZChjaHVuayksIGIiIik6CiAgICAgICAgICAgIGgudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpGRUFUVVJFX0NPREVfRklMRVMgPSAoImNvbW1vbi5weSIsICJkYXRhLnB5IiwgImJhY2tib25lcy5weSIsICJmZWF0dXJlcy5weSIpCgoKZGVmIGZlYXR1cmVfY29kZV9zaGEyNTYoKToKICAgICIiIkhhc2ggb2YgdGhlIGNvZGUgdGhhdCBkZXRlcm1pbmVzIGNhY2hlZCBmZWF0dXJlcy4gQSBtaXNtYXRjaCBtZWFucyB0aGUgY2FjaGUgaXMgc3RhbGUuIiIiCiAgICBoID0gaGFzaGxpYi5zaGEyNTYoKQogICAgZm9yIG5hbWUgaW4gRkVBVFVSRV9DT0RFX0ZJTEVTOgogICAgICAgIGgudXBkYXRlKG5hbWUuZW5jb2RlKCJ1dGYtOCIpKQogICAgICAgIGgudXBkYXRlKChQQUNLQUdFX0RJUiAvIG5hbWUpLnJlYWRfYnl0ZXMoKS5yZXBsYWNlKGIiXHJcbiIsIGIiXG4iKSkKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgoKZGVmIHNoYTI1Nl9qc29uKG9iaik6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYoanNvbi5kdW1wcyhvYmosIHNvcnRfa2V5cz1UcnVlLCBkZWZhdWx0PXN0cikuZW5jb2RlKCJ1dGYtOCIpKS5oZXhkaWdlc3QoKQoKCmRlZiBjb2RlX3NoYTI1NigpOgogICAgIiIiSGFzaCBvZiBldmVyeSAucHkgZmlsZSBpbiB0aGUgcGFja2FnZSAobmFtZSArIGJ5dGVzLCBzb3J0ZWQpLiBUZXN0cyBleGNsdWRlZC4iIiIKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICBmb3IgcCBpbiBzb3J0ZWQoUEFDS0FHRV9ESVIuZ2xvYigiKi5weSIpKToKICAgICAgICBoLnVwZGF0ZShwLm5hbWUuZW5jb2RlKCJ1dGYtOCIpKQogICAgICAgIGgudXBkYXRlKHAucmVhZF9ieXRlcygpKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgZ2l0X2NvbW1pdCgpOgogICAgdHJ5OgogICAgICAgIG91dCA9IHN1YnByb2Nlc3MucnVuKFsiZ2l0IiwgInJldi1wYXJzZSIsICJIRUFEIl0sIGN3ZD1QQUNLQUdFX0RJUiwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTEwKQogICAgICAgIGRpcnR5ID0gc3VicHJvY2Vzcy5ydW4oWyJnaXQiLCAic3RhdHVzIiwgIi0tcG9yY2VsYWluIiwgIi0tIiwgc3RyKFBBQ0tBR0VfRElSKV0sIGN3ZD1QQUNLQUdFX0RJUiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwgdGltZW91dD0xMCkKICAgICAgICBpZiBvdXQucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICByZXR1cm4geyJjb21taXQiOiBOb25lLCAiZGlydHkiOiBOb25lfQogICAgICAgIHJldHVybiB7ImNvbW1pdCI6IG91dC5zdGRvdXQuc3RyaXAoKSwgImRpcnR5IjogYm9vbChkaXJ0eS5zdGRvdXQuc3RyaXAoKSl9CiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiB7ImNvbW1pdCI6IE5vbmUsICJkaXJ0eSI6IE5vbmV9CgoKZGVmIHBhY2thZ2VfdmVyc2lvbnMoKToKICAgIHZlcnNpb25zID0ge30KICAgIGZvciBuYW1lIGluIFRSQUNLRURfUEFDS0FHRVM6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB2ZXJzaW9uc1tuYW1lXSA9IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpCiAgICAgICAgZXhjZXB0IGltcG9ydGxpYi5tZXRhZGF0YS5QYWNrYWdlTm90Rm91bmRFcnJvcjoKICAgICAgICAgICAgdmVyc2lvbnNbbmFtZV0gPSBOb25lCiAgICByZXR1cm4gdmVyc2lvbnMKCgpkZWYgZW52aXJvbm1lbnRfbWFuaWZlc3QoKToKICAgIGluZm8gPSB7CiAgICAgICAgInB5dGhvbiI6IHN5cy52ZXJzaW9uLAogICAgICAgICJleGVjdXRhYmxlIjogc3lzLmV4ZWN1dGFibGUsCiAgICAgICAgInBsYXRmb3JtIjogcGxhdGZvcm0ucGxhdGZvcm0oKSwKICAgICAgICAicGFja2FnZXMiOiBwYWNrYWdlX3ZlcnNpb25zKCksCiAgICAgICAgInRpbWVzdGFtcF91dGMiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKSwKICAgIH0KICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBpbmZvWyJ0b3JjaF9jdWRhIl0gPSB0b3JjaC52ZXJzaW9uLmN1ZGEKICAgICAgICBpbmZvWyJjdWRubiJdID0gdG9yY2guYmFja2VuZHMuY3Vkbm4udmVyc2lvbigpCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgaW5mb1siZ3B1Il0gPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKQogICAgICAgICAgICBpbmZvWyJncHVfY2FwYWJpbGl0eSJdID0gbGlzdCh0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eSgwKSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGluZm9bInRvcmNoX2Vycm9yIl0gPSByZXByKGV4YykKICAgIHRyeToKICAgICAgICBpbmZvWyJudmlkaWFfc21pIl0gPSBzdWJwcm9jZXNzLnJ1bihbIm52aWRpYS1zbWkiXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTIwKS5zdGRvdXQKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgaW5mb1sibnZpZGlhX3NtaSJdID0gTm9uZQogICAgdHJ5OgogICAgICAgIGluZm9bInBpcF9mcmVlemUiXSA9IHN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwgIi1tIiwgInBpcCIsICJmcmVlemUiXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUsIHRpbWVvdXQ9MTIwKS5zdGRvdXQuc3BsaXRsaW5lcygpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGluZm9bInBpcF9mcmVlemUiXSA9IE5vbmUKICAgIHJldHVybiBpbmZvCgoKZGVmIHByb3ZlbmFuY2UoY29uZmlnPU5vbmUsIGV4dHJhPU5vbmUsIGluY2x1ZGVfYXJ0aWZhY3RzPVRydWUpOgogICAgYmxvY2sgPSB7CiAgICAgICAgImNvZGVfc2hhMjU2IjogY29kZV9zaGEyNTYoKSwKICAgICAgICAiZ2l0IjogZ2l0X2NvbW1pdCgpLAogICAgICAgICJwYWNrYWdlcyI6IHBhY2thZ2VfdmVyc2lvbnMoKSwKICAgICAgICAicHl0aG9uIjogc3lzLnZlcnNpb24uc3BsaXQoKVswXSwKICAgIH0KICAgIGlmIGNvbmZpZyBpcyBub3QgTm9uZToKICAgICAgICBibG9ja1siY29uZmlnIl0gPSBjb25maWcKICAgICAgICBibG9ja1siY29uZmlnX3NoYTI1NiJdID0gc2hhMjU2X2pzb24oY29uZmlnKQogICAgbWFuaWZlc3QgPSBEQVRBX0RJUiAvICJtYW5pZmVzdC5qc29uIgogICAgaWYgaW5jbHVkZV9hcnRpZmFjdHMgYW5kIG1hbmlmZXN0LmlzX2ZpbGUoKToKICAgICAgICBibG9ja1siZGF0YXNldF9tYW5pZmVzdF9zaGEyNTYiXSA9IHNoYTI1Nl9maWxlKG1hbmlmZXN0KQogICAgZm1hbmlmZXN0ID0gRkVBVFVSRV9ESVIgLyAiZmVhdHVyZV9tYW5pZmVzdC5qc29uIgogICAgaWYgaW5jbHVkZV9hcnRpZmFjdHMgYW5kIGZtYW5pZmVzdC5pc19maWxlKCk6CiAgICAgICAgYmxvY2tbImZlYXR1cmVfbWFuaWZlc3Rfc2hhMjU2Il0gPSBzaGEyNTZfZmlsZShmbWFuaWZlc3QpCiAgICBpZiBleHRyYToKICAgICAgICBibG9jay51cGRhdGUoZXh0cmEpCiAgICByZXR1cm4gYmxvY2sKCgpkZWYgc2V0X3NlZWQoc2VlZCwgZGV0ZXJtaW5pc3RpYz1UcnVlKToKICAgIGltcG9ydCB0b3JjaAogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJDVUJMQVNfV09SS1NQQUNFX0NPTkZJRyIsICI6NDA5Njo4IikKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgdG9yY2gubWFudWFsX3NlZWQoc2VlZCkKICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNlZWQpCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmsgPSBGYWxzZQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uZGV0ZXJtaW5pc3RpYyA9IFRydWUKICAgIGlmIGRldGVybWluaXN0aWM6CiAgICAgICAgIyB3YXJuX29ubHk6IE1hbWJhLTIgVHJpdG9uIGJhY2t3YXJkIGtlcm5lbHMgaGF2ZSBubyBkZXRlcm1pbmlzdGljIHZhcmlhbnQuCiAgICAgICAgIyBUcmFpbmluZyBpcyB0aGVyZWZvcmUgTk9UIGJpdC1yZXByb2R1Y2libGU7IGV2YWx1YXRpb24gaXMgKGNoZWNrZWQgc2VwYXJhdGVseSkuCiAgICAgICAgdG9yY2gudXNlX2RldGVybWluaXN0aWNfYWxnb3JpdGhtcyhUcnVlLCB3YXJuX29ubHk9VHJ1ZSkKCgpkZWYgd3JpdGVfanNvbihwYXRoLCBvYmopOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAiLnRtcCIpCiAgICB3aXRoIG9wZW4odG1wLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKG9iaiwgZiwgaW5kZW50PTIsIGRlZmF1bHQ9X2pzb25fZGVmYXVsdCkKICAgIG9zLnJlcGxhY2UodG1wLCBwYXRoKQoKCmRlZiByZWFkX2pzb24ocGF0aCk6CiAgICB3aXRoIG9wZW4ocGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJldHVybiBqc29uLmxvYWQoZikKCgpkZWYgX2pzb25fZGVmYXVsdChvKToKICAgIGlmIGlzaW5zdGFuY2UobywgKG5wLmludGVnZXIsKSk6CiAgICAgICAgcmV0dXJuIGludChvKQogICAgaWYgaXNpbnN0YW5jZShvLCAobnAuZmxvYXRpbmcsKSk6CiAgICAgICAgcmV0dXJuIGZsb2F0KG8pCiAgICBpZiBpc2luc3RhbmNlKG8sIG5wLm5kYXJyYXkpOgogICAgICAgIHJldHVybiBvLnRvbGlzdCgpCiAgICBpZiBpc2luc3RhbmNlKG8sIFBhdGgpOgogICAgICAgIHJldHVybiBzdHIobykKICAgIHJhaXNlIFR5cGVFcnJvcihmIk5vdCBKU09OIHNlcmlhbGlzYWJsZToge3R5cGUobyl9IikKCgpkZWYgYmFubmVyKHRleHQpOgogICAgcHJpbnQoIj0iICogODAsIGZsdXNoPVRydWUpCiAgICBwcmludCh0ZXh0LCBmbHVzaD1UcnVlKQogICAgcHJpbnQoIj0iICogODAsIGZsdXNoPVRydWUpCg=="], "data.py": ["78d3075c4f171c94f6ec795a4b1c8fbde4522eb583e271a2224e7b47d99f20dd", "IiIiRmxpY2tyOGsgcHJlcGFyYXRpb246IGRvd25sb2FkLCBjaGVja3N1bSwgb2ZmaWNpYWwgc3BsaXRzLCBsZWFrYWdlIGNoZWNrcy4KCldyaXRlcyB0byBEQVRBX0RJUjoKICBtYW5pZmVzdC5qc29uICAgICAgICAgICAgY2Fub25pY2FsIHBhdGhzICsgYWxsIGNoZWNrc3VtcyAoc2luZ2xlIGtleSBzY2hlbWEpCiAgY2FwdGlvbnNfe3RyYWluLHZhbCx0ZXN0fS5jc3YgICBpbWFnZV9pZCwgY2FwX2lkeCwgY2FwdGlvbjsgc29ydGVkIGJ5IChpbWFnZV9pZCwgY2FwX2lkeCkKICBsZWFrYWdlX3JlcG9ydC5qc29uICAgICAgZXhhY3QvbmVhciBkdXBsaWNhdGUgaW1hZ2VzIGFuZCBkdXBsaWNhdGUgY2FwdGlvbnMgYWNyb3NzIHNwbGl0cwoKQ2Fub25pY2FsIG9yZGVyaW5nOiB3aXRoaW4gYSBzcGxpdCwgaW1hZ2UgaSBvd25zIGNhcHRpb25zIDVpLi41aSs0LiBFdmVyeQpkb3duc3RyZWFtIHNjcmlwdCAoZmVhdHVyZXMsIHRyYWluaW5nLCBldmFsdWF0aW9uKSByZWxpZXMgb24gdGhpcyBvcmRlci4KClVzYWdlOiBweXRob24gZGF0YS5weSBbLS1leHBlY3RlZC1jaGVja3N1bXMgY2hlY2tzdW1zLmpzb25dCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHVybGxpYi5yZXF1ZXN0CmltcG9ydCB6aXBmaWxlCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpmcm9tIGNvbW1vbiBpbXBvcnQgREFUQV9ESVIsIGJhbm5lciwgcHJvdmVuYW5jZSwgcmVhZF9qc29uLCBzaGEyNTZfZmlsZSwgc2hhMjU2X2pzb24sIHdyaXRlX2pzb24KCiMgVGhpcmQtcGFydHkgbWlycm9yIG9mIHRoZSBvcmlnaW5hbCBGbGlja3I4ayByZWxlYXNlLiBDaGVja3N1bXMgYXJlIHJlY29yZGVkIG9uCiMgZmlyc3QgZG93bmxvYWQ7IHBpbiB0aGVtIHdpdGggLS1leHBlY3RlZC1jaGVja3N1bXMgZm9yIGV2ZXJ5IGxhdGVyIHJ1bi4KQVJDSElWRVMgPSB7CiAgICAiRmxpY2tyOGtfRGF0YXNldC56aXAiOiAiaHR0cHM6Ly9naXRodWIuY29tL0F2YW5lZXNoNDA1ODUvRmxpY2tyOGstRGF0YXNldC9yZWxlYXNlcy9kb3dubG9hZC92MS4wL0ZsaWNrcjhrX0RhdGFzZXQuemlwIiwKICAgICJGbGlja3I4a190ZXh0LnppcCI6ICJodHRwczovL2dpdGh1Yi5jb20vQXZhbmVlc2g0MDU4NS9GbGlja3I4ay1EYXRhc2V0L3JlbGVhc2VzL2Rvd25sb2FkL3YxLjAvRmxpY2tyOGtfdGV4dC56aXAiLAp9ClNQTElUX0ZJTEVTID0geyJ0cmFpbiI6ICJGbGlja3JfOGsudHJhaW5JbWFnZXMudHh0IiwgInZhbCI6ICJGbGlja3JfOGsuZGV2SW1hZ2VzLnR4dCIsICJ0ZXN0IjogIkZsaWNrcl84ay50ZXN0SW1hZ2VzLnR4dCJ9CkVYUEVDVEVEID0geyJ0cmFpbiI6IDYwMDAsICJ2YWwiOiAxMDAwLCAidGVzdCI6IDEwMDB9CkNBUFRJT05TX1BFUl9JTUFHRSA9IDUKTkVBUl9EVVBfSEFNTUlORyA9IDIKCgpkZWYgZG93bmxvYWQodXJsLCBwYXRoKToKICAgIGlmIHBhdGguaXNfZmlsZSgpIGFuZCBwYXRoLnN0YXQoKS5zdF9zaXplID4gMDoKICAgICAgICBwcmludChmInJldXNlIHtwYXRoLm5hbWV9ICh7cGF0aC5zdGF0KCkuc3Rfc2l6ZSAvIDIqKjIwOi4xZn0gTUIpIikKICAgICAgICByZXR1cm4KICAgIHByaW50KGYiZG93bmxvYWQge3VybH0iKQogICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCh1cmwsIGhlYWRlcnM9eyJVc2VyLUFnZW50IjogIk1vemlsbGEvNS4wIn0pCiAgICB0bXAgPSBwYXRoLndpdGhfc3VmZml4KCIucGFydCIpCiAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxKSBhcyByLCBvcGVuKHRtcCwgIndiIikgYXMgZjoKICAgICAgICBzaHV0aWwuY29weWZpbGVvYmoociwgZikKICAgIHRtcC5yZXBsYWNlKHBhdGgpCgoKZGVmIGZpbmRfdW5pcXVlKHJvb3QsIG5hbWUpOgogICAgaGl0cyA9IFtwIGZvciBwIGluIHJvb3Qucmdsb2IoIioiKSBpZiBwLmlzX2ZpbGUoKSBhbmQgcC5uYW1lLmxvd2VyKCkgPT0gbmFtZS5sb3dlcigpIGFuZCAiX19NQUNPU1giIG5vdCBpbiBwLnBhcnRzXQogICAgaWYgbGVuKGhpdHMpICE9IDE6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJleHBlY3RlZCBleGFjdGx5IG9uZSB7bmFtZX0gdW5kZXIge3Jvb3R9LCBmb3VuZCB7bGVuKGhpdHMpfToge2hpdHNbOjVdfSIpCiAgICByZXR1cm4gaGl0c1swXQoKCmRlZiBmaW5kX2ltYWdlX2Rpcihyb290KToKICAgIGJlc3QsIGJlc3RfbiA9IE5vbmUsIDAKICAgIGZvciBkIGluIFtyb290LCAqW3AgZm9yIHAgaW4gcm9vdC5yZ2xvYigiKiIpIGlmIHAuaXNfZGlyKCkgYW5kICJfX01BQ09TWCIgbm90IGluIHAucGFydHNdXToKICAgICAgICBuID0gc3VtKDEgZm9yIHAgaW4gZC5pdGVyZGlyKCkgaWYgcC5pc19maWxlKCkgYW5kIHAuc3VmZml4Lmxvd2VyKCkgPT0gIi5qcGciKQogICAgICAgIGlmIG4gPiBiZXN0X246CiAgICAgICAgICAgIGJlc3QsIGJlc3RfbiA9IGQsIG4KICAgIGlmIGJlc3RfbiA8IDgwMDA6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJubyBkaXJlY3Rvcnkgd2l0aCA+PTgwMDAganBnIGltYWdlcyB1bmRlciB7cm9vdH0gKGJlc3Q6IHtiZXN0X259KSIpCiAgICByZXR1cm4gYmVzdAoKCmRlZiByZWFkX2lkcyhwYXRoKToKICAgIHJldHVybiBzb3J0ZWQoe2xpbmUuc3RyaXAoKSBmb3IgbGluZSBpbiBvcGVuKHBhdGgsIGVuY29kaW5nPSJ1dGYtOCIpIGlmIGxpbmUuc3RyaXAoKX0pCgoKZGVmIHBhcnNlX2NhcHRpb25zKHRva2VuX2ZpbGUpOgogICAgcm93cyA9IFtdCiAgICBmb3IgbGluZSBpbiBvcGVuKHRva2VuX2ZpbGUsIGVuY29kaW5nPSJ1dGYtOCIsIGVycm9ycz0ic3RyaWN0Iik6CiAgICAgICAgbGluZSA9IGxpbmUucnN0cmlwKCJcbiIpCiAgICAgICAgaWYgbm90IGxpbmUuc3RyaXAoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBrZXksIGNhcHRpb24gPSBsaW5lLnNwbGl0KCJcdCIsIDEpCiAgICAgICAgaW1hZ2VfaWQsIGlkeCA9IGtleS5zcGxpdCgiIyIsIDEpCiAgICAgICAgcm93cy5hcHBlbmQoKGltYWdlX2lkLnN0cmlwKCksIGludChpZHgpLCBjYXB0aW9uLnN0cmlwKCkpKQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzLCBjb2x1bW5zPVsiaW1hZ2VfaWQiLCAiY2FwX2lkeCIsICJjYXB0aW9uIl0pCgoKZGVmIGRoYXNoKHBhdGgsIHNpemU9OCk6CiAgICB3aXRoIEltYWdlLm9wZW4ocGF0aCkgYXMgaW06CiAgICAgICAgZyA9IG5wLmFzYXJyYXkoaW0uY29udmVydCgiTCIpLnJlc2l6ZSgoc2l6ZSArIDEsIHNpemUpLCBJbWFnZS5CSUxJTkVBUiksIGR0eXBlPW5wLmludDE2KQogICAgYml0cyA9IChnWzosIDE6XSA+IGdbOiwgOi0xXSkuZmxhdHRlbigpCiAgICByZXR1cm4gaW50KG5wLnBhY2tiaXRzKGJpdHMpLnZpZXcoIj51OCIpWzBdKQoKCmRlZiBwb3Bjb3VudDY0KHgpOgogICAgeCA9IHguYXN0eXBlKG5wLnVpbnQ2NCkKICAgIGlmIGhhc2F0dHIobnAsICJiaXR3aXNlX2NvdW50Iik6CiAgICAgICAgcmV0dXJuIG5wLmJpdHdpc2VfY291bnQoeCkKICAgIHRhYmxlID0gbnAuYXJyYXkoW2JpbihpKS5jb3VudCgiMSIpIGZvciBpIGluIHJhbmdlKDI1NildLCBkdHlwZT1ucC51aW50OCkKICAgIHJldHVybiB0YWJsZVt4LnZpZXcobnAudWludDgpLnJlc2hhcGUoLTEsIDgpXS5zdW0oYXhpcz0xKQoKCmRlZiBsZWFrYWdlX3JlcG9ydChpbWFnZV9kaXIsIGlkcywgdGFibGVzKToKICAgIHJlcG9ydCA9IHt9CiAgICAjIDEuIGV4YWN0IGR1cGxpY2F0ZSBpbWFnZSBieXRlcyBhY3Jvc3Mgc3BsaXRzCiAgICBmaWxlX2hhc2ggPSB7czoge2k6IHNoYTI1Nl9maWxlKGltYWdlX2RpciAvIGkpIGZvciBpIGluIGlkc1tzXX0gZm9yIHMgaW4gaWRzfQogICAgb3duZXIgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIHMgaW4gaWRzOgogICAgICAgIGZvciBpLCBoIGluIGZpbGVfaGFzaFtzXS5pdGVtcygpOgogICAgICAgICAgICBvd25lcltoXS5hcHBlbmQoKHMsIGkpKQogICAgZXhhY3QgPSBbdiBmb3IgdiBpbiBvd25lci52YWx1ZXMoKSBpZiBsZW4oe3MgZm9yIHMsIF8gaW4gdn0pID4gMV0KICAgIHJlcG9ydFsiZXhhY3RfZHVwbGljYXRlX2ltYWdlc19hY3Jvc3Nfc3BsaXRzIl0gPSBleGFjdAoKICAgICMgMi4gbmVhci1kdXBsaWNhdGUgaW1hZ2VzIChkSGFzaCwgSGFtbWluZyA8PSBORUFSX0RVUF9IQU1NSU5HKSB0cmFpbiB2cyB2YWwvdGVzdAogICAgaGFzaGVzID0ge3M6IG5wLmFycmF5KFtkaGFzaChpbWFnZV9kaXIgLyBpKSBmb3IgaSBpbiBpZHNbc11dLCBkdHlwZT1ucC51aW50NjQpIGZvciBzIGluIGlkc30KICAgIG5lYXIgPSBbXQogICAgZm9yIG90aGVyIGluICgidmFsIiwgInRlc3QiKToKICAgICAgICBmb3IgaiwgaCBpbiBlbnVtZXJhdGUoaGFzaGVzW290aGVyXSk6CiAgICAgICAgICAgIGRpc3QgPSBwb3Bjb3VudDY0KG5wLmJpdHdpc2VfeG9yKGhhc2hlc1sidHJhaW4iXSwgaCkpCiAgICAgICAgICAgIGZvciBrIGluIG5wLm5vbnplcm8oZGlzdCA8PSBORUFSX0RVUF9IQU1NSU5HKVswXToKICAgICAgICAgICAgICAgIG5lYXIuYXBwZW5kKHsidHJhaW4iOiBpZHNbInRyYWluIl1ba10sIG90aGVyOiBpZHNbb3RoZXJdW2pdLCAiaGFtbWluZyI6IGludChkaXN0W2tdKX0pCiAgICByZXBvcnRbIm5lYXJfZHVwbGljYXRlX2ltYWdlc190cmFpbl92c19ldmFsIl0gPSBuZWFyCgogICAgIyAzLiBpZGVudGljYWwgbm9ybWFsaXNlZCBjYXB0aW9ucyBzaGFyZWQgYmV0d2VlbiB0cmFpbiBhbmQgZXZhbHVhdGlvbiBzcGxpdHMKICAgIG5vcm0gPSB7czogc2V0KHRhYmxlc1tzXVsiY2FwdGlvbiJdLnN0ci5sb3dlcigpLnN0ci5yZXBsYWNlKHIiW15hLXowLTkgXSIsICIiLCByZWdleD1UcnVlKS5zdHIuc3BsaXQoKS5zdHIuam9pbigiICIpKQogICAgICAgICAgICBmb3IgcyBpbiB0YWJsZXN9CiAgICByZXBvcnRbImR1cGxpY2F0ZV9jYXB0aW9uX3N0cmluZ3MiXSA9IHsKICAgICAgICAidHJhaW5fdmFsIjogbGVuKG5vcm1bInRyYWluIl0gJiBub3JtWyJ2YWwiXSksCiAgICAgICAgInRyYWluX3Rlc3QiOiBsZW4obm9ybVsidHJhaW4iXSAmIG5vcm1bInRlc3QiXSksCiAgICAgICAgIm5vdGUiOiAiR2VuZXJpYyBzaG9ydCBjYXB0aW9ucyByZWN1ciBuYXR1cmFsbHk7IHJlcG9ydGVkLCBub3QgdHJlYXRlZCBhcyBsZWFrYWdlLiIsCiAgICB9CiAgICByZXBvcnRbImltYWdlX2ZpbGVfc2hhMjU2Il0gPSBmaWxlX2hhc2gKICAgIHJldHVybiByZXBvcnQKCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXhwZWN0ZWQtY2hlY2tzdW1zIiwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgIGhlbHA9IkpTT04ge2FyY2hpdmVfbmFtZTogc2hhMjU2fS4gV2hlbiBnaXZlbiwgYW55IG1pc21hdGNoIGFib3J0cy4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWFsbG93LWltYWdlLWxlYWthZ2UiLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9IkRvIG5vdCBhYm9ydCB3aGVuIGV4YWN0IGR1cGxpY2F0ZSBpbWFnZXMgZXhpc3QgYWNyb3NzIHNwbGl0cy4iKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGJhbm5lcigiRkxJQ0tSOEsgUFJFUEFSQVRJT04iKQogICAgREFUQV9ESVIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgYXJjaGl2ZV9zaGEgPSB7fQogICAgZm9yIG5hbWUsIHVybCBpbiBBUkNISVZFUy5pdGVtcygpOgogICAgICAgIHBhdGggPSBEQVRBX0RJUiAvIG5hbWUKICAgICAgICBkb3dubG9hZCh1cmwsIHBhdGgpCiAgICAgICAgYXJjaGl2ZV9zaGFbbmFtZV0gPSBzaGEyNTZfZmlsZShwYXRoKQogICAgICAgIHByaW50KGYic2hhMjU2IHtuYW1lfToge2FyY2hpdmVfc2hhW25hbWVdfSIpCiAgICBpZiBhcmdzLmV4cGVjdGVkX2NoZWNrc3VtczoKICAgICAgICBleHBlY3RlZCA9IHJlYWRfanNvbihhcmdzLmV4cGVjdGVkX2NoZWNrc3VtcykKICAgICAgICBmb3IgbmFtZSwgZGlnZXN0IGluIGV4cGVjdGVkLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIGFyY2hpdmVfc2hhLmdldChuYW1lKSAhPSBkaWdlc3Q6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJjaGVja3N1bSBtaXNtYXRjaCBmb3Ige25hbWV9OiB7YXJjaGl2ZV9zaGEuZ2V0KG5hbWUpfSAhPSB7ZGlnZXN0fSIpCiAgICAgICAgcHJpbnQoIkFSQ0hJVkVfQ0hFQ0tTVU1TOiBQQVNTIChwaW5uZWQpIikKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoIkFSQ0hJVkVfQ0hFQ0tTVU1TOiBSRUNPUkRFRCAobm90IHBpbm5lZDsgcGFzcyAtLWV4cGVjdGVkLWNoZWNrc3VtcyB0byBlbmZvcmNlKSIpCgogICAgZXh0cmFjdF9yb290ID0gREFUQV9ESVIgLyAiZXh0cmFjdGVkIgogICAgbWFya2VyID0gZXh0cmFjdF9yb290IC8gIi5jb21wbGV0ZSIKICAgIGlmIG5vdCBtYXJrZXIuaXNfZmlsZSgpOgogICAgICAgIGlmIGV4dHJhY3Rfcm9vdC5leGlzdHMoKToKICAgICAgICAgICAgc2h1dGlsLnJtdHJlZShleHRyYWN0X3Jvb3QpCiAgICAgICAgZm9yIG5hbWUgaW4gQVJDSElWRVM6CiAgICAgICAgICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKERBVEFfRElSIC8gbmFtZSkgYXMgemY6CiAgICAgICAgICAgICAgICB6Zi5leHRyYWN0YWxsKGV4dHJhY3Rfcm9vdCkKICAgICAgICBtYXJrZXIud3JpdGVfdGV4dCgib2siKQoKICAgIGltYWdlX2RpciA9IGZpbmRfaW1hZ2VfZGlyKGV4dHJhY3Rfcm9vdCkKICAgIHRva2VuX2ZpbGUgPSBmaW5kX3VuaXF1ZShleHRyYWN0X3Jvb3QsICJGbGlja3I4ay50b2tlbi50eHQiKQogICAgc3BsaXRfcGF0aHMgPSB7czogZmluZF91bmlxdWUoZXh0cmFjdF9yb290LCBmKSBmb3IgcywgZiBpbiBTUExJVF9GSUxFUy5pdGVtcygpfQoKICAgIGlkcyA9IHtzOiByZWFkX2lkcyhwKSBmb3IgcywgcCBpbiBzcGxpdF9wYXRocy5pdGVtcygpfQogICAgZm9yIHMsIG4gaW4gRVhQRUNURUQuaXRlbXMoKToKICAgICAgICBhc3NlcnQgbGVuKGlkc1tzXSkgPT0gbiwgZiJ7c306IGV4cGVjdGVkIHtufSBpbWFnZXMsIGdvdCB7bGVuKGlkc1tzXSl9IgogICAgYXNzZXJ0IHNldChpZHNbInRyYWluIl0pLmlzZGlzam9pbnQoaWRzWyJ2YWwiXSkKICAgIGFzc2VydCBzZXQoaWRzWyJ0cmFpbiJdKS5pc2Rpc2pvaW50KGlkc1sidGVzdCJdKQogICAgYXNzZXJ0IHNldChpZHNbInZhbCJdKS5pc2Rpc2pvaW50KGlkc1sidGVzdCJdKQogICAgbWlzc2luZyA9IFtpIGZvciBzIGluIGlkcyBmb3IgaSBpbiBpZHNbc10gaWYgbm90IChpbWFnZV9kaXIgLyBpKS5pc19maWxlKCldCiAgICBhc3NlcnQgbm90IG1pc3NpbmcsIGYie2xlbihtaXNzaW5nKX0gaW1hZ2VzIG1pc3NpbmcsIGUuZy4ge21pc3NpbmdbOjVdfSIKICAgIHByaW50KCJTUExJVFM6IDYwMDAvMTAwMC8xMDAwIGRpc2pvaW50LCBhbGwgaW1hZ2UgZmlsZXMgcHJlc2VudCIpCgogICAgY2FwdGlvbnMgPSBwYXJzZV9jYXB0aW9ucyh0b2tlbl9maWxlKQogICAgdGFibGVzID0ge30KICAgIGZvciBzIGluIGlkczoKICAgICAgICB0ID0gY2FwdGlvbnNbY2FwdGlvbnNbImltYWdlX2lkIl0uaXNpbihzZXQoaWRzW3NdKSldLnNvcnRfdmFsdWVzKFsiaW1hZ2VfaWQiLCAiY2FwX2lkeCJdKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICAgICAgY291bnRzID0gdC5ncm91cGJ5KCJpbWFnZV9pZCIpLnNpemUoKQogICAgICAgIGFzc2VydCBsZW4oY291bnRzKSA9PSBsZW4oaWRzW3NdKSBhbmQgKGNvdW50cyA9PSBDQVBUSU9OU19QRVJfSU1BR0UpLmFsbCgpLCBmIntzfTogbm90IDUgY2FwdGlvbnMgcGVyIGltYWdlIgogICAgICAgIGFzc2VydCB0WyJpbWFnZV9pZCJdLmlsb2NbOjpDQVBUSU9OU19QRVJfSU1BR0VdLnRvbGlzdCgpID09IGlkc1tzXSwgZiJ7c306IGNhbm9uaWNhbCBvcmRlciBicm9rZW4iCiAgICAgICAgYXNzZXJ0ICh0WyJjYXB0aW9uIl0uc3RyLmxlbigpID4gMCkuYWxsKCkKICAgICAgICB0YWJsZXNbc10gPSB0CiAgICAgICAgdC50b19jc3YoREFUQV9ESVIgLyBmImNhcHRpb25zX3tzfS5jc3YiLCBpbmRleD1GYWxzZSkKICAgIHByaW50KCJDQVBUSU9OUzogMzAwMDAvNTAwMC81MDAwLCBleGFjdGx5IDUgcGVyIGltYWdlLCBjYW5vbmljYWwgb3JkZXIgdmVyaWZpZWQiKQoKICAgIGJhbm5lcigiTEVBS0FHRSBDSEVDS1MiKQogICAgcmVwb3J0ID0gbGVha2FnZV9yZXBvcnQoaW1hZ2VfZGlyLCBpZHMsIHRhYmxlcykKICAgIG5fZXhhY3QgPSBsZW4ocmVwb3J0WyJleGFjdF9kdXBsaWNhdGVfaW1hZ2VzX2Fjcm9zc19zcGxpdHMiXSkKICAgIG5fbmVhciA9IGxlbihyZXBvcnRbIm5lYXJfZHVwbGljYXRlX2ltYWdlc190cmFpbl92c19ldmFsIl0pCiAgICBwcmludChmImV4YWN0IGR1cGxpY2F0ZSBpbWFnZXMgYWNyb3NzIHNwbGl0cyA6IHtuX2V4YWN0fSIpCiAgICBwcmludChmIm5lYXItZHVwbGljYXRlIGltYWdlcyAoZEhhc2g8PXtORUFSX0RVUF9IQU1NSU5HfSkgIDoge25fbmVhcn0iKQogICAgcHJpbnQoZiJzaGFyZWQgY2FwdGlvbiBzdHJpbmdzIHRyYWluL3ZhbC90ZXN0OiB7cmVwb3J0WydkdXBsaWNhdGVfY2FwdGlvbl9zdHJpbmdzJ119IikKICAgIHdyaXRlX2pzb24oREFUQV9ESVIgLyAibGVha2FnZV9yZXBvcnQuanNvbiIsIHJlcG9ydCkKICAgIGlmIG5fZXhhY3QgYW5kIG5vdCBhcmdzLmFsbG93X2ltYWdlX2xlYWthZ2U6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJleGFjdCBkdXBsaWNhdGUgaW1hZ2VzIGFjcm9zcyBzcGxpdHM7IGluc3BlY3QgbGVha2FnZV9yZXBvcnQuanNvbiIpCgogICAgbWFuaWZlc3QgPSB7CiAgICAgICAgImRhdGFzZXQiOiAiRmxpY2tyOGsiLAogICAgICAgICJzb3VyY2VfdXJscyI6IEFSQ0hJVkVTLAogICAgICAgICJhcmNoaXZlX3NoYTI1NiI6IGFyY2hpdmVfc2hhLAogICAgICAgICJhcmNoaXZlX2NoZWNrc3Vtc19waW5uZWQiOiBib29sKGFyZ3MuZXhwZWN0ZWRfY2hlY2tzdW1zKSwKICAgICAgICAiaW1hZ2VfZGlyIjogc3RyKGltYWdlX2RpciksCiAgICAgICAgImNhcHRpb25fZmlsZSI6IHN0cih0b2tlbl9maWxlKSwKICAgICAgICAiY2FwdGlvbl9maWxlX3NoYTI1NiI6IHNoYTI1Nl9maWxlKHRva2VuX2ZpbGUpLAogICAgICAgICJzcGxpdF9maWxlcyI6IHtzOiBzdHIocCkgZm9yIHMsIHAgaW4gc3BsaXRfcGF0aHMuaXRlbXMoKX0sCiAgICAgICAgInNwbGl0X2ZpbGVfc2hhMjU2Ijoge3M6IHNoYTI1Nl9maWxlKHApIGZvciBzLCBwIGluIHNwbGl0X3BhdGhzLml0ZW1zKCl9LAogICAgICAgICJpbWFnZV9saXN0X3NoYTI1NiI6IHtzOiBzaGEyNTZfanNvbihpZHNbc10pIGZvciBzIGluIGlkc30sCiAgICAgICAgImNhcHRpb25fdGFibGVfc2hhMjU2Ijoge3M6IHNoYTI1Nl9maWxlKERBVEFfRElSIC8gZiJjYXB0aW9uc197c30uY3N2IikgZm9yIHMgaW4gaWRzfSwKICAgICAgICAiY291bnRzIjoge3M6IHsiaW1hZ2VzIjogbGVuKGlkc1tzXSksICJjYXB0aW9ucyI6IGxlbih0YWJsZXNbc10pfSBmb3IgcyBpbiBpZHN9LAogICAgICAgICJjYXB0aW9uc19wZXJfaW1hZ2UiOiBDQVBUSU9OU19QRVJfSU1BR0UsCiAgICAgICAgImxlYWthZ2UiOiB7ImV4YWN0X2R1cGxpY2F0ZV9pbWFnZXMiOiBuX2V4YWN0LCAibmVhcl9kdXBsaWNhdGVfaW1hZ2VzIjogbl9uZWFyLAogICAgICAgICAgICAgICAgICAgICJyZXBvcnRfc2hhMjU2Ijogc2hhMjU2X2ZpbGUoREFUQV9ESVIgLyAibGVha2FnZV9yZXBvcnQuanNvbiIpfSwKICAgICAgICAic3ludGhldGljX2RhdGEiOiBGYWxzZSwKICAgIH0KICAgIG1hbmlmZXN0WyJwcm92ZW5hbmNlIl0gPSBwcm92ZW5hbmNlKGluY2x1ZGVfYXJ0aWZhY3RzPUZhbHNlKQogICAgd3JpdGVfanNvbihEQVRBX0RJUiAvICJtYW5pZmVzdC5qc29uIiwgbWFuaWZlc3QpCiAgICBwcmludChmIm1hbmlmZXN0OiB7REFUQV9ESVIgLyAnbWFuaWZlc3QuanNvbid9IikKICAgIHByaW50KCJEQVRBU0VUX1JFQURZOiBQQVNTIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="], "evaluate_independent.py": ["f45d8462d67937b2d933a03a107ead5007e14794cd7c2a16eb304b68259543b8", "IiIiSW5kZXBlbmRlbnQgcmV0cmlldmFsIGV2YWx1YXRvci4KClNoYXJlcyBubyBjb2RlIHdpdGggbWV0cmljcy5weSBvciB0aGUgbW9kZWwuIFJlYWRzIG9ubHkgdGhlIHNhdmVkIGVtYmVkZGluZ3MsIHRoZQpzYXZlZCBpbWFnZS1pZCBvcmRlciwgdGhlIG9wdGlvbmFsIHJlLXJhbmtpbmcgZmlsZSwgYW5kIHRoZSBkYXRhc2V0IGNhcHRpb24KdGFibGUsIHRoZW4gcmVjb21wdXRlcyBhbGwgbWV0cmljcyBieSAqY291bnRpbmcgc3RyaWN0bHkgaGlnaGVyIHNjb3JlcyogKGluc3RlYWQKb2Ygc29ydGluZykgYW5kIGNvbXBhcmVzIHRoZW0gd2l0aCB0ZXN0X3Jlc3VsdHMuanNvbi4KClJlLXJhbmtlZCByYW5rIG9mIGdhbGxlcnkgaXRlbSBnIGZvciBhIHF1ZXJ5IHdpdGggdG9wLUsgc2V0IFQ6CiAgZyBpbiBUICAgICAtPiAje3QgaW4gVCA6IHMyKHQpID4gczIoZyl9CiAgZyBub3QgaW4gVCAtPiBLICsgI3toIG5vdCBpbiBUIDogczEoaCkgPiBzMShnKX0KCkV4aXQgMCBpZmYgYWxsIG1ldHJpY3MgbWF0Y2ggd2l0aGluIC0tdG9sLCB0aGUgaWQgb3JkZXIgaXMgY29uc2lzdGVudCwgYW5kIHRoZXJlIGFyZSBubyB0aWVzLgpVc2FnZTogcHl0aG9uIGV2YWx1YXRlX2luZGVwZW5kZW50LnB5IC0tcnVuIDxydW5fZGlyPiBbLS10b2wgMWUtNl0KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKCmRlZiBjb3VudF9yYW5rKHMxX3JvdywgcG9zaXRpdmVfY29scywgdG9waz1Ob25lLCBzMj1Ob25lKToKICAgICIiIkJlc3QgKG1pbmltdW0pIDAtYmFzZWQgcmFuayBvdmVyIHBvc2l0aXZlIGNvbHVtbnM7IGFsc28gcmV0dXJucyB0aGUgbnVtYmVyIG9mIGV4YWN0IHRpZXMuIiIiCiAgICB0aWVzID0gMAogICAgaWYgdG9wayBpcyBOb25lOgogICAgICAgIGJlc3QgPSBzMV9yb3dbcG9zaXRpdmVfY29sc10ubWF4KCkKICAgICAgICB0aWVzICs9IGludCgoczFfcm93ID09IGJlc3QpLnN1bSgpKSAtIDEKICAgICAgICByZXR1cm4gaW50KChzMV9yb3cgPiBiZXN0KS5zdW0oKSksIHRpZXMKICAgIGluX3RvcCA9IG5wLnplcm9zKHMxX3Jvdy5zaGFwZVswXSwgZHR5cGU9Ym9vbCkKICAgIGluX3RvcFt0b3BrXSA9IFRydWUKICAgIHMyX2Z1bGwgPSBucC5mdWxsKHMxX3Jvdy5zaGFwZVswXSwgbnAubmFuKQogICAgczJfZnVsbFt0b3BrXSA9IHMyCiAgICByYW5rcyA9IFtdCiAgICBmb3IgZyBpbiBwb3NpdGl2ZV9jb2xzOgogICAgICAgIGlmIGluX3RvcFtnXToKICAgICAgICAgICAgcmFua3MuYXBwZW5kKGludCgoczIgPiBzMl9mdWxsW2ddKS5zdW0oKSkpCiAgICAgICAgICAgIHRpZXMgKz0gaW50KChzMiA9PSBzMl9mdWxsW2ddKS5zdW0oKSkgLSAxCiAgICAgICAgZWxzZToKICAgICAgICAgICAgb3V0c2lkZSA9IHMxX3Jvd1t+aW5fdG9wXQogICAgICAgICAgICByYW5rcy5hcHBlbmQobGVuKHRvcGspICsgaW50KChvdXRzaWRlID4gczFfcm93W2ddKS5zdW0oKSkpCiAgICAgICAgICAgIHRpZXMgKz0gaW50KChvdXRzaWRlID09IHMxX3Jvd1tnXSkuc3VtKCkpIC0gMQogICAgcmV0dXJuIG1pbihyYW5rcyksIHRpZXMKCgpkZWYgY29tcHV0ZShzaW0sIG93bmVyLCByZXJhbmspOgogICAgbl9pbWcsIG5fdHh0ID0gc2ltLnNoYXBlCiAgICBpMnQsIHQyaSwgdGllcyA9IFtdLCBbXSwgMAogICAgZm9yIGkgaW4gcmFuZ2Uobl9pbWcpOgogICAgICAgIHBvcyA9IG5wLm5vbnplcm8ob3duZXIgPT0gaSlbMF0KICAgICAgICByLCB0ID0gY291bnRfcmFuayhzaW1baV0sIHBvcywgKigocmVyYW5rWyJpMnRfaWR4Il1baV0sIHJlcmFua1siaTJ0X3Njb3JlIl1baV0pIGlmIHJlcmFuayBlbHNlIChOb25lLCBOb25lKSkpCiAgICAgICAgaTJ0LmFwcGVuZChyKQogICAgICAgIHRpZXMgKz0gdAogICAgZm9yIGMgaW4gcmFuZ2Uobl90eHQpOgogICAgICAgIHIsIHQgPSBjb3VudF9yYW5rKHNpbVs6LCBjXSwgbnAuYXJyYXkoW293bmVyW2NdXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgKigocmVyYW5rWyJ0MmlfaWR4Il1bY10sIHJlcmFua1sidDJpX3Njb3JlIl1bY10pIGlmIHJlcmFuayBlbHNlIChOb25lLCBOb25lKSkpCiAgICAgICAgdDJpLmFwcGVuZChyKQogICAgICAgIHRpZXMgKz0gdAogICAgaTJ0LCB0MmkgPSBucC5hcnJheShpMnQpLCBucC5hcnJheSh0MmkpCiAgICBtID0ge30KICAgIGZvciBuYW1lLCByIGluICgoImkydCIsIGkydCksICgidDJpIiwgdDJpKSk6CiAgICAgICAgZm9yIGsgaW4gKDEsIDUsIDEwKToKICAgICAgICAgICAgbVtmIntuYW1lfV9ye2t9Il0gPSAxMDAuMCAqIG5wLmNvdW50X25vbnplcm8ociA8IGspIC8gbGVuKHIpCiAgICAgICAgbVtmIntuYW1lfV9tZWRyIl0gPSBmbG9hdChucC5tZWRpYW4ociArIDEpKQogICAgICAgIG1bZiJ7bmFtZX1fbWVhbnIiXSA9IGZsb2F0KG5wLm1lYW4ociArIDEpKQogICAgbVsibWVhbl9yZWNhbGwiXSA9IHN1bShtW2Yie2R9X3J7a30iXSBmb3IgZCBpbiAoImkydCIsICJ0MmkiKSBmb3IgayBpbiAoMSwgNSwgMTApKSAvIDYuMAogICAgcmV0dXJuIG0sIHRpZXMKCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcnVuIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10b2wiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTFlLTYpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGF0YS1kaXIiLCBkZWZhdWx0PW9zLnBhdGguam9pbihvcy5lbnZpcm9uLmdldCgiSEVET19XT1JLIiwgIi9jb250ZW50L2hlZG9fd29yayIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImRhdGEiLCAiZmxpY2tyOGsiKSkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKICAgIHJ1biA9IFBhdGgoYXJncy5ydW4pCgogICAgaW1nID0gbnAubG9hZChydW4gLyAidGVzdF9pbWFnZV9lbWJlZGRpbmdzLm5weSIpLmFzdHlwZShucC5mbG9hdDY0KQogICAgdHh0ID0gbnAubG9hZChydW4gLyAidGVzdF90ZXh0X2VtYmVkZGluZ3MubnB5IikuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBzYXZlZF9pZHMgPSBqc29uLmxvYWQob3BlbihydW4gLyAidGVzdF9pbWFnZV9pZHMuanNvbiIsIGVuY29kaW5nPSJ1dGYtOCIpKQogICAgc3RvcmVkID0ganNvbi5sb2FkKG9wZW4ocnVuIC8gInRlc3RfcmVzdWx0cy5qc29uIiwgZW5jb2Rpbmc9InV0Zi04IikpCiAgICByZXJhbmsgPSBkaWN0KG5wLmxvYWQocnVuIC8gInRlc3RfcmVyYW5rLm5weiIpKSBpZiAocnVuIC8gInRlc3RfcmVyYW5rLm5weiIpLmlzX2ZpbGUoKSBlbHNlIE5vbmUKCiAgICB3aXRoIG9wZW4oUGF0aChhcmdzLmRhdGFfZGlyKSAvICJjYXB0aW9uc190ZXN0LmNzdiIsIG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgY2FwdGlvbl9pbWFnZSA9IFtyb3dbImltYWdlX2lkIl0gZm9yIHJvdyBpbiBjc3YuRGljdFJlYWRlcihmKV0KICAgIGltYWdlX29yZGVyID0gbGlzdChkaWN0LmZyb21rZXlzKGNhcHRpb25faW1hZ2UpKQogICAgcHJvYmxlbXMgPSBbXQogICAgaWYgaW1hZ2Vfb3JkZXIgIT0gc2F2ZWRfaWRzOgogICAgICAgIHByb2JsZW1zLmFwcGVuZCgiaW1hZ2UgaWQgb3JkZXIgZGlmZmVycyBmcm9tIGRhdGFzZXQgY2FwdGlvbiB0YWJsZSIpCiAgICBpZiBpbWcuc2hhcGVbMF0gIT0gbGVuKGltYWdlX29yZGVyKSBvciB0eHQuc2hhcGVbMF0gIT0gbGVuKGNhcHRpb25faW1hZ2UpOgogICAgICAgIHByb2JsZW1zLmFwcGVuZChmInNoYXBlIG1pc21hdGNoIGltZz17aW1nLnNoYXBlfSB0eHQ9e3R4dC5zaGFwZX0iKQogICAgaW5kZXhfb2YgPSB7aWlkOiBpIGZvciBpLCBpaWQgaW4gZW51bWVyYXRlKGltYWdlX29yZGVyKX0KICAgIG93bmVyID0gbnAuYXJyYXkoW2luZGV4X29mW2NdIGZvciBjIGluIGNhcHRpb25faW1hZ2VdKQogICAgc2ltID0gaW1nIEAgdHh0LlQKCiAgICBkdWFsLCB0aWVzX2R1YWwgPSBjb21wdXRlKHNpbSwgb3duZXIsIE5vbmUpCiAgICByZWNvbXB1dGVkID0ge2YiZHVhbF97a30iOiB2IGZvciBrLCB2IGluIGR1YWwuaXRlbXMoKX0KICAgIHRpZXMgPSB0aWVzX2R1YWwKICAgIGlmIHJlcmFuayBpcyBub3QgTm9uZToKICAgICAgICBwcmltYXJ5LCB0aWVzX3JyID0gY29tcHV0ZShzaW0sIG93bmVyLCByZXJhbmspCiAgICAgICAgdGllcyArPSB0aWVzX3JyCiAgICBlbHNlOgogICAgICAgIHByaW1hcnkgPSBkdWFsCiAgICByZWNvbXB1dGVkLnVwZGF0ZShwcmltYXJ5KQoKICAgIGRpZmZzID0ge2s6IGFicyhyZWNvbXB1dGVkW2tdIC0gc3RvcmVkW2tdKSBmb3IgayBpbiByZWNvbXB1dGVkfQogICAgd29yc3QgPSBtYXgoZGlmZnMudmFsdWVzKCkpCiAgICBpZiB3b3JzdCA+IGFyZ3MudG9sOgogICAgICAgIHByb2JsZW1zLmFwcGVuZChmIm1ldHJpYyBtaXNtYXRjaCAobWF4IHxkaWZmfD17d29yc3Q6LjNnfSkiKQogICAgaWYgdGllczoKICAgICAgICBwcm9ibGVtcy5hcHBlbmQoZiJ7dGllc30gZXhhY3Qgc2NvcmUgdGllczsgcmFua3MgYXJlIHRpZS1vcmRlciBkZXBlbmRlbnQiKQoKICAgIHJlcG9ydCA9IHsicmVjb21wdXRlZCI6IHJlY29tcHV0ZWQsICJhYnNfZGlmZiI6IGRpZmZzLCAibWF4X2Fic19kaWZmIjogd29yc3QsICJ0b2xlcmFuY2UiOiBhcmdzLnRvbCwKICAgICAgICAgICAgICAicmVyYW5rZWQiOiByZXJhbmsgaXMgbm90IE5vbmUsICJzY29yZV90aWVzIjogdGllcywgInByb2JsZW1zIjogcHJvYmxlbXMsICJwYXNzIjogbm90IHByb2JsZW1zfQogICAgd2l0aCBvcGVuKHJ1biAvICJpbmRlcGVuZGVudF9ldmFsLmpzb24iLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHJlcG9ydCwgZiwgaW5kZW50PTIpCiAgICBwcmludChmIklOREVQRU5ERU5UX0VWQUxVQVRPUjogeydQQVNTJyBpZiBub3QgcHJvYmxlbXMgZWxzZSAnRkFJTCd9IG1heHxkaWZmfD17d29yc3Q6LjNnfSAiCiAgICAgICAgICBmIk1SPXtyZWNvbXB1dGVkWydtZWFuX3JlY2FsbCddOi40Zn0gcmVyYW5rZWQ9e3JlcmFuayBpcyBub3QgTm9uZX0ge3Byb2JsZW1zIGlmIHByb2JsZW1zIGVsc2UgJyd9IikKICAgIHN5cy5leGl0KDAgaWYgbm90IHByb2JsZW1zIGVsc2UgMSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="], "evaluation.py": ["5bf278e6e62668281d9199942d438c8e7e575528e9d672d9a4689c1340951a8d", "IiIiU2hhcmVkIGV2YWx1YXRpb246IGNhY2hlZCBzcGxpdHMsIGR1YWwgZW1iZWRkaW5ncywgZXhjaGFuZ2UgcmUtcmFua2luZywgcnVuIGxvYWRpbmcuIiIiCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIGNvbW1vbiBpbXBvcnQgRkVBVFVSRV9ESVIsIHJlYWRfanNvbgpmcm9tIG1ldHJpY3MgaW1wb3J0IENBUFRJT05TX1BFUl9JTUFHRSwgcmV0cmlldmFsX21ldHJpY3MsIHRvcGtfY2FuZGlkYXRlcwpmcm9tIG1vZGVscyBpbXBvcnQgTW9kZWxDb25maWcsIFJldHJpZXZhbE1vZGVsCgoKY2xhc3MgTnVtZXJpY2FsRmFpbHVyZShSdW50aW1lRXJyb3IpOgogICAgcGFzcwoKCmNsYXNzIENhY2hlZFNwbGl0OgogICAgIiIiQ2FjaGVkIGZlYXR1cmVzIGZvciBvbmUgc3BsaXQuIFBhdGhzIGNhbiBiZSBvdmVycmlkZGVuIChyb2J1c3RuZXNzIGNvcnJ1cHRpb25zKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgc3BsaXQsIGluX21lbW9yeT1UcnVlLCBpbWdfcGF0aD1Ob25lLCB0eHRfcGF0aD1Ob25lLCBtYXNrX3BhdGg9Tm9uZSk6CiAgICAgICAgbW9kZSA9IE5vbmUgaWYgaW5fbWVtb3J5IGVsc2UgInIiCiAgICAgICAgc2VsZi5pbWcgPSBucC5sb2FkKGltZ19wYXRoIG9yIEZFQVRVUkVfRElSIC8gZiJpbWdfe3NwbGl0fS5ucHkiLCBtbWFwX21vZGU9bW9kZSkKICAgICAgICBzZWxmLnR4dCA9IG5wLmxvYWQodHh0X3BhdGggb3IgRkVBVFVSRV9ESVIgLyBmInR4dF97c3BsaXR9Lm5weSIsIG1tYXBfbW9kZT1tb2RlKQogICAgICAgIHNlbGYubWFzayA9IG5wLmxvYWQobWFza19wYXRoIG9yIEZFQVRVUkVfRElSIC8gZiJtYXNrX3tzcGxpdH0ubnB5IikKICAgICAgICBzZWxmLmltYWdlX2lkcyA9IHJlYWRfanNvbihGRUFUVVJFX0RJUiAvICJmZWF0dXJlX21hbmlmZXN0Lmpzb24iKVsic3BsaXRzIl1bc3BsaXRdWyJpbWFnZV9pZHNfb3JkZXIiXQogICAgICAgIGFzc2VydCBzZWxmLnR4dC5zaGFwZVswXSA9PSBDQVBUSU9OU19QRVJfSU1BR0UgKiBzZWxmLmltZy5zaGFwZVswXSA9PSBzZWxmLm1hc2suc2hhcGVbMF0KCiAgICBkZWYgaW1nX3RlbnNvcihzZWxmLCByb3dzLCBkZXZpY2UpOgogICAgICAgIHJldHVybiB0b3JjaC5mcm9tX251bXB5KG5wLmFzYXJyYXkoc2VsZi5pbWdbcm93c10sIGR0eXBlPW5wLmZsb2F0MzIpKS50byhkZXZpY2UsIG5vbl9ibG9ja2luZz1UcnVlKQoKICAgIGRlZiB0eHRfdGVuc29yKHNlbGYsIHJvd3MsIGRldmljZSk6CiAgICAgICAgdHh0ID0gdG9yY2guZnJvbV9udW1weShucC5hc2FycmF5KHNlbGYudHh0W3Jvd3NdLCBkdHlwZT1ucC5mbG9hdDMyKSkudG8oZGV2aWNlLCBub25fYmxvY2tpbmc9VHJ1ZSkKICAgICAgICBtYXNrID0gdG9yY2guZnJvbV9udW1weShucC5hc2FycmF5KHNlbGYubWFza1tyb3dzXSkuYXN0eXBlKGJvb2wpKS50byhkZXZpY2UsIG5vbl9ibG9ja2luZz1UcnVlKQogICAgICAgIHJldHVybiB0eHQsIG1hc2sKCiAgICBkZWYgYmF0Y2goc2VsZiwgaW1hZ2Vfcm93cywgZGV2aWNlKToKICAgICAgICBpbWFnZV9yb3dzID0gbnAuc29ydChucC5hc2FycmF5KGltYWdlX3Jvd3MpKQogICAgICAgIGNhcF9yb3dzID0gKGltYWdlX3Jvd3NbOiwgTm9uZV0gKiBDQVBUSU9OU19QRVJfSU1BR0UgKyBucC5hcmFuZ2UoQ0FQVElPTlNfUEVSX0lNQUdFKSkucmVzaGFwZSgtMSkKICAgICAgICB0eHQsIG1hc2sgPSBzZWxmLnR4dF90ZW5zb3IoY2FwX3Jvd3MsIGRldmljZSkKICAgICAgICBwYWlyID0gdG9yY2guYXJhbmdlKGxlbihpbWFnZV9yb3dzKSwgZGV2aWNlPWRldmljZSkucmVwZWF0X2ludGVybGVhdmUoQ0FQVElPTlNfUEVSX0lNQUdFKQogICAgICAgIHJldHVybiBzZWxmLmltZ190ZW5zb3IoaW1hZ2Vfcm93cywgZGV2aWNlKSwgdHh0LCBtYXNrLCBwYWlyLCBjYXBfcm93cwoKCmRlZiBfc2xpY2VzKG4sIHNpemUpOgogICAgcmV0dXJuIFtucC5hcmFuZ2UocywgbWluKG4sIHMgKyBzaXplKSkgZm9yIHMgaW4gcmFuZ2UoMCwgbiwgc2l6ZSldCgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgcGFzczFfYWxsKG1vZGVsLCBkYXRhLCBkZXZpY2UsIGNodW5rPTI1MCwgaW1nX3RyYW5zZm9ybT1Ob25lKToKICAgICIiIkR1YWwgZW1iZWRkaW5ncyBhbmQgdGhlIHBhc3MtMSBzdGF0ZSBuZWVkZWQgZm9yIGV4Y2hhbmdlIChrZXB0IG9uIGRldmljZSkuIiIiCiAgICBtb2RlbC5ldmFsKCkKICAgIGtlZXAgPSAoIngiLCAic3RhdGVzIiwgImNtYXNrIiwgInoiLCAiel9wb29sZWQiKQogICAgaW1nX2VtYiwgdHh0X2VtYiwgaW1nX2F1eCwgdHh0X2F1eCA9IFtdLCBbXSwge2s6IFtdIGZvciBrIGluIGtlZXB9LCB7azogW10gZm9yIGsgaW4ga2VlcH0KICAgIG1hc2tzID0gW10KICAgIGZvciByb3dzIGluIF9zbGljZXMoZGF0YS5pbWcuc2hhcGVbMF0sIGNodW5rKToKICAgICAgICBmZWF0cyA9IGRhdGEuaW1nX3RlbnNvcihyb3dzLCBkZXZpY2UpCiAgICAgICAgaWYgaW1nX3RyYW5zZm9ybSBpcyBub3QgTm9uZToKICAgICAgICAgICAgZmVhdHMgPSBpbWdfdHJhbnNmb3JtKGZlYXRzLCByb3dzKQogICAgICAgIHosIGF1eCwgXyA9IG1vZGVsLnBhc3MxKCJpbWciLCBmZWF0cywgTm9uZSwgc2FtcGxlPUZhbHNlKQogICAgICAgIGltZ19lbWIuYXBwZW5kKHopCiAgICAgICAgZm9yIGsgaW4ga2VlcDoKICAgICAgICAgICAgaWYgayBpbiBhdXg6CiAgICAgICAgICAgICAgICBpbWdfYXV4W2tdLmFwcGVuZChhdXhba10pCiAgICBmb3Igcm93cyBpbiBfc2xpY2VzKGRhdGEudHh0LnNoYXBlWzBdLCAyICogY2h1bmspOgogICAgICAgIGZlYXRzLCBtYXNrID0gZGF0YS50eHRfdGVuc29yKHJvd3MsIGRldmljZSkKICAgICAgICB6LCBhdXgsIF8gPSBtb2RlbC5wYXNzMSgidHh0IiwgZmVhdHMsIG1hc2ssIHNhbXBsZT1GYWxzZSkKICAgICAgICB0eHRfZW1iLmFwcGVuZCh6KQogICAgICAgIG1hc2tzLmFwcGVuZChtYXNrKQogICAgICAgIGZvciBrIGluIGtlZXA6CiAgICAgICAgICAgIGlmIGsgaW4gYXV4OgogICAgICAgICAgICAgICAgdHh0X2F1eFtrXS5hcHBlbmQoYXV4W2tdKQogICAgaW1nID0gdG9yY2guY2F0KGltZ19lbWIpLmZsb2F0KCkKICAgIHR4dCA9IHRvcmNoLmNhdCh0eHRfZW1iKS5mbG9hdCgpCiAgICBpZiBub3QgKHRvcmNoLmlzZmluaXRlKGltZykuYWxsKCkgYW5kIHRvcmNoLmlzZmluaXRlKHR4dCkuYWxsKCkpOgogICAgICAgIHJhaXNlIE51bWVyaWNhbEZhaWx1cmUoIm5vbi1maW5pdGUgZW1iZWRkaW5ncyBkdXJpbmcgZXZhbHVhdGlvbiIpCiAgICBjYWNoZSA9IE5vbmUKICAgIGlmIG1vZGVsLmNmZy5leGNoYW5nZXM6CiAgICAgICAgY2FjaGUgPSB7ImltZyI6IHtrOiB0b3JjaC5jYXQodikgZm9yIGssIHYgaW4gaW1nX2F1eC5pdGVtcygpIGlmIHZ9LAogICAgICAgICAgICAgICAgICJ0eHQiOiB7azogdG9yY2guY2F0KHYpIGZvciBrLCB2IGluIHR4dF9hdXguaXRlbXMoKSBpZiB2fSwKICAgICAgICAgICAgICAgICAidHh0X21hc2siOiB0b3JjaC5jYXQobWFza3MpfQogICAgcmV0dXJuIGltZy5jcHUoKS5udW1weSgpLCB0eHQuY3B1KCkubnVtcHkoKSwgY2FjaGUKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBleGNoYW5nZV9zY29yZXMobW9kZWwsIGNhY2hlLCBpbWdfcm93cywgdHh0X3Jvd3MsIGRldmljZSwgYmF0Y2g9NTEyLCBtZXNzYWdlX3NjYWxlPSgxLjAsIDEuMCkpOgogICAgb3V0ID0gW10KICAgIGZvciBzIGluIHJhbmdlKDAsIGxlbihpbWdfcm93cyksIGJhdGNoKToKICAgICAgICBpciA9IHRvcmNoLmFzX3RlbnNvcihpbWdfcm93c1tzOnMgKyBiYXRjaF0sIGRldmljZT1kZXZpY2UpCiAgICAgICAgdHIgPSB0b3JjaC5hc190ZW5zb3IodHh0X3Jvd3NbczpzICsgYmF0Y2hdLCBkZXZpY2U9ZGV2aWNlKQogICAgICAgIGFpID0ge2s6IHZbaXJdIGZvciBrLCB2IGluIGNhY2hlWyJpbWciXS5pdGVtcygpfQogICAgICAgIGF0ID0ge2s6IHZbdHJdIGZvciBrLCB2IGluIGNhY2hlWyJ0eHQiXS5pdGVtcygpfQogICAgICAgIG91dC5hcHBlbmQobW9kZWwucGFpcl9zaW1pbGFyaXR5KGFpLCBhdCwgY2FjaGVbInR4dF9tYXNrIl1bdHJdLCBtZXNzYWdlX3NjYWxlKS5mbG9hdCgpKQogICAgc2NvcmVzID0gdG9yY2guY2F0KG91dCkuY3B1KCkubnVtcHkoKQogICAgaWYgbm90IG5wLmlzZmluaXRlKHNjb3JlcykuYWxsKCk6CiAgICAgICAgcmFpc2UgTnVtZXJpY2FsRmFpbHVyZSgibm9uLWZpbml0ZSBleGNoYW5nZSBzY29yZXMiKQogICAgcmV0dXJuIHNjb3JlcwoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIHJlcmFuayhtb2RlbCwgaW1nLCB0eHQsIGNhY2hlLCBkZXZpY2UsIGspOgogICAgaTJ0X2lkeCwgdDJpX2lkeCA9IHRvcGtfY2FuZGlkYXRlcyhpbWcsIHR4dCwgaykKICAgIG5faW1nLCBuX3R4dCA9IGltZy5zaGFwZVswXSwgdHh0LnNoYXBlWzBdCiAgICBwYWlyX2tleXMgPSBucC51bmlxdWUobnAuY29uY2F0ZW5hdGUoWwogICAgICAgIChucC5hcmFuZ2Uobl9pbWcpWzosIE5vbmVdICogbl90eHQgKyBpMnRfaWR4KS5yZXNoYXBlKC0xKSwKICAgICAgICAodDJpX2lkeCAqIG5fdHh0ICsgbnAuYXJhbmdlKG5fdHh0KVs6LCBOb25lXSkucmVzaGFwZSgtMSldKSkKICAgIHNjb3JlcyA9IGV4Y2hhbmdlX3Njb3Jlcyhtb2RlbCwgY2FjaGUsIHBhaXJfa2V5cyAvLyBuX3R4dCwgcGFpcl9rZXlzICUgbl90eHQsIGRldmljZSkKICAgIGxvb2t1cCA9IGRpY3QoemlwKHBhaXJfa2V5cy50b2xpc3QoKSwgc2NvcmVzLnRvbGlzdCgpKSkKICAgIGkydF9zY29yZSA9IG5wLmFycmF5KFtbbG9va3VwW2kgKiBuX3R4dCArIHRdIGZvciB0IGluIHJvd10gZm9yIGksIHJvdyBpbiBlbnVtZXJhdGUoaTJ0X2lkeCldLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgdDJpX3Njb3JlID0gbnAuYXJyYXkoW1tsb29rdXBbaSAqIG5fdHh0ICsgY10gZm9yIGkgaW4gcm93XSBmb3IgYywgcm93IGluIGVudW1lcmF0ZSh0MmlfaWR4KV0sIGR0eXBlPW5wLmZsb2F0NjQpCiAgICByZXR1cm4geyJpMnRfaWR4IjogaTJ0X2lkeCwgImkydF9zY29yZSI6IGkydF9zY29yZSwgInQyaV9pZHgiOiB0MmlfaWR4LCAidDJpX3Njb3JlIjogdDJpX3Njb3JlfQoKCmRlZiBldmFsdWF0ZShtb2RlbCwgZGF0YSwgZGV2aWNlLCByZXJhbmtfaz0xNiwgaW1nX3RyYW5zZm9ybT1Ob25lKToKICAgIGltZywgdHh0LCBjYWNoZSA9IHBhc3MxX2FsbChtb2RlbCwgZGF0YSwgZGV2aWNlLCBpbWdfdHJhbnNmb3JtPWltZ190cmFuc2Zvcm0pCiAgICByciA9IHJlcmFuayhtb2RlbCwgaW1nLCB0eHQsIGNhY2hlLCBkZXZpY2UsIHJlcmFua19rKSBpZiBjYWNoZSBpcyBub3QgTm9uZSBhbmQgcmVyYW5rX2sgPiAwIGVsc2UgTm9uZQogICAgcmV0dXJuIHJldHJpZXZhbF9tZXRyaWNzKGltZywgdHh0LCByciksIGltZywgdHh0LCByciwgY2FjaGUKCgpkZWYgbG9hZF9ydW5fbW9kZWwocnVuX2RpciwgZGV2aWNlLCBjaGVja3BvaW50PSJiZXN0X21vZGVsLnB0Iik6CiAgICBja3B0ID0gdG9yY2gubG9hZChydW5fZGlyIC8gY2hlY2twb2ludCwgbWFwX2xvY2F0aW9uPWRldmljZSwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICBtb2RlbCA9IFJldHJpZXZhbE1vZGVsKE1vZGVsQ29uZmlnKCoqY2twdFsiY29uZmlnIl0pKS50byhkZXZpY2UpCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2twdFsibW9kZWwiXSkKICAgIHJldHVybiBtb2RlbC5ldmFsKCkK"], "features.py": ["fcd47f8fff7b144e09a9056a63ab1b1a0afe60d7d962c8c05242937ce8c7632f", "IiIiQ2FjaGUgZnJvemVuIGJhY2tib25lIGZlYXR1cmVzIG9uY2UgKEZQMzIgY29tcHV0ZSwgRlAxNiBzdG9yYWdlKS4KCldyaXRlcyB0byBGRUFUVVJFX0RJUiwgcGVyIHNwbGl0IHMgaW4ge3RyYWluLCB2YWwsIHRlc3R9OgogIGltZ197c30ubnB5ICAgIFtOX2ltYWdlcywgMTk2LCA3NjhdIGZsb2F0MTYgICAoaW1hZ2UgaSA9IHJvdyBpIG9mIG1hbmlmZXN0IG9yZGVyKQogIHR4dF97c30ubnB5ICAgIFtOX2NhcHRpb25zLCA2NCwgNzY4XSBmbG9hdDE2ICAoY2FwdGlvbiA1aStrIGJlbG9uZ3MgdG8gaW1hZ2UgaSkKICBtYXNrX3tzfS5ucHkgICBbTl9jYXB0aW9ucywgNjRdIHVpbnQ4CiAgaWRzX3tzfS5ucHkgICAgW05fY2FwdGlvbnMsIDY0XSBpbnQzMiAgICAgICAgICh0b2tlbiBpZHMsIGZvciBlbmQtdG8tZW5kIHByb2ZpbGluZykKICBzYWxpZW5jeV97c30ubnB5IFtOX2ltYWdlcywgMTk2XSBmbG9hdDMyICAgICAgKGxhc3QtYmxvY2sgQ0xTLT5wYXRjaCBhdHRlbnRpb247IEgxIHByb2JlIG9ubHkpCiAgZmVhdHVyZV9tYW5pZmVzdC5qc29uICBzaGFwZXMsIGZpbGUgU0hBMjU2LCBiYWNrYm9uZSBpZGVudGl0aWVzLCBwcmVwcm9jZXNzaW5nLCBGUDE2IHJhbmdlIGNoZWNrCiAgcm9iZXJ0YV9sb2FkaW5nX2luZm8uanNvbgoKQmVjYXVzZSBib3RoIGJhY2tib25lcyBhcmUgZnJvemVuIGFuZCBpbiBldmFsIG1vZGUgKG5vIGF1Z21lbnRhdGlvbiksIGNhY2hpbmcgaXMKZXhhY3RseSBlcXVpdmFsZW50IHRvIHJlY29tcHV0aW5nIHRoZW0gZXZlcnkgc3RlcC4KClVzYWdlOiBweXRob24gZmVhdHVyZXMucHkgWy0tYmF0Y2gtc2l6ZSA2NF0KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHRpbWUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBEYXRhc2V0Cgpmcm9tIGJhY2tib25lcyBpbXBvcnQgKE1BWF9URVhUX0xFTiwgUk9CRVJUQV9OQU1FLCBST0JFUlRBX1JFVklTSU9OLCBWSVRfV0VJR0hUUywgRnJvemVuUm9CRVJUYSwgRnJvemVuVmlULAogICAgICAgICAgICAgICAgICAgICAgIGxvYWRfdG9rZW5pemVyLCB0b2tlbml6ZSwgdml0X3RyYW5zZm9ybSkKZnJvbSBjb21tb24gaW1wb3J0IChEQVRBX0RJUiwgRkVBVFVSRV9ESVIsIGJhbm5lciwgZmVhdHVyZV9jb2RlX3NoYTI1NiwgcHJvdmVuYW5jZSwgcmVhZF9qc29uLCBzaGEyNTZfZmlsZSwKICAgICAgICAgICAgICAgICAgICB3cml0ZV9qc29uKQoKRlAxNl9NQVggPSA2NTUwNC4wCgoKY2xhc3MgSW1hZ2VGaWxlcyhEYXRhc2V0KToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbWFnZV9kaXIsIGlkcyk6CiAgICAgICAgc2VsZi5pbWFnZV9kaXIsIHNlbGYuaWRzLCBzZWxmLnRmID0gaW1hZ2VfZGlyLCBpZHMsIHZpdF90cmFuc2Zvcm0oKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi5pZHMpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGkpOgogICAgICAgIHdpdGggSW1hZ2Uub3BlbihmIntzZWxmLmltYWdlX2Rpcn0ve3NlbGYuaWRzW2ldfSIpIGFzIGltOgogICAgICAgICAgICByZXR1cm4gc2VsZi50ZihpbS5jb252ZXJ0KCJSR0IiKSkKCgpkZWYgc3RvcmUoYXJyX3BhdGgsIHNoYXBlLCBiYXRjaGVzKToKICAgIG91dCA9IG5wLmxpYi5mb3JtYXQub3Blbl9tZW1tYXAoYXJyX3BhdGgsIG1vZGU9IncrIiwgZHR5cGU9bnAuZmxvYXQxNiwgc2hhcGU9c2hhcGUpCiAgICBwb3MsIGFic21heCA9IDAsIDAuMAogICAgZm9yIGZlYXRzIGluIGJhdGNoZXM6CiAgICAgICAgZmVhdHMgPSBmZWF0cy5mbG9hdCgpCiAgICAgICAgaWYgbm90IHRvcmNoLmlzZmluaXRlKGZlYXRzKS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKGYibm9uLWZpbml0ZSBiYWNrYm9uZSBmZWF0dXJlcyB3aGlsZSB3cml0aW5nIHthcnJfcGF0aC5uYW1lfSIpCiAgICAgICAgYWJzbWF4ID0gbWF4KGFic21heCwgZmxvYXQoZmVhdHMuYWJzKCkubWF4KCkpKQogICAgICAgIGlmIGFic21heCA+PSBGUDE2X01BWDoKICAgICAgICAgICAgcmFpc2UgT3ZlcmZsb3dFcnJvcihmInthcnJfcGF0aC5uYW1lfTogfHh8PXthYnNtYXh9IGRvZXMgbm90IGZpdCBmbG9hdDE2IikKICAgICAgICBvdXRbcG9zOnBvcyArIGZlYXRzLnNoYXBlWzBdXSA9IGZlYXRzLmNwdSgpLm51bXB5KCkuYXN0eXBlKG5wLmZsb2F0MTYpCiAgICAgICAgcG9zICs9IGZlYXRzLnNoYXBlWzBdCiAgICBhc3NlcnQgcG9zID09IHNoYXBlWzBdLCAocG9zLCBzaGFwZSkKICAgIG91dC5mbHVzaCgpCiAgICBkZWwgb3V0CiAgICByZXR1cm4gYWJzbWF4CgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD02NCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1udW0td29ya2VycyIsIHR5cGU9aW50LCBkZWZhdWx0PTIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgYmFubmVyKCJGUk9aRU4gQkFDS0JPTkUgRkVBVFVSRSBDQUNIRSIpCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiKQogICAgbWFuaWZlc3QgPSByZWFkX2pzb24oREFUQV9ESVIgLyAibWFuaWZlc3QuanNvbiIpCiAgICBGRUFUVVJFX0RJUi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgdml0ID0gRnJvemVuVmlUKCkudG8oZGV2aWNlKQogICAgcm9iZXJ0YSA9IEZyb3plblJvQkVSVGEoKS50byhkZXZpY2UpCiAgICB0b2tlbml6ZXIgPSBsb2FkX3Rva2VuaXplcigpCiAgICB3cml0ZV9qc29uKEZFQVRVUkVfRElSIC8gInJvYmVydGFfbG9hZGluZ19pbmZvLmpzb24iLAogICAgICAgICAgICAgICB7ImxvYWRpbmdfaW5mbyI6IHJvYmVydGEubG9hZGluZ19pbmZvLCAiY29tbWl0X2hhc2giOiByb2JlcnRhLmNvbW1pdF9oYXNofSkKICAgIHByaW50KCJSb0JFUlRhIGNvbW1pdDoiLCByb2JlcnRhLmNvbW1pdF9oYXNoKQogICAgcHJpbnQoIlJvQkVSVGEgdW5leHBlY3RlZCBrZXlzIChleHBlY3RlZCBsbV9oZWFkLiogb25seSk6IiwgbGVuKHJvYmVydGEubG9hZGluZ19pbmZvLmdldCgidW5leHBlY3RlZF9rZXlzIiwgW10pKSkKCiAgICByZWNvcmQgPSB7InNwbGl0cyI6IHt9LCAiYWJzbWF4Ijoge319CiAgICBmb3Igc3BsaXQgaW4gKCJ0cmFpbiIsICJ2YWwiLCAidGVzdCIpOgogICAgICAgIHRhYmxlID0gcGQucmVhZF9jc3YoREFUQV9ESVIgLyBmImNhcHRpb25zX3tzcGxpdH0uY3N2Iiwga2VlcF9kZWZhdWx0X25hPUZhbHNlKQogICAgICAgIGltYWdlX2lkcyA9IHRhYmxlWyJpbWFnZV9pZCJdLmlsb2NbOjo1XS50b2xpc3QoKQogICAgICAgIHQwID0gdGltZS50aW1lKCkKCiAgICAgICAgbG9hZGVyID0gRGF0YUxvYWRlcihJbWFnZUZpbGVzKG1hbmlmZXN0WyJpbWFnZV9kaXIiXSwgaW1hZ2VfaWRzKSwgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoX3NpemUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzaHVmZmxlPUZhbHNlLCBudW1fd29ya2Vycz1hcmdzLm51bV93b3JrZXJzLCBwaW5fbWVtb3J5PVRydWUpCiAgICAgICAgaW1nX3BhdGggPSBGRUFUVVJFX0RJUiAvIGYiaW1nX3tzcGxpdH0ubnB5IgogICAgICAgIHNhbGllbmN5ID0gW10KCiAgICAgICAgZGVmIGltYWdlX2JhdGNoZXMoKToKICAgICAgICAgICAgZm9yIGksIHggaW4gZW51bWVyYXRlKGxvYWRlcik6CiAgICAgICAgICAgICAgICBmZWF0cywgc2FsID0gdml0LmZvcndhcmRfd2l0aF9zYWxpZW5jeSh4LnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpKQogICAgICAgICAgICAgICAgaWYgaSA9PSAwOiAgIyB0aGUgc2FsaWVuY3kgcGF0aCBtdXN0IHJlcHJvZHVjZSB0aGUgb2ZmaWNpYWwgZm9yd2FyZCBleGFjdGx5CiAgICAgICAgICAgICAgICAgICAgcmVmID0gdml0KHgudG8oZGV2aWNlKSkKICAgICAgICAgICAgICAgICAgICBhc3NlcnQgdG9yY2guYWxsY2xvc2UocmVmLCBmZWF0cywgYXRvbD0xZS01KSwgZmxvYXQoKHJlZiAtIGZlYXRzKS5hYnMoKS5tYXgoKSkKICAgICAgICAgICAgICAgIHNhbGllbmN5LmFwcGVuZChzYWwuZmxvYXQoKS5jcHUoKS5udW1weSgpKQogICAgICAgICAgICAgICAgeWllbGQgZmVhdHMKCiAgICAgICAgcmVjb3JkWyJhYnNtYXgiXVtmImltZ197c3BsaXR9Il0gPSBzdG9yZShpbWdfcGF0aCwgKGxlbihpbWFnZV9pZHMpLCAxOTYsIDc2OCksIGltYWdlX2JhdGNoZXMoKSkKICAgICAgICBucC5zYXZlKEZFQVRVUkVfRElSIC8gZiJzYWxpZW5jeV97c3BsaXR9Lm5weSIsIG5wLmNvbmNhdGVuYXRlKHNhbGllbmN5KS5hc3R5cGUobnAuZmxvYXQzMikpCgogICAgICAgIHRvayA9IHRva2VuaXplKHRva2VuaXplciwgdGFibGVbImNhcHRpb24iXS50b2xpc3QoKSkKICAgICAgICBpZHMgPSB0b2tbImlucHV0X2lkcyJdLnRvKHRvcmNoLmludDMyKQogICAgICAgIG1hc2sgPSB0b2tbImF0dGVudGlvbl9tYXNrIl0udG8odG9yY2gudWludDgpCiAgICAgICAgbnAuc2F2ZShGRUFUVVJFX0RJUiAvIGYiaWRzX3tzcGxpdH0ubnB5IiwgaWRzLm51bXB5KCkpCiAgICAgICAgbnAuc2F2ZShGRUFUVVJFX0RJUiAvIGYibWFza197c3BsaXR9Lm5weSIsIG1hc2subnVtcHkoKSkKICAgICAgICB0cnVuY2F0ZWQgPSBpbnQoKHRva1siYXR0ZW50aW9uX21hc2siXS5zdW0oMSkgPT0gTUFYX1RFWFRfTEVOKS5zdW0oKSkKCiAgICAgICAgZGVmIHRleHRfYmF0Y2hlcygpOgogICAgICAgICAgICBmb3IgcyBpbiByYW5nZSgwLCBsZW4odGFibGUpLCAyNTYpOgogICAgICAgICAgICAgICAgeWllbGQgcm9iZXJ0YShpZHNbczpzICsgMjU2XS5sb25nKCkudG8oZGV2aWNlKSwgbWFza1tzOnMgKyAyNTZdLmxvbmcoKS50byhkZXZpY2UpKQoKICAgICAgICB0eHRfcGF0aCA9IEZFQVRVUkVfRElSIC8gZiJ0eHRfe3NwbGl0fS5ucHkiCiAgICAgICAgcmVjb3JkWyJhYnNtYXgiXVtmInR4dF97c3BsaXR9Il0gPSBzdG9yZSh0eHRfcGF0aCwgKGxlbih0YWJsZSksIE1BWF9URVhUX0xFTiwgNzY4KSwgdGV4dF9iYXRjaGVzKCkpCgogICAgICAgIHJlY29yZFsic3BsaXRzIl1bc3BsaXRdID0gewogICAgICAgICAgICAiaW1hZ2VzIjogbGVuKGltYWdlX2lkcyksCiAgICAgICAgICAgICJjYXB0aW9ucyI6IGxlbih0YWJsZSksCiAgICAgICAgICAgICJjYXB0aW9uc19hdF9tYXhfbGVuX3Bvc3NpYmx5X3RydW5jYXRlZCI6IHRydW5jYXRlZCwKICAgICAgICAgICAgImltYWdlX2lkc19vcmRlciI6IGltYWdlX2lkcywKICAgICAgICAgICAgInNoYTI1NiI6IHtwLm5hbWU6IHNoYTI1Nl9maWxlKHApIGZvciBwIGluIFtpbWdfcGF0aCwgdHh0X3BhdGgsIEZFQVRVUkVfRElSIC8gZiJtYXNrX3tzcGxpdH0ubnB5IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBGRUFUVVJFX0RJUiAvIGYiaWRzX3tzcGxpdH0ubnB5Il19LAogICAgICAgIH0KICAgICAgICBwcmludChmIntzcGxpdH06IHtsZW4oaW1hZ2VfaWRzKX0gaW1hZ2VzLCB7bGVuKHRhYmxlKX0gY2FwdGlvbnMsIHt0aW1lLnRpbWUoKSAtIHQwOi4wZn1zLCAiCiAgICAgICAgICAgICAgZiJ0cnVuY2F0ZWQ9e3RydW5jYXRlZH0iKQoKICAgIHJlY29yZC51cGRhdGUoewogICAgICAgICJ2aXRfd2VpZ2h0cyI6IHN0cihWSVRfV0VJR0hUUyksCiAgICAgICAgInZpdF90cmFuc2Zvcm0iOiByZXByKHZpdF90cmFuc2Zvcm0oKSksCiAgICAgICAgInJvYmVydGEiOiB7Im5hbWUiOiBST0JFUlRBX05BTUUsICJyZXF1ZXN0ZWRfcmV2aXNpb24iOiBST0JFUlRBX1JFVklTSU9OLCAiY29tbWl0X2hhc2giOiByb2JlcnRhLmNvbW1pdF9oYXNofSwKICAgICAgICAiYmFja2JvbmVfY29tcHV0ZV9kdHlwZSI6ICJmbG9hdDMyIiwKICAgICAgICAic3RvcmFnZV9kdHlwZSI6ICJmbG9hdDE2IiwKICAgICAgICAibWF4X3RleHRfbGVuIjogTUFYX1RFWFRfTEVOLAogICAgICAgICJ2aXRfdG9rZW5fb3V0cHV0IjogIjE5NiBwYXRjaCB0b2tlbnMgYWZ0ZXIgZW5jb2RlciBMYXllck5vcm0gKGNsYXNzIHRva2VuIGRyb3BwZWQpIiwKICAgICAgICAidGV4dF90b2tlbl9vdXRwdXQiOiAibGFzdF9oaWRkZW5fc3RhdGUsIHJpZ2h0IHBhZGRlZCIsCiAgICAgICAgImRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2Ijogc2hhMjU2X2ZpbGUoREFUQV9ESVIgLyAibWFuaWZlc3QuanNvbiIpLAogICAgICAgICJmZWF0dXJlX2NvZGVfc2hhMjU2IjogZmVhdHVyZV9jb2RlX3NoYTI1NigpLAogICAgfSkKICAgIHJlY29yZFsicHJvdmVuYW5jZSJdID0gcHJvdmVuYW5jZShpbmNsdWRlX2FydGlmYWN0cz1GYWxzZSkKICAgIHdyaXRlX2pzb24oRkVBVFVSRV9ESVIgLyAiZmVhdHVyZV9tYW5pZmVzdC5qc29uIiwgcmVjb3JkKQogICAgcHJpbnQoIkZFQVRVUkVfQ0FDSEU6IFBBU1MiKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"], "gate.py": ["4585f22c3da8384e23497f3133a6058be6efb1d1d643825252c1205e6a527374", "IiIiUHJlLXRyYWluaW5nIGdhdGUgb24gcmVhbCBjYWNoZWQgRmxpY2tyOGsgZmVhdHVyZXMuIE11c3QgcGFzcyBiZWZvcmUgYW55IHJ1bi4KCiAxLiBvZmZpY2lhbCBtYW1iYV9zc20gTWFtYmEyIGlzIHRoZSBjbGFzcyB1c2VkIGJ5IGV2ZXJ5IE1hbWJhLTIgdmFyaWFudAogMi4gY2h1bmstYm91bmRhcnkgaW5kaWNlcyBmb3IgbGVuZ3RocyAxLi4zMyBhbmQgcGFkZGVkIGJhdGNoZXMKIDMuIFUgeCBOIG11bHRpLXBvc2l0aXZlIEluZm9OQ0UgZ2VvbWV0cnkgKDggaW1hZ2VzIHggNDAgY2FwdGlvbnMpCiA0LiBldmVyeSB2YXJpYW50OiByZWFsLWJhdGNoIGZvcndhcmQvYmFja3dhcmQgZmluaXRlLCBldmVyeSB0cmFpbmFibGUgcGFyYW1ldGVyIHJlY2VpdmVzIGEgZ3JhZGllbnQKIDUuIE1hbWJhLTIgZnVzZWQgKG1lbS1lZmZpY2llbnQpIHBhdGggdnMgcmVmZXJlbmNlIHBhdGggb24gcmVhbCBpbnB1dHMsIGZvcndhcmQgYW5kIGdyYWRpZW50CiA2LiBwYWRkaW5nIGludmFyaWFuY2Ugb2YgdGhlIHRleHQgYnJhbmNoIChmdWxsIDY0LXRva2VuIHBhZGRpbmcgdnMgdHJ1bmNhdGVkIHRvIGNhcHRpb24gbGVuZ3RoKQogNy4gbGVnYWN5IEhFRE8gaXMgYWZmaW5lICh3aHkgaXQgd2FzIHJlcGxhY2VkKQogOC4gVGhlb3JlbSAxOiBFbmVyZ3lIRURPIGVuZXJneSBuZXZlciBpbmNyZWFzZXMgKGZsb2F0NjQsIHRyYWluZWQtc2NhbGUgYW5kIGFkdmVyc2FyaWFsIHBhcmFtZXRlcnMsIHJlYWwgdG9rZW5zKQogOS4gZXhjaGFuZ2UgSFZTQzogZXhhY3Qgbm8tb3AgYXQgaW5pdGlhbGlzYXRpb24gKHNjb3JlID09IHBhc3MtMSBjb3NpbmUpOyBkZXBlbmRzIG9uIHRoZSBwYXJ0bmVyIG9uY2UgZ2F0ZWQKMTAuIHBhcmFtZXRlciBjb3VudHMgZm9yIGFsbCB2YXJpYW50czsgbWF0Y2hlZCBjb250cm9scyB3aXRoaW4gMSUKCldyaXRlcyBSRVBPUlRfRElSL2dhdGVfcmVwb3J0Lmpzb24gYW5kIHBhcmFtX2NvdW50cy5qc29uLiBFeGl0IDEgb24gYW55IGZhaWx1cmUuCiIiIgoKaW1wb3J0IHN5cwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAoKZnJvbSBjb21tb24gaW1wb3J0IEZFQVRVUkVfRElSLCBSRVBPUlRfRElSLCBiYW5uZXIsIGVudmlyb25tZW50X21hbmlmZXN0LCBwcm92ZW5hbmNlLCBzZXRfc2VlZCwgd3JpdGVfanNvbgpmcm9tIG1vZGVscyBpbXBvcnQgKEhFRE8sIFBST1BPU0VELCBWQVJJQU5UUywgRW5lcmd5SEVETywgUmV0cmlldmFsTW9kZWwsIGFzc2VydF9uYXRpdmVfbWFtYmEyLCBidWlsZF9jb25maWcsCiAgICAgICAgICAgICAgICAgICAgY2h1bmtfYm91bmRhcmllcywgY291bnRfdHJhaW5hYmxlLCBtdWx0aXBvc2l0aXZlX2luZm9uY2UsIHRvdGFsX2xvc3MpCgpyZXBvcnQgPSB7fQoKCmRlZiBjaGVjayhuYW1lLCBvaywgZGV0YWlsPU5vbmUpOgogICAgcmVwb3J0W25hbWVdID0geyJwYXNzIjogYm9vbChvayksICJkZXRhaWwiOiBkZXRhaWx9CiAgICBwcmludChmIlt7J1BBU1MnIGlmIG9rIGVsc2UgJ0ZBSUwnfV0ge25hbWV9IiArIChmIjoge2RldGFpbH0iIGlmIGRldGFpbCBpcyBub3QgTm9uZSBlbHNlICIiKSwgZmx1c2g9VHJ1ZSkKCgpkZWYgbWFpbigpOgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIikKICAgIHNldF9zZWVkKDApCiAgICBiYW5uZXIoIlBSRS1UUkFJTklORyBHQVRFIikKCiAgICBmcm9tIG1hbWJhX3NzbSBpbXBvcnQgTWFtYmEyCiAgICBjaGVjaygibWFtYmEyX21vZHVsZSIsIE1hbWJhMi5fX21vZHVsZV9fID09ICJtYW1iYV9zc20ubW9kdWxlcy5tYW1iYTIiLCBNYW1iYTIuX19tb2R1bGVfXykKCiAgICBleHBlY3RlZCA9IHsxOiBbMF0sIDc6IFs2XSwgODogWzddLCA5OiBbOF0sIDE1OiBbMTRdLCAxNjogWzE1XSwgMTc6IFsxNSwgMTZdLCAzMTogWzE1LCAzMF0sIDMyOiBbMTUsIDMxXSwKICAgICAgICAgICAgICAgIDMzOiBbMTUsIDMxLCAzMl19CiAgICBvaywgZGV0YWlsID0gVHJ1ZSwge30KICAgIGZvciBMLCB3YW50IGluIGV4cGVjdGVkLml0ZW1zKCk6CiAgICAgICAgeSA9IHRvcmNoLmFyYW5nZShMLCBkdHlwZT10b3JjaC5mbG9hdDMyKS52aWV3KDEsIEwsIDEpCiAgICAgICAgc3RhdGVzLCBjbWFzayA9IGNodW5rX2JvdW5kYXJpZXMoeSwgdG9yY2gub25lcygxLCBMLCBkdHlwZT10b3JjaC5ib29sKSwgMTYpCiAgICAgICAgZ290ID0gc3RhdGVzLnZpZXcoLTEpW2NtYXNrLnZpZXcoLTEpID4gMF0ubG9uZygpLnRvbGlzdCgpCiAgICAgICAgZGV0YWlsW0xdID0gZ290CiAgICAgICAgb2sgJj0gZ290ID09IHdhbnQKICAgIHkgPSB0b3JjaC5hcmFuZ2UoNDAsIGR0eXBlPXRvcmNoLmZsb2F0MzIpLnZpZXcoMSwgNDAsIDEpLnJlcGVhdCgyLCAxLCAxKQogICAgbSA9IHRvcmNoLnplcm9zKDIsIDQwLCBkdHlwZT10b3JjaC5ib29sKQogICAgbVswLCA6NDBdLCBtWzEsIDoxN10gPSBUcnVlLCBUcnVlCiAgICBzdGF0ZXMsIGNtYXNrID0gY2h1bmtfYm91bmRhcmllcyh5LCBtLCAxNikKICAgIG9rICY9IGNtYXNrLnRvbGlzdCgpID09IFtbMSwgMSwgMV0sIFsxLCAxLCAwXV0gYW5kIHN0YXRlc1sxLCA6Ml0udmlldygtMSkudG9saXN0KCkgPT0gWzE1LCAxNl0KICAgIGNoZWNrKCJjaHVua19ib3VuZGFyaWVzIiwgb2ssIGRldGFpbCkKCiAgICBpbWcgPSB0b3JjaC5mcm9tX251bXB5KG5wLmxvYWQoRkVBVFVSRV9ESVIgLyAiaW1nX3RyYWluLm5weSIsIG1tYXBfbW9kZT0iciIpWzo4XS5hc3R5cGUobnAuZmxvYXQzMikpLnRvKGRldmljZSkKICAgIHR4dCA9IHRvcmNoLmZyb21fbnVtcHkobnAubG9hZChGRUFUVVJFX0RJUiAvICJ0eHRfdHJhaW4ubnB5IiwgbW1hcF9tb2RlPSJyIilbOjQwXS5hc3R5cGUobnAuZmxvYXQzMikpLnRvKGRldmljZSkKICAgIG1hc2sgPSB0b3JjaC5mcm9tX251bXB5KG5wLmxvYWQoRkVBVFVSRV9ESVIgLyAibWFza190cmFpbi5ucHkiKVs6NDBdLmFzdHlwZShib29sKSkudG8oZGV2aWNlKQogICAgcGFpciA9IHRvcmNoLmFyYW5nZSg4LCBkZXZpY2U9ZGV2aWNlKS5yZXBlYXRfaW50ZXJsZWF2ZSg1KQogICAgY2hlY2soInJlYWxfZmVhdHVyZXNfZmluaXRlIiwgYm9vbCh0b3JjaC5pc2Zpbml0ZShpbWcpLmFsbCgpIGFuZCB0b3JjaC5pc2Zpbml0ZSh0eHQpLmFsbCgpKSwKICAgICAgICAgIHsiaW1nIjogbGlzdChpbWcuc2hhcGUpLCAidHh0IjogbGlzdCh0eHQuc2hhcGUpLCAibWVhbl9jYXB0aW9uX2xlbiI6IGZsb2F0KG1hc2suc3VtKDEpLmZsb2F0KCkubWVhbigpKX0pCgogICAgemkgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLm5vcm1hbGl6ZSh0b3JjaC5yYW5kbig4LCAxMjgsIGRldmljZT1kZXZpY2UpLCBkaW09LTEpCiAgICB6dCA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwubm9ybWFsaXplKHRvcmNoLnJhbmRuKDQwLCAxMjgsIGRldmljZT1kZXZpY2UpLCBkaW09LTEpCiAgICBsb3NzX3JhbmQgPSBmbG9hdChtdWx0aXBvc2l0aXZlX2luZm9uY2UoemksIHp0LCBwYWlyLCB0b3JjaC50ZW5zb3IoMS4wLCBkZXZpY2U9ZGV2aWNlKSkpCiAgICB6dF9wZXJmZWN0ID0gemlbcGFpcl0KICAgIGxvc3NfcGVyZmVjdCA9IGZsb2F0KG11bHRpcG9zaXRpdmVfaW5mb25jZSh6aSwgenRfcGVyZmVjdCwgcGFpciwgdG9yY2gudGVuc29yKDEwMC4wLCBkZXZpY2U9ZGV2aWNlKSkpCiAgICBmbG9vciA9IDAuNSAqIGZsb2F0KG5wLmxvZyg1KSkgICMgNSBlcXVhbCBwb3NpdGl2ZXMgcGVyIGltYWdlOiBpMnQgZmxvb3IgbG9nIDUsIHQyaSBmbG9vciAwCiAgICBjaGVjaygiaW5mb25jZV9nZW9tZXRyeSIsIGFicyhsb3NzX3BlcmZlY3QgLSBmbG9vcikgPCAxZS0yIGFuZCBsb3NzX3JhbmQgPiBmbG9vciArIDAuNSwKICAgICAgICAgIHsicmFuZG9tX3NjYWxlMSI6IGxvc3NfcmFuZCwgImFsaWduZWRfc2NhbGUxMDAiOiBsb3NzX3BlcmZlY3QsICJleHBlY3RlZF9mbG9vciI6IGZsb29yfSkKCiAgICBwYXJhbXMgPSB7fQogICAgZm9yIHZhcmlhbnQgaW4gVkFSSUFOVFM6CiAgICAgICAgbW9kZWwgPSBSZXRyaWV2YWxNb2RlbChidWlsZF9jb25maWcodmFyaWFudCkpLnRvKGRldmljZSkKICAgICAgICBuX25hdGl2ZSA9IGFzc2VydF9uYXRpdmVfbWFtYmEyKG1vZGVsKQogICAgICAgIHBhcmFtc1t2YXJpYW50XSA9IGNvdW50X3RyYWluYWJsZShtb2RlbCkKICAgICAgICBvdXQgPSBtb2RlbChpbWcsIHR4dCwgbWFzaywgcGFpciwgc2FtcGxlPVRydWUpCiAgICAgICAgbG9zcywgcGFydHMgPSB0b3RhbF9sb3NzKG1vZGVsLCBvdXQsIHBhaXIsIDFlLTQpCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgbm9fZ3JhZCA9IFtuIGZvciBuLCBwIGluIG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWQgYW5kIChwLmdyYWQgaXMgTm9uZSldCiAgICAgICAgYmFkID0gW24gZm9yIG4sIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpIGlmIHAuZ3JhZCBpcyBub3QgTm9uZSBhbmQgbm90IHRvcmNoLmlzZmluaXRlKHAuZ3JhZCkuYWxsKCldCiAgICAgICAgY2hlY2soZiJ2YXJpYW50X3t2YXJpYW50fSIsIHRvcmNoLmlzZmluaXRlKGxvc3MpIGFuZCBub3Qgbm9fZ3JhZCBhbmQgbm90IGJhZCwKICAgICAgICAgICAgICB7Imxvc3MiOiBmbG9hdChsb3NzKSwgKip7azogZmxvYXQodikgZm9yIGssIHYgaW4gcGFydHMuaXRlbXMoKX0sICJuYXRpdmVfbWFtYmEyX21vZHVsZXMiOiBuX25hdGl2ZSwKICAgICAgICAgICAgICAgInRyYWluYWJsZV9wYXJhbXMiOiBwYXJhbXNbdmFyaWFudF0sICJwYXJhbXNfd2l0aG91dF9ncmFkIjogbm9fZ3JhZCwgIm5vbmZpbml0ZV9ncmFkcyI6IGJhZH0pCgogICAgdG9yY2gubWFudWFsX3NlZWQoMCkKICAgIGZ1c2VkID0gTWFtYmEyKGRfbW9kZWw9MTI4LCBkX3N0YXRlPTY0LCBkX2NvbnY9NCwgZXhwYW5kPTIsIGhlYWRkaW09NjQsIHVzZV9tZW1fZWZmX3BhdGg9VHJ1ZSkudG8oZGV2aWNlKQogICAgcmVmID0gTWFtYmEyKGRfbW9kZWw9MTI4LCBkX3N0YXRlPTY0LCBkX2NvbnY9NCwgZXhwYW5kPTIsIGhlYWRkaW09NjQsIHVzZV9tZW1fZWZmX3BhdGg9RmFsc2UpLnRvKGRldmljZSkKICAgIHJlZi5sb2FkX3N0YXRlX2RpY3QoZnVzZWQuc3RhdGVfZGljdCgpKQogICAgcHJvaiA9IHRvcmNoLm5uLkxpbmVhcig3NjgsIDEyOCkudG8oZGV2aWNlKQogICAgeCA9IHByb2ooaW1nKS5kZXRhY2goKQogICAgeDEsIHgyID0geC5jbG9uZSgpLnJlcXVpcmVzX2dyYWRfKFRydWUpLCB4LmNsb25lKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgIHkxLCB5MiA9IGZ1c2VkKHgxKSwgcmVmKHgyKQogICAgKHkxLnNxdWFyZSgpLm1lYW4oKSkuYmFja3dhcmQoKQogICAgKHkyLnNxdWFyZSgpLm1lYW4oKSkuYmFja3dhcmQoKQogICAgZndkID0gZmxvYXQoKHkxIC0geTIpLmFicygpLm1heCgpIC8geTIuYWJzKCkubWF4KCkuY2xhbXAobWluPTFlLTEyKSkKICAgIGJ3ZCA9IGZsb2F0KCh4MS5ncmFkIC0geDIuZ3JhZCkuYWJzKCkubWF4KCkgLyB4Mi5ncmFkLmFicygpLm1heCgpLmNsYW1wKG1pbj0xZS0xMikpCiAgICBjaGVjaygibWFtYmEyX2Z1c2VkX3ZzX3JlZmVyZW5jZSIsIGZ3ZCA8IDFlLTMgYW5kIGJ3ZCA8IDFlLTMgYW5kIHRvcmNoLmlzZmluaXRlKHkxKS5hbGwoKSwKICAgICAgICAgIHsicmVsX2ZvcndhcmRfZGlmZiI6IGZ3ZCwgInJlbF9pbnB1dF9ncmFkX2RpZmYiOiBid2R9KQoKICAgIG1vZGVsID0gUmV0cmlldmFsTW9kZWwoYnVpbGRfY29uZmlnKCJmdWxsIikpLnRvKGRldmljZSkuZXZhbCgpCiAgICBsZW5ndGhzID0gbWFzay5zdW0oMSkKICAgIHdvcnN0ID0gMS4wCiAgICBmb3IgaSBpbiByYW5nZSgwLCA0MCwgNSk6CiAgICAgICAgTCA9IGludChsZW5ndGhzW2ldKQogICAgICAgIGEgPSBtb2RlbC5lbmNvZGVfdGV4dCh0eHRbaTppICsgMV0sIG1hc2tbaTppICsgMV0pCiAgICAgICAgYiA9IG1vZGVsLmVuY29kZV90ZXh0KHR4dFtpOmkgKyAxLCA6TF0sIG1hc2tbaTppICsgMSwgOkxdKQogICAgICAgIHdvcnN0ID0gbWluKHdvcnN0LCBmbG9hdCgoYSAqIGIpLnN1bSgpKSkKICAgIGNoZWNrKCJwYWRkaW5nX2ludmFyaWFuY2VfdGV4dCIsIHdvcnN0ID4gMC45OTk5LCB7Im1pbl9jb3NpbmUiOiB3b3JzdH0pCgogICAgaGVkbyA9IEhFRE8oMTI4KS50byhkZXZpY2UpLmRvdWJsZSgpCiAgICBhLCBiID0gdG9yY2gucmFuZG4oMywgNSwgMTI4LCBkZXZpY2U9ZGV2aWNlLCBkdHlwZT10b3JjaC5mbG9hdDY0KSwgdG9yY2gucmFuZG4oMywgNSwgMTI4LCBkZXZpY2U9ZGV2aWNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGR0eXBlPXRvcmNoLmZsb2F0NjQpCiAgICB6ZXJvID0gdG9yY2guemVyb3NfbGlrZShhKQogICAgcmVzaWR1YWwgPSBmbG9hdCgoaGVkbyhhICsgYikgLSBoZWRvKGEpIC0gaGVkbyhiKSArIGhlZG8oemVybykpLmFicygpLm1heCgpKQogICAgY2hlY2soImxlZ2FjeV9oZWRvX2lzX2FmZmluZSIsIHJlc2lkdWFsIDwgMWUtOSwgeyJhZGRpdGl2aXR5X3Jlc2lkdWFsIjogcmVzaWR1YWx9KQoKICAgIHRva2VucyA9IFJldHJpZXZhbE1vZGVsKGJ1aWxkX2NvbmZpZyhQUk9QT1NFRCkpLnRvKGRldmljZSkucHJvalsiaW1nIl0oaW1nKS5kZXRhY2goKS5kb3VibGUoKQogICAgd29yc3QsIGRldGFpbCA9IC1mbG9hdCgiaW5mIiksIHt9CiAgICBmb3IgbmFtZSwgc2NhbGVfdSwgb21lZ2EsIHRoZXRhLCBkYW1wIGluICgoImluaXQiLCAxLjAsIE5vbmUsIE5vbmUsIE5vbmUpLCAoInN0aWZmIiwgMzAuMCwgNS4wLCA1LjAsIC01LjApLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKCJ1bmRhbXBlZCIsIDMuMCwgMi4wLCA1LjAsIC0yMC4wKSwgKCJoZWF2eSIsIDEwLjAsIDEuMCwgMC4wLCA1LjApKToKICAgICAgICB0b3JjaC5tYW51YWxfc2VlZCgxKQogICAgICAgIG9wID0gRW5lcmd5SEVETygxMjgsIDY0LCBzdGVwcz04KS50byhkZXZpY2UpLmRvdWJsZSgpCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIG9wLlUubXVsXyhzY2FsZV91KQogICAgICAgICAgICBvcC5iLm5vcm1hbF8oKQogICAgICAgICAgICBvcC5wX3Byb2oud2VpZ2h0Lm5vcm1hbF8oc3RkPTAuNSkKICAgICAgICAgICAgaWYgb21lZ2EgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBvcC5vbWVnYS5maWxsXyhvbWVnYSkKICAgICAgICAgICAgICAgIG9wLnRoZXRhLmZpbGxfKHRoZXRhKQogICAgICAgICAgICAgICAgb3AuZGFtcGluZy5iaWFzLmZpbGxfKGRhbXApCiAgICAgICAgZSA9IG9wLnRyYWplY3RvcnkodG9rZW5zKVsiZW5lcmdpZXMiXQogICAgICAgIGluYyA9IGZsb2F0KCgoZVsuLi4sIDE6XSAtIGVbLi4uLCA6LTFdKSAvIGVbLi4uLCA6LTFdLmFicygpLmNsYW1wKG1pbj0xLjApKS5tYXgoKSkKICAgICAgICBkZXRhaWxbbmFtZV0gPSB7Im1heF9yZWxhdGl2ZV9zdGVwX2luY3JlYXNlIjogaW5jLCAibWVhbl9kaXNzaXBhdGVkIjogZmxvYXQoKGVbLi4uLCAwXSAtIGVbLi4uLCAtMV0pLm1lYW4oKSl9CiAgICAgICAgd29yc3QgPSBtYXgod29yc3QsIGluYykKICAgIGNoZWNrKCJlbmVyZ3lfdGhlb3JlbV9ub25pbmNyZWFzaW5nIiwgd29yc3QgPD0gMWUtMTAsIGRldGFpbCkKCiAgICB4bSA9IFJldHJpZXZhbE1vZGVsKGJ1aWxkX2NvbmZpZygiaHZzY194IikpLnRvKGRldmljZSkuZXZhbCgpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICB6aSwgYWksIF8gPSB4bS5wYXNzMSgiaW1nIiwgaW1nWzo0XSwgTm9uZSwgc2FtcGxlPUZhbHNlKQogICAgICAgIHp0LCBhdCwgXyA9IHhtLnBhc3MxKCJ0eHQiLCB0eHRbOjRdLCBtYXNrWzo0XSwgc2FtcGxlPUZhbHNlKQogICAgICAgIHNfaW5pdCA9IHhtLnBhaXJfc2ltaWxhcml0eShhaSwgYXQsIG1hc2tbOjRdKQogICAgICAgIG5vb3AgPSBmbG9hdCgoc19pbml0IC0gKHppICogenQpLnN1bSgtMSkpLmFicygpLm1heCgpKQogICAgICAgIGZvciBtIGluICgiaW1nIiwgInR4dCIpOgogICAgICAgICAgICB4bS54aHZzYy5nYXRlW21dLmZpbGxfKDEuMCkKICAgICAgICBzX2EgPSB4bS5wYWlyX3NpbWlsYXJpdHkoYWksIGF0LCBtYXNrWzo0XSkKICAgICAgICBwZXJtID0gdG9yY2gudGVuc29yKFsxLCAyLCAzLCAwXSwgZGV2aWNlPWRldmljZSkKICAgICAgICBzX2IgPSB4bS5wYWlyX3NpbWlsYXJpdHkoYWksIHtrOiB2W3Blcm1dIGZvciBrLCB2IGluIGF0Lml0ZW1zKCl9LCBtYXNrWzo0XVtwZXJtXSkKICAgICAgICB6dF9iID0genRbcGVybV0KICAgICAgICBwYXJ0bmVyX2VmZmVjdCA9IGZsb2F0KCgoc19iIC0gc19hKSAtICgoemkgKiB6dF9iKS5zdW0oLTEpIC0gKHppICogenQpLnN1bSgtMSkpKS5hYnMoKS5tYXgoKSkKICAgIGNoZWNrKCJleGNoYW5nZV9ub29wX2F0X2luaXRfYW5kX2FjdGl2ZV93aGVuX2dhdGVkIiwgbm9vcCA8IDFlLTUgYW5kIHBhcnRuZXJfZWZmZWN0ID4gMWUtNCwKICAgICAgICAgIHsiaW5pdF9hYnNfZGlmZiI6IG5vb3AsICJnYXRlZF9wYXJ0bmVyX2VmZmVjdCI6IHBhcnRuZXJfZWZmZWN0fSkKCiAgICBjaGVjaygibGluZWFyX29wZXJhdG9yX3BhcmFtX21hdGNoIiwgcGFyYW1zWyJmdWxsX2xpbmVhcl9vcGVyYXRvciJdID09IHBhcmFtc1siZnVsbCJdLAogICAgICAgICAge2s6IHBhcmFtc1trXSBmb3IgayBpbiAoImZ1bGwiLCAiZnVsbF9saW5lYXJfb3BlcmF0b3IiKX0pCiAgICByZWwgPSB7ImJhc2VsaW5lX3BhcmFtX21hdGNoZWRfdnNfZnVsbCI6IGFicyhwYXJhbXNbImJhc2VsaW5lX3BhcmFtX21hdGNoZWQiXSAtIHBhcmFtc1siZnVsbCJdKSAvIHBhcmFtc1siZnVsbCJdLAogICAgICAgICAgICJiYXNlbGluZV9wYXJhbV9tYXRjaGVkX3hfdnNfZnVsbF94IjoKICAgICAgICAgICAgICAgYWJzKHBhcmFtc1siYmFzZWxpbmVfcGFyYW1fbWF0Y2hlZF94Il0gLSBwYXJhbXNbImZ1bGxfeCJdKSAvIHBhcmFtc1siZnVsbF94Il0sCiAgICAgICAgICAgInRyYW5zZm9ybWVyX3ZzX2Jhc2VsaW5lIjoKICAgICAgICAgICAgICAgYWJzKHBhcmFtc1sidHJhbnNmb3JtZXJfcGFyYW1fbWF0Y2hlZCJdIC0gcGFyYW1zWyJiYXNlbGluZSJdKSAvIHBhcmFtc1siYmFzZWxpbmUiXX0KICAgIGNoZWNrKCJwYXJhbV9tYXRjaGVkX2NvbnRyb2xzIiwgbWF4KHJlbC52YWx1ZXMoKSkgPCAwLjAxLCByZWwpCgogICAgUkVQT1JUX0RJUi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3cml0ZV9qc29uKFJFUE9SVF9ESVIgLyAicGFyYW1fY291bnRzLmpzb24iLCBwYXJhbXMpCiAgICBwYXNzZWQgPSBhbGwodlsicGFzcyJdIGZvciB2IGluIHJlcG9ydC52YWx1ZXMoKSkKICAgIHdyaXRlX2pzb24oUkVQT1JUX0RJUiAvICJnYXRlX3JlcG9ydC5qc29uIiwgeyJwYXNzIjogcGFzc2VkLCAiY2hlY2tzIjogcmVwb3J0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImVudmlyb25tZW50IjogZW52aXJvbm1lbnRfbWFuaWZlc3QoKSwgInByb3ZlbmFuY2UiOiBwcm92ZW5hbmNlKCl9KQogICAgcHJpbnQoIkdBVEU6IiwgIlBBU1MiIGlmIHBhc3NlZCBlbHNlICJGQUlMIikKICAgIHN5cy5leGl0KDAgaWYgcGFzc2VkIGVsc2UgMSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="], "hypotheses.py": ["60e6993a115684870d2e4b6f6f77ed34e42cf7123e903f58c97fae28f382a5dd", "IiIiRXZhbHVhdGUgaHlwb3RoZXNlcyBIMS1INCBhZ2FpbnN0IGNyaXRlcmlhIGZpeGVkIGluIE1FVEhPRC5tZCAoc2VjdGlvbiA1KSAqYmVmb3JlKiB0aGUgcnVucy4KCklucHV0cyAoYWxsIHJlZ2VuZXJhdGVkIGZyb20gcmF3IGFydGlmYWN0cyk6IHJlc3VsdHMvYWxsX3J1bnMuY3N2LCByZXN1bHRzL3Byb2Jlcy5jc3YsCnJlc3VsdHMvcm9idXN0bmVzcy5jc3YsIHJlcG9ydC9zY2FsaW5nL3NjYWxpbmcuanNvbi4gU2VlZC1sZXZlbCA5NSUgQ0lzIHVzZSB0aGUgdCBkaXN0cmlidXRpb24Kb3ZlciBtYXRjaGVkIHNlZWRzLiBBIGh5cG90aGVzaXMgaXMgInN1cHBvcnRlZCIgb25seSBpZiBldmVyeSBjcml0ZXJpb24gcGFzc2VzLCAibm90IHN1cHBvcnRlZCIKaWYgYSBjcml0ZXJpb24gZmFpbHMsICJpbnN1ZmZpY2llbnQgZGF0YSIgaWYgYW4gaW5wdXQgaXMgbWlzc2luZyBvciBuIDwgMyBzZWVkcy4KCk91dHB1dDogcmVwb3J0L2h5cG90aGVzZXMuanNvbiwgcmVwb3J0L3RhYmxlX2h5cG90aGVzZXMudGV4ClVzYWdlOiBweXRob24gaHlwb3RoZXNlcy5weQoiIiIKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2NpcHkgaW1wb3J0IHN0YXRzCgpmcm9tIGNvbW1vbiBpbXBvcnQgUkVQT1JUX0RJUiwgUlVOU19ESVIsIGJhbm5lciwgcHJvdmVuYW5jZSwgcmVhZF9qc29uLCB3cml0ZV9qc29uCmZyb20gbW9kZWxzIGltcG9ydCBQUk9QT1NFRAoKUkVTVUxUUyA9IFJVTlNfRElSLnBhcmVudCAvICJyZXN1bHRzIgpOT05JTkZFUklPUklUWV9NUiA9IDEuMCAgICAgIyBIMShjKTogb3VycyBtYXkgbm90IGxvc2UgbW9yZSB0aGFuIDEgTVIgcG9pbnQgdG8gdGhlIGJhc2VsaW5lCk9WRVJIRUFEX01BWCA9IDEuNSAgICAgICAgICAjIEg0KGEpOiA8PSAxLjV4IGJhc2VsaW5lIGxhdGVuY3kgLyB0cmFpbiBzdGVwIGF0IDE5NiB0b2tlbnMKU0xPUEVfTUFYID0gMS4yICAgICAgICAgICAgICMgSDQoYik6IGxvZy1sb2cgc2xvcGUgb2YgbGF0ZW5jeSB2cyBsZW5ndGggKDEgPSBsaW5lYXIpCkVORVJHWV9UT0wgPSAxZS05CgoKZGVmIGNpOTUoeCk6CiAgICB4ID0gbnAuYXNhcnJheSh4LCBkdHlwZT1mbG9hdCkKICAgIGlmIGxlbih4KSA8IDI6CiAgICAgICAgcmV0dXJuIFtmbG9hdCgibmFuIiksIGZsb2F0KCJuYW4iKV0KICAgIGggPSBzdGF0cy50LnBwZigwLjk3NSwgbGVuKHgpIC0gMSkgKiB4LnN0ZChkZG9mPTEpIC8gbnAuc3FydChsZW4oeCkpCiAgICByZXR1cm4gW2Zsb2F0KHgubWVhbigpIC0gaCksIGZsb2F0KHgubWVhbigpICsgaCldCgoKZGVmIHBhaXJlZChkZiwgY29sLCBhLCBiKToKICAgIEEgPSBkZltkZlsidmFyaWFudCJdID09IGFdLnNldF9pbmRleCgic2VlZCIpW2NvbF0KICAgIEIgPSBkZltkZlsidmFyaWFudCJdID09IGJdLnNldF9pbmRleCgic2VlZCIpW2NvbF0KICAgIHNlZWRzID0gc29ydGVkKHNldChBLmluZGV4KSAmIHNldChCLmluZGV4KSkKICAgIGQgPSAoQS5sb2Nbc2VlZHNdIC0gQi5sb2Nbc2VlZHNdKS52YWx1ZXMgaWYgc2VlZHMgZWxzZSBucC5hcnJheShbXSkKICAgIHJldHVybiB7ImEiOiBhLCAiYiI6IGIsICJtZXRyaWMiOiBjb2wsICJuIjogbGVuKHNlZWRzKSwgIm1lYW5fZGlmZiI6IGZsb2F0KGQubWVhbigpKSBpZiBsZW4oZCkgZWxzZSBOb25lLAogICAgICAgICAgICAiY2k5NSI6IGNpOTUoZCksICJwZXJfc2VlZCI6IGRpY3QoemlwKG1hcChpbnQsIHNlZWRzKSwgbWFwKGZsb2F0LCBkKSkpfQoKCmRlZiB2ZXJkaWN0KGNyaXRlcmlhKToKICAgIGlmIGFueShjLmdldCgicGFzcyIpIGlzIE5vbmUgZm9yIGMgaW4gY3JpdGVyaWEpOgogICAgICAgIHJldHVybiAiaW5zdWZmaWNpZW50IGRhdGEiCiAgICByZXR1cm4gInN1cHBvcnRlZCIgaWYgYWxsKGNbInBhc3MiXSBmb3IgYyBpbiBjcml0ZXJpYSkgZWxzZSAibm90IHN1cHBvcnRlZCIKCgpkZWYgZGVmYXVsdF9ydW5zKGRmKToKICAgIHJldHVybiBkZlsoZGZbImtsX3dlaWdodCJdID09IDFlLTQpICYgKGRmWyJodnNjX2NodW5rX3NpemUiXSA9PSAxNildCgoKZGVmIGxvYWRfY3N2KG5hbWUpOgogICAgcGF0aCA9IFJFU1VMVFMgLyBuYW1lCiAgICByZXR1cm4gZGVmYXVsdF9ydW5zKHBkLnJlYWRfY3N2KHBhdGgpKSBpZiBwYXRoLmlzX2ZpbGUoKSBlbHNlIE5vbmUKCgpkZWYgaDEocnVucywgcHJvYmVzKToKICAgIGlmIHByb2JlcyBpcyBOb25lOgogICAgICAgIHJldHVybiB7ImNyaXRlcmlhIjogW3sibmFtZSI6ICJwcm9iZXMgYXZhaWxhYmxlIiwgInBhc3MiOiBOb25lfV19CiAgICBvdXJzID0gcHJvYmVzW3Byb2Jlc1sidmFyaWFudCJdID09IFBST1BPU0VEXQogICAgYyA9IFtdCiAgICBpbmMgPSBvdXJzWyJlbmVyZ3lfbWF4X2luY3JlYXNlX2Zsb2F0NjQiXS5tYXgoKSBpZiBsZW4ob3VycykgZWxzZSBOb25lCiAgICBjLmFwcGVuZCh7Im5hbWUiOiAiKGEpIGVuZXJneSBuZXZlciBpbmNyZWFzZXMgb24gcmVhbCB0b2tlbnMgKGZsb2F0NjQpIiwgInZhbHVlIjogaW5jLAogICAgICAgICAgICAgICJwYXNzIjogTm9uZSBpZiBpbmMgaXMgTm9uZSBvciBucC5pc25hbihpbmMpIGVsc2UgYm9vbChpbmMgPD0gRU5FUkdZX1RPTCl9KQogICAgZCA9KG91cnNbImF0dGVudWF0aW9uX2ZnIl0gLSBvdXJzWyJhdHRlbnVhdGlvbl9iZyJdKS52YWx1ZXMKICAgIGxvLCBoaSA9IGNpOTUoZCkKICAgIGMuYXBwZW5kKHsibmFtZSI6ICIoYikgYmFja2dyb3VuZCB0b2tlbnMgYXR0ZW51YXRlZCBtb3JlIHRoYW4gZm9yZWdyb3VuZDogbWVhbihhdHRfZmcgLSBhdHRfYmcpID4gMCwgQ0kgZXhjbHVkZXMgMCIsCiAgICAgICAgICAgICAgIm4iOiBsZW4oZCksICJtZWFuIjogZmxvYXQoZC5tZWFuKCkpIGlmIGxlbihkKSBlbHNlIE5vbmUsICJjaTk1IjogW2xvLCBoaV0sCiAgICAgICAgICAgICAgInNwZWFybWFuX3NhbGllbmN5X2F0dGVudWF0aW9uIjogZmxvYXQob3Vyc1siYXR0ZW51YXRpb25fc3BlYXJtYW5fc2FsaWVuY3kiXS5tZWFuKCkpIGlmIGxlbihkKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgInBhc3MiOiBOb25lIGlmIGxlbihkKSA8IDMgZWxzZSBib29sKGxvID4gMCl9KQogICAgbXIgPSBwYWlyZWQocnVucywgIm1lYW5fcmVjYWxsIiwgUFJPUE9TRUQsICJiYXNlbGluZSIpCiAgICBjLmFwcGVuZCh7Im5hbWUiOiBmIihjMSkgbm8gbG9zcyBvZiBzZW1hbnRpYyBpbmZvcm1hdGlvbjogTVIob3VycykgLSBNUihiYXNlbGluZSkgQ0kgbG93ZXIgYm91bmQgPiAte05PTklORkVSSU9SSVRZX01SfSIsCiAgICAgICAgICAgICAgKiptciwgInBhc3MiOiBOb25lIGlmIG1yWyJuIl0gPCAzIGVsc2UgYm9vbChtclsiY2k5NSJdWzBdID4gLU5PTklORkVSSU9SSVRZX01SKX0pCiAgICBnYXAgPSAob3Vyc1sib2NjbHVkZV9iZ19kdWFsX21lYW5fcmVjYWxsIl0gLSBvdXJzWyJvY2NsdWRlX2ZnX2R1YWxfbWVhbl9yZWNhbGwiXSkudmFsdWVzCiAgICBsbywgaGkgPSBjaTk1KGdhcCkKICAgIGMuYXBwZW5kKHsibmFtZSI6ICIoYzIpIHNlbWFudGljcyBzdGF5IGluIHRoZSBmb3JlZ3JvdW5kOiBNUihiYWNrZ3JvdW5kIG9jY2x1ZGVkKSAtIE1SKGZvcmVncm91bmQgb2NjbHVkZWQpICIKICAgICAgICAgICAgICAgICAgICAgICJDSSBsb3dlciA+IDAgZm9yIG91cnMiLAogICAgICAgICAgICAgICJuIjogbGVuKGdhcCksICJtZWFuIjogZmxvYXQoZ2FwLm1lYW4oKSkgaWYgbGVuKGdhcCkgZWxzZSBOb25lLCAiY2k5NSI6IFtsbywgaGldLAogICAgICAgICAgICAgICJwYXNzIjogTm9uZSBpZiBsZW4oZ2FwKSA8IDMgZWxzZSBib29sKGxvID4gMCl9KQogICAgcmV0dXJuIHsiY3JpdGVyaWEiOiBjfQoKCmRlZiBoMihwcm9iZXMpOgogICAgaWYgcHJvYmVzIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHsiY3JpdGVyaWEiOiBbeyJuYW1lIjogInByb2JlcyBhdmFpbGFibGUiLCAicGFzcyI6IE5vbmV9XX0KICAgIGcgPSBwYWlyZWQocHJvYmVzLCAiZ3JhZF9hYnNfbG9nMTBfcmF0aW8iLCBQUk9QT1NFRCwgImZ1bGxfeF9ub2V4Y2hhbmdlIikKICAgIG91cnMgPSBwcm9iZXNbcHJvYmVzWyJ2YXJpYW50Il0gPT0gUFJPUE9TRURdCiAgICByZXR1cm4geyJjcml0ZXJpYSI6IFsKICAgICAgICB7Im5hbWUiOiAiKGEpIGV4Y2hhbmdlIHJlZHVjZXMgZ3JhZGllbnQgaW1iYWxhbmNlIGJldHdlZW4gbW9kYWxpdGllczogfGxvZzEwIHJhdGlvfChvdXJzKSAtIChubyBleGNoYW5nZSkgQ0kgdXBwZXIgPCAwIiwKICAgICAgICAgKipnLCAicGFzcyI6IE5vbmUgaWYgZ1sibiJdIDwgMyBlbHNlIGJvb2woZ1siY2k5NSJdWzFdIDwgMCl9LAogICAgICAgIHsibmFtZSI6ICIoYikgZXhjaGFuZ2UgaXMgdXNlZCBpbiBib3RoIGRpcmVjdGlvbnM6IGJvdGggZ2F0ZXMgbm9uLXplcm8gYW5kIGRvbWluYW5jZSBpbmRleCA8IDAuNSBvbiBhdmVyYWdlIiwKICAgICAgICAgImRvbWluYW5jZV9pbmRleF9tZWFuIjogZmxvYXQob3Vyc1siZG9taW5hbmNlX2luZGV4Il0ubWVhbigpKSBpZiAiZG9taW5hbmNlX2luZGV4IiBpbiBvdXJzIGFuZCBsZW4ob3VycykgZWxzZSBOb25lLAogICAgICAgICAiZ2F0ZV9pbWdfbWVhbiI6IGZsb2F0KG91cnNbImdhdGVfaW1nIl0uYWJzKCkubWVhbigpKSBpZiAiZ2F0ZV9pbWciIGluIG91cnMgYW5kIGxlbihvdXJzKSBlbHNlIE5vbmUsCiAgICAgICAgICJnYXRlX3R4dF9tZWFuIjogZmxvYXQob3Vyc1siZ2F0ZV90eHQiXS5hYnMoKS5tZWFuKCkpIGlmICJnYXRlX3R4dCIgaW4gb3VycyBhbmQgbGVuKG91cnMpIGVsc2UgTm9uZSwKICAgICAgICAgInBhc3MiOiBOb25lIGlmIGxlbihvdXJzKSA8IDMgb3IgImRvbWluYW5jZV9pbmRleCIgbm90IGluIG91cnMgZWxzZQogICAgICAgICBib29sKG91cnNbImRvbWluYW5jZV9pbmRleCJdLm1lYW4oKSA8IDAuNSBhbmQgb3Vyc1siZ2F0ZV9pbWciXS5hYnMoKS5taW4oKSA+IDFlLTMKICAgICAgICAgICAgICBhbmQgb3Vyc1siZ2F0ZV90eHQiXS5hYnMoKS5taW4oKSA+IDFlLTMpfSwKICAgIF19CgoKZGVmIGgzKHJvYnVzdCk6CiAgICBpZiByb2J1c3QgaXMgTm9uZToKICAgICAgICByZXR1cm4geyJjcml0ZXJpYSI6IFt7Im5hbWUiOiAicm9idXN0bmVzcyByZXN1bHRzIGF2YWlsYWJsZSIsICJwYXNzIjogTm9uZX1dfQogICAgY29ycnVwdGVkID0gcm9idXN0W3JvYnVzdFsiY29ycnVwdGlvbiJdICE9ICJjbGVhbiJdCiAgICBwZXJfcnVuID0gY29ycnVwdGVkLmdyb3VwYnkoWyJ2YXJpYW50IiwgInNlZWQiLCAibW9kYWxpdHkiXSlbInJlbGF0aXZlX21yIl0ubWVhbigpLnJlc2V0X2luZGV4KCkKICAgIGNyaXQgPSBbXQogICAgZm9yIG1vZGFsaXR5IGluICgiaW1hZ2UiLCAidGV4dCIpOgogICAgICAgIHN1YiA9IHBlcl9ydW5bcGVyX3J1blsibW9kYWxpdHkiXSA9PSBtb2RhbGl0eV0KICAgICAgICBwID0gcGFpcmVkKHN1YiwgInJlbGF0aXZlX21yIiwgUFJPUE9TRUQsICJiYXNlbGluZSIpCiAgICAgICAgY3JpdC5hcHBlbmQoeyJuYW1lIjogZiJyZWxhdGl2ZSBNUiB1bmRlciB7bW9kYWxpdHl9IGNvcnJ1cHRpb25zOiBvdXJzIC0gYmFzZWxpbmUgQ0kgbG93ZXIgPiAwIiwgKipwLAogICAgICAgICAgICAgICAgICAgICAicGFzcyI6IE5vbmUgaWYgcFsibiJdIDwgMyBlbHNlIGJvb2wocFsiY2k5NSJdWzBdID4gMCl9KQogICAgcmV0dXJuIHsiY3JpdGVyaWEiOiBjcml0LCAicGVyX2NvcnJ1cHRpb25fbWVhbiI6IGNvcnJ1cHRlZC5ncm91cGJ5KFsidmFyaWFudCIsICJjb3JydXB0aW9uIl0pWyJyZWxhdGl2ZV9tciJdCiAgICAgICAgICAgIC5tZWFuKCkudW5zdGFjaygpLnJvdW5kKDQpLnRvX2RpY3QoKX0KCgpkZWYgaDQoKToKICAgIHBhdGggPSBSRVBPUlRfRElSIC8gInNjYWxpbmciIC8gInNjYWxpbmcuanNvbiIKICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICByZXR1cm4geyJjcml0ZXJpYSI6IFt7Im5hbWUiOiAic2NhbGluZyByZXN1bHRzIGF2YWlsYWJsZSIsICJwYXNzIjogTm9uZX1dfQogICAgc2MgPSByZWFkX2pzb24ocGF0aCkKICAgIHJvd3MgPSBwZC5EYXRhRnJhbWUoc2NbInJvd3MiXSkKICAgIGF0MTk2ID0gcm93c1socm93c1sidmFyaWFudCJdID09IFBST1BPU0VEKSAmIChyb3dzWyJpbWdfbGVuIl0gPT0gMTk2KSAmIChyb3dzWyJzdGF0dXMiXSA9PSAib2siKV0KICAgIGNyaXQgPSBbXQogICAgZm9yIGNvbCBpbiAoImluZmVyX2JzMV9tc192c19iYXNlbGluZSIsICJ0cmFpbl9zdGVwX21zX3ZzX2Jhc2VsaW5lIik6CiAgICAgICAgdiA9IGZsb2F0KGF0MTk2W2NvbF0uaWxvY1swXSkgaWYgbGVuKGF0MTk2KSBhbmQgY29sIGluIGF0MTk2IGVsc2UgTm9uZQogICAgICAgIGNyaXQuYXBwZW5kKHsibmFtZSI6IGYiKGEpIG92ZXJoZWFkIHtjb2x9IDw9IHtPVkVSSEVBRF9NQVh9IGF0IDE5NiB0b2tlbnMiLCAidmFsdWUiOiB2LAogICAgICAgICAgICAgICAgICAgICAicGFzcyI6IE5vbmUgaWYgdiBpcyBOb25lIGVsc2UgYm9vbCh2IDw9IE9WRVJIRUFEX01BWCl9KQogICAgZm9yIGNvbCBpbiAoImluZmVyX2JzMV9tcyIsICJ0cmFpbl9zdGVwX21zIik6CiAgICAgICAgc2xvcGUgPSBzY1sibG9nbG9nX3Nsb3Blc19MX2dlXzc4NCJdLmdldChmIntQUk9QT1NFRH18e2NvbH0iKQogICAgICAgIGNyaXQuYXBwZW5kKHsibmFtZSI6IGYiKGIpIG5lYXItbGluZWFyIHNjYWxpbmc6IGxvZy1sb2cgc2xvcGUgb2Yge2NvbH0gPD0ge1NMT1BFX01BWH0iLCAidmFsdWUiOiBzbG9wZSwKICAgICAgICAgICAgICAgICAgICAgInRyYW5zZm9ybWVyX3Nsb3BlIjogc2NbImxvZ2xvZ19zbG9wZXNfTF9nZV83ODQiXS5nZXQoZiJ0cmFuc2Zvcm1lcl94fHtjb2x9IiksCiAgICAgICAgICAgICAgICAgICAgICJwYXNzIjogTm9uZSBpZiBzbG9wZSBpcyBOb25lIGVsc2UgYm9vbChzbG9wZSA8PSBTTE9QRV9NQVgpfSkKICAgIHJldHVybiB7ImNyaXRlcmlhIjogY3JpdH0KCgpkZWYgbWFpbigpOgogICAgYmFubmVyKCJIWVBPVEhFU0VTIEgxLUg0IikKICAgIHJ1bnMgPSBsb2FkX2NzdigiYWxsX3J1bnMuY3N2IikKICAgIHJ1bnMgPSBydW5zW3J1bnNbInN0YXR1cyJdID09ICJDT01QTEVURUQiXSBpZiBydW5zIGlzIG5vdCBOb25lIGVsc2UgTm9uZQogICAgcHJvYmVzLCByb2J1c3QgPSBsb2FkX2NzdigicHJvYmVzLmNzdiIpLCBsb2FkX2Nzdigicm9idXN0bmVzcy5jc3YiKQogICAgcmVzdWx0ID0geyJIMSBlbmVyZ3kgZGlzc2lwYXRpb24gc3VwcHJlc3NlcyBiYWNrZ3JvdW5kIHdpdGhvdXQgbG9zaW5nIHNlbWFudGljcyI6IGgxKHJ1bnMsIHByb2JlcyksCiAgICAgICAgICAgICAgIkgyIGV4Y2hhbmdlIHJlZHVjZXMgbW9kYWxpdHkgZG9taW5hbmNlIjogaDIocHJvYmVzKSwKICAgICAgICAgICAgICAiSDMgcm9idXN0bmVzcyB0byBub2lzeSBtdWx0aW1vZGFsIGlucHV0cyI6IGgzKHJvYnVzdCksCiAgICAgICAgICAgICAgIkg0IHNtYWxsIG92ZXJoZWFkIGFuZCBsaW5lYXIgc2NhbGluZyI6IGg0KCl9CiAgICBmb3IgbmFtZSwgciBpbiByZXN1bHQuaXRlbXMoKToKICAgICAgICByWyJ2ZXJkaWN0Il0gPSB2ZXJkaWN0KHJbImNyaXRlcmlhIl0pCiAgICAgICAgcHJpbnQoZiJ7clsndmVyZGljdCddOj4xOHN9ICB7bmFtZX0iKQogICAgICAgIGZvciBjIGluIHJbImNyaXRlcmlhIl06CiAgICAgICAgICAgIHByaW50KGYieycnOjIwc317J1BBU1MnIGlmIGNbJ3Bhc3MnXSBlbHNlICgnbi9hJyBpZiBjWydwYXNzJ10gaXMgTm9uZSBlbHNlICdGQUlMJyk6NXN9IHtjWyduYW1lJ119IikKICAgIHdyaXRlX2pzb24oUkVQT1JUX0RJUiAvICJoeXBvdGhlc2VzLmpzb24iLCB7Imh5cG90aGVzZXMiOiByZXN1bHQsICJ0aHJlc2hvbGRzIjogewogICAgICAgICJub25pbmZlcmlvcml0eV9tciI6IE5PTklORkVSSU9SSVRZX01SLCAib3ZlcmhlYWRfbWF4IjogT1ZFUkhFQURfTUFYLCAic2xvcGVfbWF4IjogU0xPUEVfTUFYLAogICAgICAgICJlbmVyZ3lfdG9sIjogRU5FUkdZX1RPTH0sICJwcm92ZW5hbmNlIjogcHJvdmVuYW5jZSgpfSkKICAgIGxpbmVzID0gWyJcXGJlZ2lue3RhYmxlfVt0XSIsICJcXGNlbnRlcmluZyIsCiAgICAgICAgICAgICAiXFxjYXB0aW9ue1ByZS1yZWdpc3RlcmVkIGh5cG90aGVzaXMgdGVzdHMgKGNyaXRlcmlhIGluIFNlY3Rpb25+NSBvZiB0aGUgbWV0aG9kKS4gQ0lzOiA5NVxcJSBvdmVyIG1hdGNoZWQgc2VlZHMufSIsCiAgICAgICAgICAgICAiXFxsYWJlbHt0YWI6aHlwb3RoZXNlc30iLCAiXFxiZWdpbnt0YWJ1bGFyfXtscHswLjYyXFxsaW5ld2lkdGh9bH0iLCAiXFxobGluZSIsCiAgICAgICAgICAgICAiSCAmIENyaXRlcmlvbiAmIFJlc3VsdCBcXFxcIiwgIlxcaGxpbmUiXQogICAgZm9yIGksIChuYW1lLCByKSBpbiBlbnVtZXJhdGUocmVzdWx0Lml0ZW1zKCksIDEpOgogICAgICAgIGZvciBjIGluIHJbImNyaXRlcmlhIl06CiAgICAgICAgICAgIHJlcyA9ICJwYXNzIiBpZiBjWyJwYXNzIl0gZWxzZSAoIm4vYSIgaWYgY1sicGFzcyJdIGlzIE5vbmUgZWxzZSAiZmFpbCIpCiAgICAgICAgICAgIGNyaXQgPSBjWyJuYW1lIl0ucmVwbGFjZSgiXyIsICJcXF8iKS5yZXBsYWNlKCIlIiwgIlxcJSIpLnJlcGxhY2UoIj4iLCAiJD4kIikucmVwbGFjZSgiPCIsICIkPCQiKQogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZiJIe2l9ICYge2NyaXR9ICYge3Jlc30gXFxcXCIpCiAgICAgICAgbGluZXMuYXBwZW5kKGYiICYgXFx0ZXh0YmZ7e1ZlcmRpY3R9fSAmIFxcdGV4dGJme3t7clsndmVyZGljdCddfX19IFxcXFwgXFxobGluZSIpCiAgICBsaW5lcyArPSBbIlxcZW5ke3RhYnVsYXJ9IiwgIlxcZW5ke3RhYmxlfSJdCiAgICAoUkVQT1JUX0RJUiAvICJ0YWJsZV9oeXBvdGhlc2VzLnRleCIpLndyaXRlX3RleHQoIlxuIi5qb2luKGxpbmVzKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludCgiSFlQT1RIRVNFUzogUEFTUyIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="], "METHOD.md": ["e5640542a12691e6dd3fb52b98ab479920cda2fb3b66459335812db0b1e75a75", "IyBNZXRob2Q6IEUtSEVETyBhbmQgWC1IVlNDIGZvciBtdWx0aW1vZGFsIE1hbWJhLTIKClBoYXNlLTEgZm9ybXVsYXRpb24sIG1hdGNoaW5nIGBtb2RlbHMucHlgIGxpbmUgZm9yIGxpbmUuIE5vdGF0aW9uOiB0b2tlbiAkeF9pIFxpbiBcbWF0aGJie1J9XmQkICgkZD0xMjgkKSBhZnRlciB0aGUgbW9kYWxpdHkgcHJvamVjdGlvbjsgaW1hZ2Ugc2VxdWVuY2UgbGVuZ3RoICRMX0kgPSAxOTYkLCBjYXB0aW9uIGxlbmd0aCAkTF9UIFxsZSA2NCQ7IGNodW5rIHNpemUgJEMgPSAxNiQuCgojIyAxLiBFbmVyZ3ktZGlzc2lwYXRpbmcgb3BlcmF0b3IgKEUtSEVETykKCioqRW5lcmd5LioqIEZvciBlYWNoIHRva2VuLCBhIGNvb3JkaW5hdGUgJHEkIGFuZCBhIG1vbWVudHVtICRwJCBjYXJyeQoKJCRIKHEscCkgPSBcdGZyYWMxMlx8cFx8XjIgKyBWKHEpLCBccXF1YWQgVihxKSA9IFxzdW1fe3I9MX1ee1J9IHdfciBcLFxsb2dcY29zaCh1X3JeXHRvcCBxICsgYl9yKSwgXHF1YWQgd19yID0gXG9wZXJhdG9ybmFtZXtzb2Z0cGx1c30oXG9tZWdhX3IpIFxnZSAwIC4kJAoKJFYgXGdlIDAkLCBzbyAkSCBcZ2UgMCQuIFRoZSBncmFkaWVudCBpcyAkXG5hYmxhIFYocSkgPSBVXlx0b3BcYmlnKHcgXG9kb3QgXHRhbmgoVXEgKyBiKVxiaWcpJCwgYW5kIHRoZSBIZXNzaWFuIHNhdGlzZmllcwoKJCQwIFxwcmVjZXEgXG5hYmxhXjIgVihxKSA9IFVeXHRvcCBcb3BlcmF0b3JuYW1le2RpYWd9XCFcYmlnKHcgXG9kb3QgXG9wZXJhdG9ybmFtZXtzZWNofV4yKFVxK2IpXGJpZykgVSBccHJlY2VxIFxoYXQgTFwsIEksIFxxcXVhZCBcaGF0IEwgPSBcbWF4X3Igd19yIFwsXHxVXHxfMl4yICwkJAoKc2luY2UgJDAgPCBcb3BlcmF0b3JuYW1le3NlY2h9XjIgXGxlIDEkLiAkViQgaXMgdGhlcmVmb3JlICRcaGF0IEwkLXNtb290aC4KCioqRHluYW1pY3MuKiogJHFfMCA9IHhfaSQsICRwXzAgPSBXX3AgeF9pJC4gRm9yICRrID0gMCxcZG90cyxLLTEkICh3aXRoICRLID0gMyQpOgoKJCRwX3trKzF9ID0gY19pXCwgcF9rIC0gXERlbHRhIHRcLCBcbmFibGEgVihxX2spLCBccXF1YWQgcV97aysxfSA9IHFfayArIFxEZWx0YSB0XCwgcF97aysxfSwkJAoKd2hlcmUKLSB0aGUgc3RlcCBzaXplIGlzICRcRGVsdGEgdCA9IFxtaW5cIVxiaWcoXERlbHRhIHRfe1xtYXh9XCxcc2lnbWEoXHRoZXRhKSxcIFxzcXJ0eygxLVx2YXJlcHNpbG9uKS9caGF0IEx9XGJpZykkOwotIHRoZSB0b2tlbi13aXNlIGRhbXBpbmcgcmF0ZSBpcyAkXGdhbW1hX2kgPSBcb3BlcmF0b3JuYW1le3NvZnRwbHVzfShnXlx0b3AgeF9pICsgZ18wKSQgKGlucHV0LWRlcGVuZGVudDogdGhpcyBpcyB3aGF0IGxldHMgdGhlIG9wZXJhdG9yIHRyZWF0IGJhY2tncm91bmQgYW5kIGZvcmVncm91bmQgdG9rZW5zIGRpZmZlcmVudGx5KTsKLSB0aGUgZGFtcGluZyBmYWN0b3IgaXMgJGNfaSA9IFxtaW5cIVxiaWcoZV57LVxnYW1tYV9pIFxEZWx0YSB0fSxcIFxzcXJ0ezEgLSBcaGF0IEwgXERlbHRhIHReMn1cYmlnKSQuCgoqKk91dHB1dC4qKiBUaGUgZGlzc2lwYXRlZCBmcmFjdGlvbiBpcyAkXHJob19pID0gXGRmcmFje0gocV8wLHBfMCkgLSBIKHFfSyxwX0spfXtIKHFfMCxwXzApICsgMX0gXGluIFswLDEpJCwgYW5kIHRoZSBvdXRwdXQgdG9rZW4gaXMKCiQkXHRpbGRlIHhfaSA9IGVeey1cZXRhIFxyaG9faX1cLCBxX0ssIFxxcXVhZCBcZXRhID0gXG9wZXJhdG9ybmFtZXtzb2Z0cGx1c30oXGV0YV97XHRleHR7cmF3fX0pLCQkCgp3aGljaCBpcyB3aGF0IGdvZXMgaW50byBNYW1iYS0yLgoKKipUaGVvcmVtIDEgKGVuZXJneSBub24taW5jcmVhc2UpLioqIElmICRWJCBpcyAkXGhhdCBMJC1zbW9vdGgsICRcaGF0IExcRGVsdGEgdF4yIDwgMSQsIGFuZCAkMCBcbGUgYyBcbGUgXHNxcnR7MS1caGF0IExcRGVsdGEgdF4yfSQsIHRoZW4gJEgocV97aysxfSxwX3trKzF9KSBcbGUgSChxX2sscF9rKSQgZm9yIGV2ZXJ5ICRrJC4KCipQcm9vZi4qIFdyaXRlICRwJyA9IHBfe2srMX0kIGFuZCAkcScgPSBxX2sgKyBcRGVsdGEgdFwscCckLiBCeSB0aGUgZGVzY2VudCBsZW1tYSwKCiQkVihxJykgXGxlIFYocV9rKSArIFxEZWx0YSB0XGxhbmdsZSBcbmFibGEgVihxX2spLCBwJ1xyYW5nbGUgKyBcdGZyYWN7XGhhdCBMXERlbHRhIHReMn17Mn1cfHAnXHxeMiAuJCQKCkZyb20gdGhlIG1vbWVudHVtIHVwZGF0ZSwgJFxEZWx0YSB0XCxcbmFibGEgVihxX2spID0gY1wscF9rIC0gcCckLCBzbwoKJCRcRGVsdGEgdFxsYW5nbGUgXG5hYmxhIFYocV9rKSwgcCdccmFuZ2xlID0gY1xsYW5nbGUgcF9rLCBwJ1xyYW5nbGUgLSBcfHAnXHxeMiAuJCQKCkxldCAkYSA9IDEgLSBcaGF0IExcRGVsdGEgdF4yID4gMCQuIEJ5IFlvdW5nJ3MgaW5lcXVhbGl0eSwgJGNcbGFuZ2xlIHBfayxwJ1xyYW5nbGUgXGxlIFx0ZnJhY3tjXjJ9ezJhfVx8cF9rXHxeMiArIFx0ZnJhYyBhMlx8cCdcfF4yJC4gSGVuY2UKCiQkSChxJyxwJykgXGxlIFYocV9rKSArIFx8cCdcfF4yXEJpZyhcdGZyYWMxMiAtIDEgKyBcdGZyYWN7XGhhdCBMXERlbHRhIHReMn17Mn0gKyBcdGZyYWMgYTJcQmlnKSArIFx0ZnJhY3tjXjJ9ezJhfVx8cF9rXHxeMiA9IFYocV9rKSArIFx0ZnJhY3tjXjJ9ezJhfVx8cF9rXHxeMiBcbGUgVihxX2spICsgXHRmcmFjMTJcfHBfa1x8XjIgPSBIKHFfayxwX2spLCQkCgpiZWNhdXNlIHRoZSBicmFja2V0IGVxdWFscyAwIGFuZCAkY14yIFxsZSBhJC4gJFxibGFja3NxdWFyZSQKCioqQ29yb2xsYXJpZXMuKiogQnkgaW5kdWN0aW9uICRIKHFfaywgcF9rKSBcbGUgSF8wIDo9IEgocV8wLCBwXzApJCBmb3IgYWxsICRrJC4gSGVuY2UgJFx8cF9rXHwgXGxlIFxzcXJ0ezJIXzB9JCBhbmQgJFx8cV9LIC0geF9pXHwgXGxlIEtcLFxEZWx0YSB0XCxcc3FydHsySF8wfSQ6IHRoZSBkaXNwbGFjZW1lbnQgaXMgYm91bmRlZCBieSB0aGUgaW5pdGlhbCBlbmVyZ3ksIHdoYXRldmVyIHRoZSBsZWFybmVkIHBhcmFtZXRlcnMuICRccmhvX2kgXGluIFswLDEpJCwgc28gdGhlIGF0dGVudWF0aW9uIGZhY3RvciBsaWVzIGluICQoZV57LVxldGF9LCAxXSQuCgoqKldoYXQgaXMgYW5kIGlzIG5vdCBjbGFpbWVkLioqIFRoZSBzY2hlbWUgaXMgYSBjb25mb3JtYWxseSBkYW1wZWQgc2VtaS1pbXBsaWNpdCAoc3ltcGxlY3RpYy1FdWxlci10eXBlKSBzdGVwLiBJdCBpcyAqbm90KiBzeW1wbGVjdGljIHdoZW4gJGM8MSQsIGFuZCBub3RoaW5nIGlzIGNvbnNlcnZlZC4gIkhhbWlsdG9uaWFuLWluc3BpcmVkIiByZWZlcnMgdG8gdGhlIGVuZXJneSAkSCQgYW5kIHRoZSBjb29yZGluYXRl4oCTbW9tZW50dW0gc3RydWN0dXJlLiBXaGV0aGVyIGRpc3NpcGF0aW9uIGNvbmNlbnRyYXRlcyBvbiBiYWNrZ3JvdW5kIHRva2VucyBpcyBhbiBlbXBpcmljYWwgcXVlc3Rpb24gKEgxKSwgbm90IGEgdGhlb3JlbS4gSW1wbGVtZW50YXRpb24gY2hlY2tzOiB0aGUgZ2F0ZSAoYGdhdGUucHlgKSBhbmQgdGhlIHVuaXQgdGVzdHMgdmVyaWZ5IHRoZSBIZXNzaWFuIGJvdW5kIGFnYWluc3QgYXV0b2dyYWQsIGFuZCB2ZXJpZnkgZW5lcmd5IG5vbi1pbmNyZWFzZSBpbiBmbG9hdDY0IG9uIHJlYWwgdG9rZW5zIGFuZCBhZHZlcnNhcmlhbCBwYXJhbWV0ZXJzLiBUaGUgY2xhbXAgYGRpc3NpcGF0ZWQgPj0gMGAgb25seSByZW1vdmVzIGZsb2F0MzIgcm91bmQtb2ZmLgoKIyMgMi4gQ2h1bmstYm91bmRhcnkgZXhjaGFuZ2Ugd2l0aCB2YXJpYXRpb25hbCBib3R0bGVuZWNrIChYLUhWU0MpCgoqKlBhc3MgMSAoaW5kZXBlbmRlbnQsIHBlciBtb2RhbGl0eSAkbSQpLioqICR5Xm0gPSBcdGV4dHtNYW1iYTJ9Xm0oXHRpbGRlIHhebSkkLiBMZXQgJHNebV9rJCBiZSB0aGUgc3RhdGUgYXQgdGhlIGxhc3QgdmFsaWQgdG9rZW4gb2YgY2h1bmsgJGskLCAkayA9IDEuLktfbSQsICRLX20gPSBcbGNlaWwgTF9tIC8gQ1xyY2VpbCQuCgoqKkJvdHRsZW5lY2suKiogJHEoel5tX2sgXG1pZCBzXm1faykgPSBcbWF0aGNhbCBOKFxtdV5tKHNebV9rKSwgXG9wZXJhdG9ybmFtZXtkaWFnfVxzaWdtYV4yKHNebV9rKSkkIHdpdGggJFxtdSA9IDEwXHRhbmgoXGNkb3QvMTApJCBhbmQgJFxsb2dcc2lnbWFeMiBcaW4gKC00LCAxLjUpJCB0aHJvdWdoIGEgc2lnbW9pZC4gVGhlIHJhdGUgdGVybSBpcyAkXG1hdGhybXtLTH1cYmlnKHEoel5tX2tcbWlkIHNebV9rKVwsXHxcLFxtYXRoY2FsIE4oMCxJKVxiaWcpJCwgYXZlcmFnZWQgb3ZlciBkaW1lbnNpb25zIGFuZCB2YWxpZCBjaHVua3MuCgoqKkR1YWwgZW1iZWRkaW5nLioqICRlXm0gPSBcb3BlcmF0b3JuYW1le25vcm19XCFcYmlnKGhebShcb3BlcmF0b3JuYW1le21lYW59X3QgeV5tX3QgKyBcb3BlcmF0b3JuYW1le21lYW59X2sgUl5tIHpebV9rKVxiaWcpJCwgdXNlZCBmb3Igc2NhbGFibGUgcmV0cmlldmFsLgoKKipFeGNoYW5nZSAocGFzcyAyKSwgZm9yIGEgcGFpciAoaW1hZ2UsIGNhcHRpb24pLioqIFRoZSByZWNlaXZlcidzIGNodW5rICRrJCBnZXRzIHRoZSBtZXNzYWdlCgokJFx0ZXh0e21zZ31ee219X2sgPSBcdGFuaChcYWxwaGFebSlcY2RvdCBcb3BlcmF0b3JuYW1le01IQX1cYmlnKFEgPSBzXm1fe2stMX0sXCBLID0gViA9IFx7el57XGJhciBtfV9qXH1falxiaWcpIFxxcXVhZCAoUSA9IFx0ZXh0e2xlYXJuZWQgfSBxXzBebSBcdGV4dHsgZm9yIH0gaz0xKSwkJAoKYWRkZWQgdG8gdGhlIGZpcnN0IHRva2VuIG9mIGNodW5rICRrJCBvbmx5LiBUaGVuICR5J15tID0gXHRleHR7TWFtYmEyfV5tKFx0aWxkZSB4Xm0gKyBcdGV4dHttc2d9Xm0pJCByZXVzZXMgdGhlIHNhbWUgd2VpZ2h0cywgYW5kIHRoZSBleGNoYW5nZSBlbWJlZGRpbmcgaXMgJGUnXm0gPSBcb3BlcmF0b3JuYW1le25vcm19KGhebShcb3BlcmF0b3JuYW1le21lYW59X3QgeSdebV90ICsgXG9wZXJhdG9ybmFtZXttZWFufV9rIFJebSB6Xm1faykpJC4gVGhlIHBhaXIgc2NvcmUgaXMgJHMnID0gXGxhbmdsZSBlJ15JLCBlJ15UXHJhbmdsZSQuIFdpdGggJFxhbHBoYV5tID0gMCQgYXQgaW5pdGlhbGlzYXRpb24sIHRoZSBleGNoYW5nZSBpcyBleGFjdGx5IGEgbm8tb3AgKCRzJyA9IFxsYW5nbGUgZV5JLCBlXlRccmFuZ2xlJCksIHdoaWNoIHRoZSBnYXRlIGNoZWNrcy4KCioqUmV0cmlldmFsLioqIFJhbmsgYWxsIGl0ZW1zIGJ5IHRoZSBkdWFsIHNjb3JlLCB0aGVuIHJlLXJhbmsgZWFjaCBxdWVyeSdzIHRvcC0kSyQgKCRLPTE2JCkgYnkgJHMnJC4gQm90aCB0aGUgZHVhbCBhbmQgdGhlIHJlLXJhbmtlZCBtZXRyaWNzIGFyZSByZXBvcnRlZC4KCioqT2JqZWN0aXZlLioqCgokJFxtYXRoY2FsIEwgPSBcdGV4dHtJbmZvTkNFfV97XHRleHR7bXVsdGktcG9zfX0oZV5JLCBlXlQpICsgXGxhbWJkYV94XCwgXHRleHR7SW5mb05DRX1fe1x0ZXh0e211bHRpLXBvc319KHMnX3tVXHRpbWVzIE59KSArIFxiZXRhIFxzdW1fbSBcb3ZlcmxpbmV7XG1hdGhybXtLTH19KHFcLFx8XCxcbWF0aGNhbCBOKDAsSSkpLCQkCgp3aXRoICRcbGFtYmRhX3ggPSAxJCBhbmQgJFxiZXRhID0gMTBeey0zfSQuICRzJ197VVx0aW1lcyBOfSQgaXMgY29tcHV0ZWQgZm9yIGFsbCAkVXs9fTgkIGltYWdlcyAkXHRpbWVzJCAkTns9fTQwJCBjYXB0aW9ucyBpbiB0aGUgYmF0Y2guCgojIyAzLiBDb21wbGV4aXR5CgokZCQ6IHdpZHRoOyAkUiQ6IHBvdGVudGlhbCByYW5rOyAkS19zJDogSEVETyBzdGVwczsgJGRfcyQ6IFNTTSBzdGF0ZSBzaXplOyAkaCQ6IGF0dGVudGlvbiBoZWFkcy4KCnwgQ29tcG9uZW50IHwgVGltZSBwZXIgc2VxdWVuY2UgfAp8LS0tfC0tLXwKfCBFLUhFRE8gfCAkTyhLX3NcLCBMXCwgZCBSKSQsIGxpbmVhciBpbiAkTCQgKG5vIHRva2VuIGludGVyYWN0aW9uKSB8CnwgTWFtYmEtMiBwYXNzIHwgJE8oTFwsIGRcLCBkX3MpJCwgbGluZWFyIGluICRMJCB8CnwgWC1IVlNDIHBhc3MgMiB8IG9uZSBleHRyYSBNYW1iYS0yIHBhc3MgcGx1cyBhdHRlbnRpb24gb3ZlciBjaHVua3MsICRPXGJpZyhMXCwgZFwsIGRfcyArIFx0ZnJhY3tMX0l9e0N9XHRmcmFje0xfVH17Q31cLCBkXGJpZykkOiBsaW5lYXIgaW4gJExfSSQgZm9yIGEgZml4ZWQgY2FwdGlvbiB8CnwgUmV0cmlldmFsLCAkTl9JJCBpbWFnZXMgYW5kICROX1QkIGNhcHRpb25zIHwgZHVhbCAkTyhOX0kgTl9UIGQpJCBkb3QgcHJvZHVjdHMsIHBsdXMgJEsgKE5fSSArIE5fVCkkIHBhc3MtMiBldmFsdWF0aW9ucyAobm90ICROX0kgTl9UJCkgfAoKYHNjYWxpbmcucHlgIG1lYXN1cmVzIHRoZSBhY3R1YWwgc2xvcGUgKEg0KS4gSW50cmEtY2h1bmsgY29tcHV0YXRpb24gc3RheXMgdGhlIG5hdGl2ZSBNYW1iYS0yIHNjYW46IG1lc3NhZ2VzIGVudGVyIG9ubHkgYXMgYWRkaXRpdmUgaW5wdXRzIGF0IGNodW5rLXN0YXJ0IHRva2Vucy4KCiMjIDQuIENvbXB1dGF0aW9uYWwgZ3JhcGgKCmBgYApWaVQtQi8xNiAoZnJvemVuLCBjYWNoZWQpIC0+IExpbmVhciA3NjgtPjEyOCAtPiBFLUhFRE8gLS0rCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB8LT4gTWFtYmEtMiAocGFzcyAxKSAtPiBib3VuZGFyeSBzdGF0ZXMgLT4gcSh6fHMpIC0tKy0tPiBkdWFsIGVtYmVkZGluZyBlXkkKUm9CRVJUYSAoZnJvemVuLCBjYWNoZWQpICAtPiBMaW5lYXIgNzY4LT4xMjggLT4gRS1IRURPIC0tKyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgfAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjaHVuay1zdGFydCBtZXNzYWdlcyBNSEEoc197ay0xfSwgel57b3RoZXJ9KSAgLT4gTWFtYmEtMiAocGFzcyAyLCBzYW1lIHdlaWdodHMpIC0+IGUnXkksIGUnXlQgLT4gcycKYGBgCgojIyA1LiBIeXBvdGhlc2VzOiBvcGVyYXRpb25hbCBjcml0ZXJpYSAoZml4ZWQgYmVmb3JlIHJ1bm5pbmc7IGltcGxlbWVudGVkIGluIGBoeXBvdGhlc2VzLnB5YCkKClZhcmlhbnQgbmFtZXM6ICpvdXJzKiA9IGBmdWxsX3hgLCAqYmFzZWxpbmUqID0gYGJhc2VsaW5lYC4gQ0lzIGFyZSA5NSUgdC1pbnRlcnZhbHMgb3ZlciBtYXRjaGVkIHNlZWRzLCB3aXRoIGF0IGxlYXN0IDMgc2VlZHMgKDUgcGxhbm5lZCkuCgotICoqSDEg4oCUIGVuZXJneSBkaXNzaXBhdGlvbi4qKiBBbGwgZm91ciBtdXN0IGhvbGQ6CiAgLSAoYSkgbWF4IGZsb2F0NjQgcGVyLXN0ZXAgZW5lcmd5IGluY3JlYXNlIG9uIHJlYWwgdGVzdCB0b2tlbnMg4omkIDFlLTkuCiAgLSAoYikgYXR0ZW51YXRpb24oZm9yZWdyb3VuZCkg4oiSIGF0dGVudWF0aW9uKGJhY2tncm91bmQpID4gMCwgQ0kgZXhjbHVkaW5nIDAuIEZvcmVncm91bmQgPSB0b3AtMjUlIFZpVCBDTFMtYXR0ZW50aW9uIHBhdGNoZXMsIGJhY2tncm91bmQgPSBib3R0b20gNTAlLgogIC0gKGMxKSBNUihvdXJzKSDiiJIgTVIoYmFzZWxpbmUpIENJIGxvd2VyIGJvdW5kID4g4oiSMS4wIChub24taW5mZXJpb3JpdHkpLgogIC0gKGMyKSBNUihiYWNrZ3JvdW5kIG9jY2x1ZGVkKSDiiJIgTVIoZm9yZWdyb3VuZCBvY2NsdWRlZCkgQ0kgbG93ZXIgPiAwIGZvciBvdXJzLgotICoqSDIg4oCUIG1vZGFsaXR5IGRvbWluYW5jZS4qKiBCb3RoIG11c3QgaG9sZDoKICAtIChhKSBtZWFuIHxsb2cxMCAo4oCW4oiCTC/iiIJ4X2ltZ+KAliAvIOKAluKIgkwv4oiCeF90eHTigJYpfCBvdmVyIHRoZSBsYXN0IGVwb2NoIGlzIGxvd2VyIGZvciBvdXJzIHRoYW4gZm9yIGBmdWxsX3hfbm9leGNoYW5nZWAsIENJIHVwcGVyIDwgMC4KICAtIChiKSBib3RoIGV4Y2hhbmdlIGdhdGVzIGFyZSBub24temVybywgYW5kIHRoZSBtZXNzYWdlLXJlbGlhbmNlIGRvbWluYW5jZSBpbmRleCAkfHJfSSAtIHJfVHwvKHJfSSArIHJfVCkkIGF2ZXJhZ2VzIGJlbG93IDAuNS4KLSAqKkgzIOKAlCByb2J1c3RuZXNzLioqIFJlbGF0aXZlIE1SICgkXHRleHR7TVJ9X3tcdGV4dHtjb3JydXB0fX0vXHRleHR7TVJ9X3tcdGV4dHtjbGVhbn19JCksIGF2ZXJhZ2VkIG92ZXIgdGhlIGltYWdlIGNvcnJ1cHRpb25zIGFuZCBzZXBhcmF0ZWx5IG92ZXIgdGhlIHRleHQgY29ycnVwdGlvbnMgaW4gYHJvYnVzdG5lc3MucHlgOiBvdXJzIOKIkiBiYXNlbGluZSwgQ0kgbG93ZXIgPiAwIGZvciBib3RoIG1vZGFsaXRpZXMuCi0gKipINCDigJQgZWZmaWNpZW5jeS4qKiBCb3RoIG11c3QgaG9sZDoKICAtIChhKSBoZWFkIGluZmVyZW5jZSBsYXRlbmN5IGFuZCB0cmFpbmluZy1zdGVwIHRpbWUgYXQgMTk2IHRva2VucyDiiaQgMS41w5cgYmFzZWxpbmUuCiAgLSAoYikgbG9nLWxvZyBzbG9wZSBvZiBsYXRlbmN5IGFnYWluc3QgaW1hZ2UgbGVuZ3RoICg3ODQgdG8gMTIsNTQ0IHRva2Vucykg4omkIDEuMi4KCkEgbmVnYXRpdmUgdmVyZGljdCBpcyByZXBvcnRlZCBhcyBzdWNoLgoKIyMgNi4gU2NvcGUgYW5kIGhvbmVzdCBsaW1pdGF0aW9ucwoKLSAqKlRhc2s6KiogaW1hZ2XigJN0ZXh0IHJldHJpZXZhbCBvbiBGbGlja3I4ayAoMUsgdGVzdCksIHdpdGggZnJvemVuIFZpVC1CLzE2IGFuZCBSb0JFUlRhLWJhc2UuIE1ldHJpY3MgYXJlIFJAMS81LzEwLCBNZWRSLCBhbmQgbWVhbiByZWNhbGwuIEFjY3VyYWN5LCBGMSwgQVVST0MgYW5kIG1Jb1UgZnJvbSB0aGUgb3JpZ2luYWwgcHJvcG9zYWwgZG8gbm90IGFwcGx5IHRvIHRoaXMgdGFzay4gQmFzZWxpbmVzIGZyb20gb3RoZXIgdGFza3MgKE1hbWJhQUQsIFRpbWVWaXBlciwg4oCmKSBhcmUgbm90IGNvbXBhcmFibGU7IHRoZSBjb250cm9scyBhcmUgcmV0cmFpbmVkIHVuZGVyIGFuIGlkZW50aWNhbCBwcm90b2NvbCBpbnN0ZWFkIChwYXJhbS1tYXRjaGVkIE1hbWJhLTIsIHBhcmFtLW1hdGNoZWQgVHJhbnNmb3JtZXIsIFRyYW5zZm9ybWVyIHdpdGggb3VyIG1vZHVsZXMsIG5vLW1peGVyKS4KLSAqKlJlc29sdXRpb246KiogYmFja2JvbmVzIGFyZSBmaXhlZCBhdCAxOTYgcGF0Y2hlcy4gSDQgYXQgaGlnaGVyIGxlbmd0aHMgdXNlcyByYW5kb20gdGVuc29ycyBvZiB0b2tlbiBzaGFwZSwgc28gaXQgbWVhc3VyZXMgY29tcHV0ZSwgbm90IGFjY3VyYWN5LgotICoqUGFpcndpc2UgY29zdDoqKiBleGNoYW5nZSBzY29yZXMgYXJlIHBhaXJ3aXNlLiBUaGUgcmV0cmlldmFsIGNvc3QgaXMgc3RhdGVkIGFzIGR1YWwgKyB0b3AtSyByZS1yYW5raW5nLCBub3QgYXMgYSBwdXJlIGR1YWwgZW5jb2Rlci4KLSAqKkZvcmVncm91bmQgcHJveHk6KiogImJhY2tncm91bmQiIGlzIGRlZmluZWQgYnkgYSBzYWxpZW5jeSBwcm94eSAoVmlUIGF0dGVudGlvbiksIG5vdCBieSBodW1hbiBtYXNrcy4K"], "metrics.py": ["58c0dbe7366251feb02549d3bf7f5d4c067edab0cfcff1f25ab016f9825034b1", "IiIiUmV0cmlldmFsIG1ldHJpY3MgdW5kZXIgdGhlIGNhbm9uaWNhbCBvcmRlcjogaW1hZ2UgaSBvd25zIGNhcHRpb25zIDVpLi41aSs0LgoKUmFua3MgYXJlIDAtYmFzZWQ7IE1lZFIgLyBNZWFuUiBhcmUgcmVwb3J0ZWQgMS1iYXNlZC4gaTJ0IHVzZXMgdGhlIGJlc3QtcmFua2VkCm9mIHRoZSA1IGdyb3VuZC10cnV0aCBjYXB0aW9ucyAoc3RhbmRhcmQgRmxpY2tyIHByb3RvY29sKS4KClJlLXJhbmtpbmcgKGV4Y2hhbmdlIG1vZGVscyk6IGZvciBlYWNoIHF1ZXJ5IHRoZSB0b3AtSyBnYWxsZXJ5IGl0ZW1zIGJ5IGR1YWwKKHBhc3MtMSkgc2ltaWxhcml0eSBhcmUgcmUtb3JkZXJlZCBieSB0aGUgZXhjaGFuZ2Ugc2NvcmU7IGl0ZW1zIG91dHNpZGUgdGhlCnRvcC1LIGtlZXAgdGhlaXIgZHVhbCBvcmRlciBhZnRlciB0aGVtLgoiIiIKCmltcG9ydCBudW1weSBhcyBucAoKQ0FQVElPTlNfUEVSX0lNQUdFID0gNQpSRUNBTExfS1MgPSAoMSwgNSwgMTApCgoKZGVmIGNhcHRpb25fb3duZXIobl9pbWcsIGs9Q0FQVElPTlNfUEVSX0lNQUdFKToKICAgIHJldHVybiBucC5yZXBlYXQobnAuYXJhbmdlKG5faW1nKSwgaykKCgpkZWYgX3JhbmtzX2Zyb21fb3JkZXJzKG9yZGVyX2kydCwgb3JkZXJfdDJpLCBvd25lcik6CiAgICBuX2ltZyA9IG9yZGVyX2kydC5zaGFwZVswXQogICAgaTJ0ID0gKG93bmVyW29yZGVyX2kydF0gPT0gbnAuYXJhbmdlKG5faW1nKVs6LCBOb25lXSkuYXJnbWF4KGF4aXM9MSkKICAgIHQyaSA9IChvcmRlcl90MmkgPT0gb3duZXJbOiwgTm9uZV0pLmFyZ21heChheGlzPTEpCiAgICByZXR1cm4gaTJ0LCB0MmkKCgpkZWYgcmV0cmlldmFsX3JhbmtzKGltZywgdHh0LCBrPUNBUFRJT05TX1BFUl9JTUFHRSk6CiAgICBuX2ltZyA9IGltZy5zaGFwZVswXQogICAgYXNzZXJ0IHR4dC5zaGFwZVswXSA9PSBrICogbl9pbWcsIChpbWcuc2hhcGUsIHR4dC5zaGFwZSkKICAgIHNpbSA9IGltZy5hc3R5cGUobnAuZmxvYXQ2NCkgQCB0eHQuYXN0eXBlKG5wLmZsb2F0NjQpLlQKICAgIHJldHVybiBfcmFua3NfZnJvbV9vcmRlcnMobnAuYXJnc29ydCgtc2ltLCBheGlzPTEsIGtpbmQ9InN0YWJsZSIpLCBucC5hcmdzb3J0KC1zaW0uVCwgYXhpcz0xLCBraW5kPSJzdGFibGUiKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2FwdGlvbl9vd25lcihuX2ltZywgaykpCgoKZGVmIHRvcGtfY2FuZGlkYXRlcyhpbWcsIHR4dCwgayk6CiAgICBzaW0gPSBpbWcuYXN0eXBlKG5wLmZsb2F0NjQpIEAgdHh0LmFzdHlwZShucC5mbG9hdDY0KS5UCiAgICBpMnQgPSBucC5hcmdzb3J0KC1zaW0sIGF4aXM9MSwga2luZD0ic3RhYmxlIilbOiwgOmtdCiAgICB0MmkgPSBucC5hcmdzb3J0KC1zaW0uVCwgYXhpcz0xLCBraW5kPSJzdGFibGUiKVs6LCA6a10KICAgIHJldHVybiBpMnQsIHQyaQoKCmRlZiBfcmVyYW5rX29yZGVycyhzaW0sIHRvcGtfaWR4LCB0b3BrX3Njb3Jlcyk6CiAgICBvcmRlcnMgPSBucC5hcmdzb3J0KC1zaW0sIGF4aXM9MSwga2luZD0ic3RhYmxlIikKICAgIG91dCA9IG5wLmVtcHR5X2xpa2Uob3JkZXJzKQogICAgayA9IHRvcGtfaWR4LnNoYXBlWzFdCiAgICBmb3IgciBpbiByYW5nZShzaW0uc2hhcGVbMF0pOgogICAgICAgIGhlYWQgPSB0b3BrX2lkeFtyXVtucC5hcmdzb3J0KC10b3BrX3Njb3Jlc1tyXSwga2luZD0ic3RhYmxlIildCiAgICAgICAgcmVzdCA9IG9yZGVyc1tyXVt+bnAuaXNpbihvcmRlcnNbcl0sIGhlYWQpXQogICAgICAgIG91dFtyLCA6a10sIG91dFtyLCBrOl0gPSBoZWFkLCByZXN0CiAgICByZXR1cm4gb3V0CgoKZGVmIHJlcmFua2VkX3JhbmtzKGltZywgdHh0LCByZXJhbmspOgogICAgbl9pbWcgPSBpbWcuc2hhcGVbMF0KICAgIHNpbSA9IGltZy5hc3R5cGUobnAuZmxvYXQ2NCkgQCB0eHQuYXN0eXBlKG5wLmZsb2F0NjQpLlQKICAgIG9yZGVyX2kydCA9IF9yZXJhbmtfb3JkZXJzKHNpbSwgcmVyYW5rWyJpMnRfaWR4Il0sIHJlcmFua1siaTJ0X3Njb3JlIl0pCiAgICBvcmRlcl90MmkgPSBfcmVyYW5rX29yZGVycyhzaW0uVCwgcmVyYW5rWyJ0MmlfaWR4Il0sIHJlcmFua1sidDJpX3Njb3JlIl0pCiAgICByZXR1cm4gX3JhbmtzX2Zyb21fb3JkZXJzKG9yZGVyX2kydCwgb3JkZXJfdDJpLCBjYXB0aW9uX293bmVyKG5faW1nKSkKCgpkZWYgbWV0cmljc19mcm9tX3JhbmtzKGkydCwgdDJpKToKICAgIG0gPSB7fQogICAgZm9yIG5hbWUsIHIgaW4gKCgiaTJ0IiwgaTJ0KSwgKCJ0MmkiLCB0MmkpKToKICAgICAgICBmb3IgayBpbiBSRUNBTExfS1M6CiAgICAgICAgICAgIG1bZiJ7bmFtZX1fcntrfSJdID0gZmxvYXQobnAubWVhbihyIDwgaykgKiAxMDAuMCkKICAgICAgICBtW2Yie25hbWV9X21lZHIiXSA9IGZsb2F0KG5wLm1lZGlhbihyICsgMSkpCiAgICAgICAgbVtmIntuYW1lfV9tZWFuciJdID0gZmxvYXQobnAubWVhbihyICsgMSkpCiAgICBtWyJtZWFuX3JlY2FsbCJdID0gZmxvYXQobnAubWVhbihbbVtmIntkfV9ye2t9Il0gZm9yIGQgaW4gKCJpMnQiLCAidDJpIikgZm9yIGsgaW4gUkVDQUxMX0tTXSkpCiAgICByZXR1cm4gbQoKCmRlZiByZXRyaWV2YWxfbWV0cmljcyhpbWcsIHR4dCwgcmVyYW5rPU5vbmUpOgogICAgIiIiUHJpbWFyeSBtZXRyaWNzIChyZS1yYW5rZWQgd2hlbiBgcmVyYW5rYCBpcyBnaXZlbikgcGx1cyB0aGUgZHVhbCBtZXRyaWNzIHVuZGVyICJkdWFsXyoiLiIiIgogICAgZHVhbCA9IG1ldHJpY3NfZnJvbV9yYW5rcygqcmV0cmlldmFsX3JhbmtzKGltZywgdHh0KSkKICAgIGlmIHJlcmFuayBpcyBOb25lOgogICAgICAgIHJldHVybiB7KipkdWFsLCAqKntmImR1YWxfe2t9IjogdiBmb3IgaywgdiBpbiBkdWFsLml0ZW1zKCl9fQogICAgcHJpbWFyeSA9IG1ldHJpY3NfZnJvbV9yYW5rcygqcmVyYW5rZWRfcmFua3MoaW1nLCB0eHQsIHJlcmFuaykpCiAgICByZXR1cm4geyoqcHJpbWFyeSwgKip7ZiJkdWFsX3trfSI6IHYgZm9yIGssIHYgaW4gZHVhbC5pdGVtcygpfX0K"], "models.py": ["..."], "probe.py": ["b75ff0895aa7515b812f7a25323340d08b749e97a8c59c36b9550d8c4978bdc3", "IiIiTWVjaGFuaXNtIHByb2JlcyBmb3IgSDEgKGVuZXJneSBkaXNzaXBhdGlvbiAvIGJhY2tncm91bmQgc3VwcHJlc3Npb24pIGFuZCBIMiAobW9kYWxpdHkgZG9taW5hbmNlKS4KClBlciBjb21wbGV0ZWQgcnVuIChiZXN0IGNoZWNrcG9pbnQsIHRlc3Qgc3BsaXQpIHdyaXRlcyA8cnVuPi9wcm9iZS5qc29uOyBhbGwgcm93cyBnbyB0bwpyZXN1bHRzL3Byb2Jlcy5jc3YuCgpIMSAoYWxsIHJ1bnM7IGVuZXJneSB0ZXJtcyBvbmx5IGZvciBFbmVyZ3lIRURPKToKICBzYWxpZW5jeSAgIGxhc3QtYmxvY2sgVmlUIENMUy0+cGF0Y2ggYXR0ZW50aW9uLCBwZXIgaW1hZ2UuIGJhY2tncm91bmQgPSB0b2tlbnMgYXQgb3IgYmVsb3cgdGhlCiAgICAgICAgICAgICBpbWFnZSBtZWRpYW47IGZvcmVncm91bmQgPSB0b3AgMjUlLgogIG5vcm1fcmF0aW8gfHxvcGVyYXRvcih4KXx8IC8gfHx4fHwgYXZlcmFnZWQgb3ZlciBiZyBhbmQgZmcgdG9rZW5zIChpZGVudGl0eSBvcGVyYXRvciAtPiAxKQogIGF0dGVudWF0aW9uIC8gZGlzc2lwYXRlZCBlbmVyZ3kgb24gYmcgdnMgZmcsIFNwZWFybWFuKHNhbGllbmN5LCBhdHRlbnVhdGlvbikKICBlbmVyZ3lfbWF4X2luY3JlYXNlX2Zsb2F0NjQ6IGxhcmdlc3QgcGVyLXN0ZXAgaW5jcmVhc2Ugb2YgSCBvdmVyIDY0IHRlc3QgaW1hZ2VzIGluIGZsb2F0NjQKICAgICAgICAgICAgIChUaGVvcmVtIDEgcHJlZGljdHMgPD0gMCB1cCB0byByb3VuZC1vZmYpCiAgb2NjbHVzaW9uICBkdWFsIHRlc3QgbWVhbiByZWNhbGwgd2l0aCBiYWNrZ3JvdW5kIHRva2VucyB6ZXJvZWQgdnMgZm9yZWdyb3VuZCB0b2tlbnMgemVyb2VkCkgyOgogIGdyYWRfbG9nMTBfcmF0aW8gIG1lYW4gbG9nMTAofHxkTC9keF9pbWd8fCAvIHx8ZEwvZHhfdHh0fHwpIG92ZXIgdGhlIGxhc3QgdHJhaW5pbmcgZXBvY2ggKHwufCA9IGltYmFsYW5jZSkKICBleGNoYW5nZSBydW5zOiAgICBtZXNzYWdlIHJlbGlhbmNlIHJfaW1nID0gbWVhbiB8cyhmdWxsKSAtIHMobm8gbWVzc2FnZSBpbnRvIGltYWdlIHN0cmVhbSl8IG9uIHRoZQogICAgICAgICAgICAgICAgICAgIDUwMDAgcG9zaXRpdmUgdGVzdCBwYWlycywgcl90eHQgbGlrZXdpc2U7IGRvbWluYW5jZSBEID0gfHJfaW1nIC0gcl90eHR8IC8gKHJfaW1nICsgcl90eHQpCiAgZWZmZWN0aXZlX3JhbmsgICAgZXhwKGVudHJvcHkgb2Ygbm9ybWFsaXNlZCBlaWdlbnZhbHVlcykgb2YgdGhlIHRlc3QgZW1iZWRkaW5nIGNvdmFyaWFuY2UgKHNhdHVyYXRpb24gcHJveHkpCgpVc2FnZTogcHl0aG9uIHByb2JlLnB5IFstLXJ1bnMtZ2xvYiAiKnNlZWQqIl0KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IG9zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSBzY2lweSBpbXBvcnQgc3RhdHMKCmZyb20gY29tbW9uIGltcG9ydCBGRUFUVVJFX0RJUiwgUlVOU19ESVIsIGJhbm5lciwgcHJvdmVuYW5jZSwgcmVhZF9qc29uLCB3cml0ZV9qc29uCmZyb20gZXZhbHVhdGlvbiBpbXBvcnQgQ2FjaGVkU3BsaXQsIGV2YWx1YXRlLCBleGNoYW5nZV9zY29yZXMsIHBhc3MxX2FsbCwgbG9hZF9ydW5fbW9kZWwKZnJvbSBtb2RlbHMgaW1wb3J0IEVuZXJneUhFRE8KCgpkZWYgZWZmZWN0aXZlX3JhbmsoeCk6CiAgICB4ID0geCAtIHgubWVhbigwLCBrZWVwZGltcz1UcnVlKQogICAgZXYgPSBucC5jbGlwKG5wLmxpbmFsZy5laWd2YWxzaChucC5jb3YoeC5UKSksIDAsIE5vbmUpCiAgICBwID0gZXYgLyBldi5zdW0oKQogICAgcCA9IHBbcCA+IDBdCiAgICByZXR1cm4gZmxvYXQobnAuZXhwKC0ocCAqIG5wLmxvZyhwKSkuc3VtKCkpKQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGgxX3Byb2JlKG1vZGVsLCB0ZXN0LCBzYWxpZW5jeSwgZGV2aWNlKToKICAgIG5faW1hZ2VzID0gdGVzdC5pbWcuc2hhcGVbMF0KICAgIG9wID0gbW9kZWwub3BlcmF0b3JbImltZyJdCiAgICBmZ19tYXNrID0gc2FsaWVuY3kgPj0gbnAucXVhbnRpbGUoc2FsaWVuY3ksIDAuNzUsIGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKICAgIGJnX21hc2sgPSBzYWxpZW5jeSA8PSBucC5tZWRpYW4oc2FsaWVuY3ksIGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKICAgIGFjYyA9IHsibm9ybV9yYXRpbyI6IFtdLCAiYXR0ZW51YXRpb24iOiBbXSwgImRpc3NpcGF0ZWRfZnJhY3Rpb24iOiBbXX0KICAgIGZvciBzIGluIHJhbmdlKDAsIG5faW1hZ2VzLCAxMDApOgogICAgICAgIHJvd3MgPSBucC5hcmFuZ2UocywgbWluKG5faW1hZ2VzLCBzICsgMTAwKSkKICAgICAgICB4ID0gbW9kZWwucHJvalsiaW1nIl0odGVzdC5pbWdfdGVuc29yKHJvd3MsIGRldmljZSkpCiAgICAgICAgaWYgaXNpbnN0YW5jZShvcCwgRW5lcmd5SEVETyk6CiAgICAgICAgICAgIHQgPSBvcC50cmFqZWN0b3J5KHgpCiAgICAgICAgICAgIHkgPSB0WyJvdXQiXQogICAgICAgICAgICBhY2NbImF0dGVudWF0aW9uIl0uYXBwZW5kKHRbImF0dGVudWF0aW9uIl0uY3B1KCkubnVtcHkoKSkKICAgICAgICAgICAgYWNjWyJkaXNzaXBhdGVkX2ZyYWN0aW9uIl0uYXBwZW5kKHRbImRpc3NpcGF0ZWRfZnJhY3Rpb24iXS5jcHUoKS5udW1weSgpKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHkgPSBvcCh4KQogICAgICAgIGFjY1sibm9ybV9yYXRpbyJdLmFwcGVuZCgoeS5ub3JtKGRpbT0tMSkgLyB4Lm5vcm0oZGltPS0xKS5jbGFtcChtaW49MWUtMTIpKS5jcHUoKS5udW1weSgpKQogICAgcmVzID0ge30KICAgIGZnLCBiZywgc2FsID0gZmdfbWFza1s6bl9pbWFnZXNdLCBiZ19tYXNrWzpuX2ltYWdlc10sIHNhbGllbmN5WzpuX2ltYWdlc10KICAgIGZvciBuYW1lLCBjaHVua3MgaW4gYWNjLml0ZW1zKCk6CiAgICAgICAgaWYgbm90IGNodW5rczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICB2ID0gbnAuY29uY2F0ZW5hdGUoY2h1bmtzKQogICAgICAgIHJlc1tmIntuYW1lfV9iZyJdID0gZmxvYXQodltiZ10ubWVhbigpKQogICAgICAgIHJlc1tmIntuYW1lfV9mZyJdID0gZmxvYXQodltmZ10ubWVhbigpKQogICAgICAgIHJlc1tmIntuYW1lfV9zcGVhcm1hbl9zYWxpZW5jeSJdID0gZmxvYXQoc3RhdHMuc3BlYXJtYW5yKHNhbC5yZXNoYXBlKC0xKSwgdi5yZXNoYXBlKC0xKSkuc3RhdGlzdGljKQogICAgaWYgaXNpbnN0YW5jZShvcCwgRW5lcmd5SEVETyk6CiAgICAgICAgb3A2NCA9IEVuZXJneUhFRE8ob3AuVS5zaGFwZVsxXSwgb3AuVS5zaGFwZVswXSwgb3Auc3RlcHMsIG9wLmR0X21heCwgb3AuYWRhcHRpdmUsIG9wLmVwcykudG8oZGV2aWNlKS5kb3VibGUoKQogICAgICAgIG9wNjQubG9hZF9zdGF0ZV9kaWN0KHtrOiB2LmRvdWJsZSgpIGZvciBrLCB2IGluIG9wLnN0YXRlX2RpY3QoKS5pdGVtcygpfSkKICAgICAgICBlID0gb3A2NC50cmFqZWN0b3J5KG1vZGVsLnByb2pbImltZyJdKHRlc3QuaW1nX3RlbnNvcihucC5hcmFuZ2UobWluKDY0LCBuX2ltYWdlcykpLCBkZXZpY2UpKS5kb3VibGUoKSlbImVuZXJnaWVzIl0KICAgICAgICByZXNbImVuZXJneV9tYXhfaW5jcmVhc2VfZmxvYXQ2NCJdID0gZmxvYXQoKGVbLi4uLCAxOl0gLSBlWy4uLiwgOi0xXSkubWF4KCkpCiAgICAgICAgcmVzWyJlbmVyZ3lfbWVhbl9pbml0aWFsIl0gPSBmbG9hdChlWy4uLiwgMF0ubWVhbigpKQogICAgICAgIHJlc1siZW5lcmd5X21lYW5fZmluYWwiXSA9IGZsb2F0KGVbLi4uLCAtMV0ubWVhbigpKQogICAgcmV0dXJuIHJlcywgZmdfbWFzaywgYmdfbWFzawoKCmRlZiBvY2NsdXNpb25fbWV0cmljcyhtb2RlbCwgdGVzdCwgZGV2aWNlLCBmZ19tYXNrLCBiZ19tYXNrKToKICAgIHJlcyA9IHt9CiAgICBmb3IgbmFtZSwgbSBpbiAoKCJvY2NsdWRlX2JnIiwgYmdfbWFzayksICgib2NjbHVkZV9mZyIsIGZnX21hc2spKToKICAgICAgICBrZWVwID0gdG9yY2guZnJvbV9udW1weSh+bSkudG8oZGV2aWNlKQoKICAgICAgICBkZWYgdHJhbnNmb3JtKGZlYXRzLCByb3dzLCBrZWVwPWtlZXApOgogICAgICAgICAgICByZXR1cm4gZmVhdHMgKiBrZWVwW3RvcmNoLmFzX3RlbnNvcihyb3dzLCBkZXZpY2U9ZGV2aWNlKV0udW5zcXVlZXplKC0xKS50byhmZWF0cy5kdHlwZSkKCiAgICAgICAgcmVzW2Yie25hbWV9X2R1YWxfbWVhbl9yZWNhbGwiXSA9IGV2YWx1YXRlKG1vZGVsLCB0ZXN0LCBkZXZpY2UsIHJlcmFua19rPTAsIGltZ190cmFuc2Zvcm09dHJhbnNmb3JtKVswXVsKICAgICAgICAgICAgImR1YWxfbWVhbl9yZWNhbGwiXQogICAgcmV0dXJuIHJlcwoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGgyX3Byb2JlKG1vZGVsLCB0ZXN0LCBkZXZpY2UsIHJ1bl9kaXIpOgogICAgcmVzID0ge30KICAgIGxvZyA9IHBkLnJlYWRfanNvbihydW5fZGlyIC8gImJhdGNoX2xvZy5qc29ubCIsIGxpbmVzPVRydWUpCiAgICBsYXN0ID0gbG9nW2xvZ1siZXBvY2giXSA9PSBsb2dbImVwb2NoIl0ubWF4KCldCiAgICBpZiB7ImdyYWRfaW1nX3Rva2VuIiwgImdyYWRfdHh0X3Rva2VuIn0gPD0gc2V0KGxhc3QuY29sdW1ucyk6CiAgICAgICAgciA9IG5wLmxvZzEwKGxhc3RbImdyYWRfaW1nX3Rva2VuIl0gLyBsYXN0WyJncmFkX3R4dF90b2tlbiJdKQogICAgICAgIHJlc1siZ3JhZF9sb2cxMF9yYXRpbyJdID0gZmxvYXQoci5tZWFuKCkpCiAgICAgICAgcmVzWyJncmFkX2Fic19sb2cxMF9yYXRpbyJdID0gZmxvYXQoci5hYnMoKS5tZWFuKCkpCiAgICBpbWcsIHR4dCwgY2FjaGUgPSBwYXNzMV9hbGwobW9kZWwsIHRlc3QsIGRldmljZSkKICAgIHJlc1siZWZmZWN0aXZlX3JhbmtfaW1nIl0gPSBlZmZlY3RpdmVfcmFuayhpbWcpCiAgICByZXNbImVmZmVjdGl2ZV9yYW5rX3R4dCJdID0gZWZmZWN0aXZlX3JhbmsodHh0KQogICAgaWYgY2FjaGUgaXMgbm90IE5vbmU6CiAgICAgICAgbiA9IHR4dC5zaGFwZVswXQogICAgICAgIGlyLCB0ciA9IG5wLmFyYW5nZShuKSAvLyA1LCBucC5hcmFuZ2UobikKICAgICAgICBmdWxsID0gZXhjaGFuZ2Vfc2NvcmVzKG1vZGVsLCBjYWNoZSwgaXIsIHRyLCBkZXZpY2UpCiAgICAgICAgbm9faW1nID0gZXhjaGFuZ2Vfc2NvcmVzKG1vZGVsLCBjYWNoZSwgaXIsIHRyLCBkZXZpY2UsIG1lc3NhZ2Vfc2NhbGU9KDAuMCwgMS4wKSkKICAgICAgICBub190eHQgPSBleGNoYW5nZV9zY29yZXMobW9kZWwsIGNhY2hlLCBpciwgdHIsIGRldmljZSwgbWVzc2FnZV9zY2FsZT0oMS4wLCAwLjApKQogICAgICAgIHJfaW1nLCByX3R4dCA9IGZsb2F0KG5wLmFicyhmdWxsIC0gbm9faW1nKS5tZWFuKCkpLCBmbG9hdChucC5hYnMoZnVsbCAtIG5vX3R4dCkubWVhbigpKQogICAgICAgIHJlcy51cGRhdGUocmVsaWFuY2VfaW1nX29uX3R4dD1yX2ltZywgcmVsaWFuY2VfdHh0X29uX2ltZz1yX3R4dCwKICAgICAgICAgICAgICAgICAgIGRvbWluYW5jZV9pbmRleD1hYnMocl9pbWcgLSByX3R4dCkgLyBtYXgocl9pbWcgKyByX3R4dCwgMWUtMTIpLAogICAgICAgICAgICAgICAgICAgZ2F0ZV9pbWc9ZmxvYXQodG9yY2gudGFuaChtb2RlbC54aHZzYy5nYXRlWyJpbWciXSkpLAogICAgICAgICAgICAgICAgICAgZ2F0ZV90eHQ9ZmxvYXQodG9yY2gudGFuaChtb2RlbC54aHZzYy5nYXRlWyJ0eHQiXSkpKQogICAgcmV0dXJuIHJlcwoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ydW5zLWdsb2IiLCBkZWZhdWx0PSIqIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKICAgIGRldmljZSA9IHRvcmNoLmRldmljZShvcy5lbnZpcm9uLmdldCgiSEVET19ERVZJQ0UiLCAiY3VkYSIpKQogICAgYmFubmVyKCJNRUNIQU5JU00gUFJPQkVTIChIMSwgSDIpIikKICAgIHRlc3QgPSBDYWNoZWRTcGxpdCgidGVzdCIpCiAgICBzYWxpZW5jeSA9IG5wLmxvYWQoRkVBVFVSRV9ESVIgLyAic2FsaWVuY3lfdGVzdC5ucHkiKQogICAgcm93cyA9IFtdCiAgICBmb3Igc3VtbWFyeV9wYXRoIGluIHNvcnRlZChSVU5TX0RJUi5nbG9iKGYie2FyZ3MucnVuc19nbG9ifS9ydW5fc3VtbWFyeS5qc29uIikpOgogICAgICAgIHJ1bl9kaXIgPSBzdW1tYXJ5X3BhdGgucGFyZW50CiAgICAgICAgcyA9IHJlYWRfanNvbihzdW1tYXJ5X3BhdGgpCiAgICAgICAgaWYgcy5nZXQoInN0YXR1cyIpICE9ICJDT01QTEVURUQiOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG1vZGVsID0gbG9hZF9ydW5fbW9kZWwocnVuX2RpciwgZGV2aWNlKQogICAgICAgIGgxLCBmZywgYmcgPSBoMV9wcm9iZShtb2RlbCwgdGVzdCwgc2FsaWVuY3ksIGRldmljZSkKICAgICAgICByb3cgPSB7InJ1biI6IHJ1bl9kaXIubmFtZSwgInZhcmlhbnQiOiBzWyJ2YXJpYW50Il0sICJzZWVkIjogc1sic2VlZCJdLCAia2xfd2VpZ2h0Ijogc1sia2xfd2VpZ2h0Il0sCiAgICAgICAgICAgICAgICJodnNjX2NodW5rX3NpemUiOiBzWyJodnNjX2NodW5rX3NpemUiXSwgKipoMSwgKipvY2NsdXNpb25fbWV0cmljcyhtb2RlbCwgdGVzdCwgZGV2aWNlLCBmZywgYmcpLAogICAgICAgICAgICAgICAqKmgyX3Byb2JlKG1vZGVsLCB0ZXN0LCBkZXZpY2UsIHJ1bl9kaXIpfQogICAgICAgIHdyaXRlX2pzb24ocnVuX2RpciAvICJwcm9iZS5qc29uIiwgeyoqcm93LCAicHJvdmVuYW5jZSI6IHByb3ZlbmFuY2UoKX0pCiAgICAgICAgcm93cy5hcHBlbmQocm93KQogICAgICAgIHByaW50KHtrOiAocm91bmQodiwgNCkgaWYgaXNpbnN0YW5jZSh2LCBmbG9hdCkgZWxzZSB2KSBmb3IgaywgdiBpbiByb3cuaXRlbXMoKX0sIGZsdXNoPVRydWUpCiAgICBvdXQgPSBSVU5TX0RJUi5wYXJlbnQgLyAicmVzdWx0cyIgLyAicHJvYmVzLmNzdiIKICAgIG91dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGQuRGF0YUZyYW1lKHJvd3MpLnRvX2NzdihvdXQsIGluZGV4PUZhbHNlKQogICAgcHJpbnQoZiJ7bGVuKHJvd3MpfSBydW5zIHByb2JlZCAtPiB7b3V0fSIpCiAgICBwcmludCgiUFJPQkVTOiBQQVNTIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="], "profile_efficiency.py": ["7571a30e9b64a429fbb13f6f982ea71ee3335b6f9fee7b6bd15371a1c26b847e", "IiIiRW5kLXRvLWVuZCBlZmZpY2llbmN5IHByb2ZpbGUgb24gcmVhbCBGbGlja3I4ayB0ZXN0IGlucHV0cywgc2FtZSBHUFUsIHNhbWUgYmF0Y2ggc2l6ZXMuCgpGb3IgZWFjaCB2YXJpYW50IGl0IHJlcG9ydHMKICAqIHBhcmFtZXRlcnM6IHRyYWluYWJsZSBoZWFkLCBmcm96ZW4gVmlULCBmcm96ZW4gUm9CRVJUYSwgdG90YWwKICAqIGluZmVyZW5jZSBsYXRlbmN5IChtZWRpYW4gLyBwOTAgbXMpIGFuZCB0aHJvdWdocHV0IGZvcjogVmlULCBSb0JFUlRhLCBoZWFkLCBhbmQgdGhlCiAgICBmdWxsIHBpcGVsaW5lIChpbWFnZSArIGNhcHRpb24gLT4gZW1iZWRkaW5ncyksIGF0IGJhdGNoIHNpemVzIDEgYW5kIDMyCiAgKiBwZWFrIGluZmVyZW5jZSBtZW1vcnkgYWJvdmUgdGhlIGxvYWRlZCB3ZWlnaHRzCiAgKiB0cmFpbmluZyB0aHJvdWdocHV0IG9mIHRoZSBoZWFkIG9uIGNhY2hlZCBmZWF0dXJlcyAoY2FwdGlvbnMvcyksIDggaW1hZ2VzIHggNSBjYXB0aW9ucwogICogRkxPUHMgZnJvbSB0b3JjaC51dGlscy5mbG9wX2NvdW50ZXIuIEN1c3RvbSBUcml0b24vQ1VEQSBrZXJuZWxzIChNYW1iYS0yIHNlbGVjdGl2ZSBzY2FuLAogICAgY2F1c2FsIGNvbnYxZCkgYXJlIE5PVCBjb3VudGVkLCBzbyBGTE9QcyBmb3IgTWFtYmEtMiB2YXJpYW50cyBhcmUgYSBsb3dlciBib3VuZC4KCklucHV0cyBhcmUgcmVhbCBwcmVwcm9jZXNzZWQgdGVzdCBpbWFnZXMgYW5kIHJlYWwgdG9rZW5pc2VkIHRlc3QgY2FwdGlvbnMsIG5ldmVyIHJhbmRvbSB0ZW5zb3JzLgpJZiBhIHRyYWluZWQgc2VlZC00MiBjaGVja3BvaW50IGV4aXN0cyBpdCBpcyBsb2FkZWQgKHdlaWdodHMgZG8gbm90IGNoYW5nZSB0aW1pbmcsIGJ1dCBudW1lcmljcyBzdGF5IHJlYWxpc3RpYykuCgpVc2FnZTogcHl0aG9uIHByb2ZpbGVfZWZmaWNpZW5jeS5weSBbLS12YXJpYW50cyBiYXNlbGluZSBmdWxsIC4uLl0gWy0tYmF0Y2gtc2l6ZXMgMSAzMl0KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHN0YXRpc3RpY3MKaW1wb3J0IHRpbWUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKZnJvbSB0b3JjaC51dGlscy5mbG9wX2NvdW50ZXIgaW1wb3J0IEZsb3BDb3VudGVyTW9kZQoKZnJvbSBiYWNrYm9uZXMgaW1wb3J0IEZyb3plblJvQkVSVGEsIEZyb3plblZpVCwgdml0X3RyYW5zZm9ybQpmcm9tIGNvbW1vbiBpbXBvcnQgREFUQV9ESVIsIEZFQVRVUkVfRElSLCBSRVBPUlRfRElSLCBSVU5TX0RJUiwgYmFubmVyLCBlbnZpcm9ubWVudF9tYW5pZmVzdCwgcHJvdmVuYW5jZSwgcmVhZF9qc29uLCB3cml0ZV9qc29uCmZyb20gbW9kZWxzIGltcG9ydCBSZXRyaWV2YWxNb2RlbCwgYnVpbGRfY29uZmlnLCBjb3VudF90cmFpbmFibGUsIHRvdGFsX2xvc3MKCkRFRkFVTFRfVkFSSUFOVFMgPSBbImJhc2VsaW5lIiwgImhlZG9fZW5lcmd5IiwgImh2c2NfeCIsICJmdWxsX3giLCAiZnVsbCIsICJiYXNlbGluZV9wYXJhbV9tYXRjaGVkX3giLAogICAgICAgICAgICAgICAgICAgICJub19taXhlcl9tZWFucG9vbCIsICJ0cmFuc2Zvcm1lcl9wYXJhbV9tYXRjaGVkIiwgInRyYW5zZm9ybWVyX3giXQoKCmRlZiB0aW1lZChmbiwgd2FybXVwLCBpdGVycyk6CiAgICBmb3IgXyBpbiByYW5nZSh3YXJtdXApOgogICAgICAgIGZuKCkKICAgIHRpbWVzID0gW10KICAgIGZvciBfIGluIHJhbmdlKGl0ZXJzKToKICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBmbigpCiAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICAgICAgdGltZXMuYXBwZW5kKCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMCkKICAgIHRpbWVzLnNvcnQoKQogICAgcmV0dXJuIHsibWVkaWFuX21zIjogc3RhdGlzdGljcy5tZWRpYW4odGltZXMpLCAicDkwX21zIjogdGltZXNbaW50KDAuOSAqIChsZW4odGltZXMpIC0gMSkpXSwKICAgICAgICAgICAgIm1lYW5fbXMiOiBzdGF0aXN0aWNzLmZtZWFuKHRpbWVzKX0KCgpkZWYgZmxvcHMoZm4pOgogICAgd2l0aCBGbG9wQ291bnRlck1vZGUoZGlzcGxheT1GYWxzZSkgYXMgY291bnRlcjoKICAgICAgICBmbigpCiAgICByZXR1cm4gaW50KGNvdW50ZXIuZ2V0X3RvdGFsX2Zsb3BzKCkpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXZhcmlhbnRzIiwgbmFyZ3M9IisiLCBkZWZhdWx0PURFRkFVTFRfVkFSSUFOVFMpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZXMiLCBuYXJncz0iKyIsIHR5cGU9aW50LCBkZWZhdWx0PVsxLCAzMl0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td2FybXVwIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taXRlcnMiLCB0eXBlPWludCwgZGVmYXVsdD01MCkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIpCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmsgPSBGYWxzZQogICAgYmFubmVyKCJFTkQtVE8tRU5EIEVGRklDSUVOQ1kgUFJPRklMRSIpCgogICAgbWFuaWZlc3QgPSByZWFkX2pzb24oREFUQV9ESVIgLyAibWFuaWZlc3QuanNvbiIpCiAgICB0YWJsZSA9IHBkLnJlYWRfY3N2KERBVEFfRElSIC8gImNhcHRpb25zX3Rlc3QuY3N2Iiwga2VlcF9kZWZhdWx0X25hPUZhbHNlKQogICAgbWF4X2JzID0gbWF4KGFyZ3MuYmF0Y2hfc2l6ZXMpCiAgICB0ZiA9IHZpdF90cmFuc2Zvcm0oKQogICAgaW1hZ2VzID0gW10KICAgIGZvciBpaWQgaW4gdGFibGVbImltYWdlX2lkIl0uaWxvY1s6OjVdLnRvbGlzdCgpWzptYXhfYnNdOgogICAgICAgIHdpdGggSW1hZ2Uub3BlbihmInttYW5pZmVzdFsnaW1hZ2VfZGlyJ119L3tpaWR9IikgYXMgaW06CiAgICAgICAgICAgIGltYWdlcy5hcHBlbmQodGYoaW0uY29udmVydCgiUkdCIikpKQogICAgaW1hZ2VzID0gdG9yY2guc3RhY2soaW1hZ2VzKS50byhkZXZpY2UpCiAgICBpZHMgPSB0b3JjaC5mcm9tX251bXB5KG5wLmxvYWQoRkVBVFVSRV9ESVIgLyAiaWRzX3Rlc3QubnB5IilbOm1heF9ic10pLmxvbmcoKS50byhkZXZpY2UpCiAgICBtYXNrID0gdG9yY2guZnJvbV9udW1weShucC5sb2FkKEZFQVRVUkVfRElSIC8gIm1hc2tfdGVzdC5ucHkiKVs6bWF4X2JzXSkubG9uZygpLnRvKGRldmljZSkKCiAgICB2aXQsIHJvYmVydGEgPSBGcm96ZW5WaVQoKS50byhkZXZpY2UpLCBGcm96ZW5Sb0JFUlRhKCkudG8oZGV2aWNlKQogICAgZnJvemVuID0geyJ2aXQiOiBzdW0ocC5udW1lbCgpIGZvciBwIGluIHZpdC5wYXJhbWV0ZXJzKCkpLCAicm9iZXJ0YSI6IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gcm9iZXJ0YS5wYXJhbWV0ZXJzKCkpfQoKICAgIHRyYWluX2ltZyA9IG5wLmxvYWQoRkVBVFVSRV9ESVIgLyAiaW1nX3RyYWluLm5weSIsIG1tYXBfbW9kZT0iciIpCiAgICB0cmFpbl90eHQgPSBucC5sb2FkKEZFQVRVUkVfRElSIC8gInR4dF90cmFpbi5ucHkiLCBtbWFwX21vZGU9InIiKQogICAgdHJhaW5fbWFzayA9IG5wLmxvYWQoRkVBVFVSRV9ESVIgLyAibWFza190cmFpbi5ucHkiLCBtbWFwX21vZGU9InIiKQogICAgdF9pbWcgPSB0b3JjaC5mcm9tX251bXB5KG5wLmFzYXJyYXkodHJhaW5faW1nWzo4XSwgZHR5cGU9bnAuZmxvYXQzMikpLnRvKGRldmljZSkKICAgIHRfdHh0ID0gdG9yY2guZnJvbV9udW1weShucC5hc2FycmF5KHRyYWluX3R4dFs6NDBdLCBkdHlwZT1ucC5mbG9hdDMyKSkudG8oZGV2aWNlKQogICAgdF9tYXNrID0gdG9yY2guZnJvbV9udW1weShucC5hc2FycmF5KHRyYWluX21hc2tbOjQwXSkuYXN0eXBlKGJvb2wpKS50byhkZXZpY2UpCiAgICBwYWlyID0gdG9yY2guYXJhbmdlKDgsIGRldmljZT1kZXZpY2UpLnJlcGVhdF9pbnRlcmxlYXZlKDUpCgogICAgcmVzdWx0cyA9IFtdCiAgICBmb3IgdmFyaWFudCBpbiBhcmdzLnZhcmlhbnRzOgogICAgICAgIG1vZGVsID0gUmV0cmlldmFsTW9kZWwoYnVpbGRfY29uZmlnKHZhcmlhbnQpKS50byhkZXZpY2UpCiAgICAgICAgY2twdCA9IFJVTlNfRElSIC8gZiJ7dmFyaWFudH1fX2tsMC4wMDAxX19jaHVuazE2X19zZWVkNDIiIC8gImJlc3RfbW9kZWwucHQiCiAgICAgICAgaWYgY2twdC5pc19maWxlKCk6CiAgICAgICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdCh0b3JjaC5sb2FkKGNrcHQsIG1hcF9sb2NhdGlvbj1kZXZpY2UsIHdlaWdodHNfb25seT1UcnVlKVsibW9kZWwiXSkKICAgICAgICBtb2RlbC5ldmFsKCkKICAgICAgICByb3cgPSB7InZhcmlhbnQiOiB2YXJpYW50LCAidHJhaW5lZF93ZWlnaHRzIjogY2twdC5pc19maWxlKCksICJ0cmFpbmFibGVfcGFyYW1zIjogY291bnRfdHJhaW5hYmxlKG1vZGVsKSwKICAgICAgICAgICAgICAgImZyb3plbl92aXRfcGFyYW1zIjogZnJvemVuWyJ2aXQiXSwgImZyb3plbl9yb2JlcnRhX3BhcmFtcyI6IGZyb3plblsicm9iZXJ0YSJdfQogICAgICAgIHJvd1sidG90YWxfcGFyYW1zIl0gPSByb3dbInRyYWluYWJsZV9wYXJhbXMiXSArIGZyb3plblsidml0Il0gKyBmcm96ZW5bInJvYmVydGEiXQoKICAgICAgICBmb3IgYnMgaW4gYXJncy5iYXRjaF9zaXplczoKICAgICAgICAgICAgaW0sIHRpZCwgdG0gPSBpbWFnZXNbOmJzXSwgaWRzWzpic10sIG1hc2tbOmJzXQogICAgICAgICAgICBmZWF0cyA9IHt9CgogICAgICAgICAgICBkZWYgcnVuX3ZpdCgpOgogICAgICAgICAgICAgICAgZmVhdHNbImltZyJdID0gdml0KGltKQoKICAgICAgICAgICAgZGVmIHJ1bl9yb2JlcnRhKCk6CiAgICAgICAgICAgICAgICBmZWF0c1sidHh0Il0gPSByb2JlcnRhKHRpZCwgdG0pCgogICAgICAgICAgICBydW5fdml0KCkKICAgICAgICAgICAgcnVuX3JvYmVydGEoKQoKICAgICAgICAgICAgQHRvcmNoLm5vX2dyYWQoKQogICAgICAgICAgICBkZWYgaGVhZChpbWdfZmVhdHMsIHR4dF9mZWF0cyk6CiAgICAgICAgICAgICAgICAjIGR1YWwgZW5jb2Rpbmc7IGV4Y2hhbmdlIG1vZGVscyBhZGQgb25lIHBhc3MtMiBzY29yZSBwZXIgaW1hZ2UtY2FwdGlvbiBwYWlyIChyZS1yYW5raW5nIGNvc3QpCiAgICAgICAgICAgICAgICBfLCBhaSwgXyA9IG1vZGVsLnBhc3MxKCJpbWciLCBpbWdfZmVhdHMsIE5vbmUsIHNhbXBsZT1GYWxzZSkKICAgICAgICAgICAgICAgIF8sIGF0LCBfID0gbW9kZWwucGFzczEoInR4dCIsIHR4dF9mZWF0cywgdG0uYm9vbCgpLCBzYW1wbGU9RmFsc2UpCiAgICAgICAgICAgICAgICBpZiBtb2RlbC5jZmcuZXhjaGFuZ2VzOgogICAgICAgICAgICAgICAgICAgIG1vZGVsLnBhaXJfc2ltaWxhcml0eShhaSwgYXQsIHRtLmJvb2woKSkKCiAgICAgICAgICAgIGRlZiBydW5faGVhZCgpOgogICAgICAgICAgICAgICAgaGVhZChmZWF0c1siaW1nIl0sIGZlYXRzWyJ0eHQiXSkKCiAgICAgICAgICAgIGRlZiBydW5fZTJlKCk6CiAgICAgICAgICAgICAgICBoZWFkKHZpdChpbSksIHJvYmVydGEodGlkLCB0bSkpCgogICAgICAgICAgICBmb3IgbmFtZSwgZm4gaW4gKCgidml0IiwgcnVuX3ZpdCksICgicm9iZXJ0YSIsIHJ1bl9yb2JlcnRhKSwgKCJoZWFkIiwgcnVuX2hlYWQpKToKICAgICAgICAgICAgICAgIHQgPSB0aW1lZChmbiwgYXJncy53YXJtdXAsIGFyZ3MuaXRlcnMpCiAgICAgICAgICAgICAgICByb3dbZiJic3tic31fe25hbWV9X21lZGlhbl9tcyJdID0gdFsibWVkaWFuX21zIl0KICAgICAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgICAgIGJhc2VfbWVtID0gdG9yY2guY3VkYS5tZW1vcnlfYWxsb2NhdGVkKCkKICAgICAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICAgICAgICAgIHQgPSB0aW1lZChydW5fZTJlLCBhcmdzLndhcm11cCwgYXJncy5pdGVycykKICAgICAgICAgICAgcm93W2YiYnN7YnN9X2UyZV9tZWRpYW5fbXMiXSA9IHRbIm1lZGlhbl9tcyJdCiAgICAgICAgICAgIHJvd1tmImJze2JzfV9lMmVfcDkwX21zIl0gPSB0WyJwOTBfbXMiXQogICAgICAgICAgICByb3dbZiJic3tic31fZTJlX3BhaXJzX3Blcl9zZWMiXSA9IGJzIC8gKHRbIm1lZGlhbl9tcyJdIC8gMTAwMCkKICAgICAgICAgICAgcm93W2YiYnN7YnN9X2UyZV9wZWFrX21lbV9hYm92ZV93ZWlnaHRzX21iIl0gPSAodG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpIC0gYmFzZV9tZW0pIC8gMioqMjAKICAgICAgICAgICAgaWYgYnMgPT0gMToKICAgICAgICAgICAgICAgIHJvd1siZmxvcHNfaGVhZF9iczEiXSA9IGZsb3BzKHJ1bl9oZWFkKQogICAgICAgICAgICAgICAgcm93WyJmbG9wc19lMmVfYnMxIl0gPSBmbG9wcyhydW5fZTJlKQoKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLCBscj0wLjApCgogICAgICAgIGRlZiB0cmFpbl9zdGVwKCk6CiAgICAgICAgICAgIG9wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICAgICAgb3V0ID0gbW9kZWwodF9pbWcsIHRfdHh0LCB0X21hc2ssIHBhaXIsIHNhbXBsZT1UcnVlKQogICAgICAgICAgICB0b3RhbF9sb3NzKG1vZGVsLCBvdXQsIHBhaXIsIDFlLTQpWzBdLmJhY2t3YXJkKCkKICAgICAgICAgICAgb3B0LnN0ZXAoKQoKICAgICAgICB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCkKICAgICAgICB0ID0gdGltZWQodHJhaW5fc3RlcCwgYXJncy53YXJtdXAsIGFyZ3MuaXRlcnMpCiAgICAgICAgcm93WyJ0cmFpbl9zdGVwX21lZGlhbl9tcyJdID0gdFsibWVkaWFuX21zIl0KICAgICAgICByb3dbInRyYWluX2NhcHRpb25zX3Blcl9zZWMiXSA9IDQwIC8gKHRbIm1lZGlhbl9tcyJdIC8gMTAwMCkKICAgICAgICByb3dbInRyYWluX3N0ZXBfcGVha19tZW1fbWIiXSA9IHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSAvIDIqKjIwCiAgICAgICAgcmVzdWx0cy5hcHBlbmQocm93KQogICAgICAgIHByaW50KHtrOiAocm91bmQodiwgMikgaWYgaXNpbnN0YW5jZSh2LCBmbG9hdCkgZWxzZSB2KSBmb3IgaywgdiBpbiByb3cuaXRlbXMoKX0sIGZsdXNoPVRydWUpCiAgICAgICAgZGVsIG1vZGVsLCBvcHQKICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKCiAgICBvdXQgPSBSRVBPUlRfRElSIC8gImVmZmljaWVuY3kiCiAgICBvdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGQuRGF0YUZyYW1lKHJlc3VsdHMpLnRvX2NzdihvdXQgLyAiZWZmaWNpZW5jeV9wcm9maWxlLmNzdiIsIGluZGV4PUZhbHNlKQogICAgd3JpdGVfanNvbihvdXQgLyAiZWZmaWNpZW5jeV9wcm9maWxlLmpzb24iLCB7CiAgICAgICAgInJvd3MiOiByZXN1bHRzLAogICAgICAgICJncHUiOiB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSwKICAgICAgICAicHJlY2lzaW9uIjogImZsb2F0MzIgZm9yIGJhY2tib25lcyBhbmQgaGVhZCIsCiAgICAgICAgImZsb3BzX25vdGUiOiAidG9yY2gudXRpbHMuZmxvcF9jb3VudGVyIGRvZXMgbm90IGNvdW50IGN1c3RvbSBUcml0b24vQ1VEQSBrZXJuZWxzIChNYW1iYS0yIFNTRCBzY2FuLCAiCiAgICAgICAgICAgICAgICAgICAgICAiY2F1c2FsIGNvbnYxZCk7IE1hbWJhLTIgRkxPUHMgYXJlIGEgbG93ZXIgYm91bmQuIiwKICAgICAgICAiaW5wdXRzIjogInJlYWwgRmxpY2tyOGsgdGVzdCBpbWFnZXMgKG9mZmljaWFsIFZpVCB0cmFuc2Zvcm0pIGFuZCB0b2tlbmlzZWQgdGVzdCBjYXB0aW9ucyIsCiAgICAgICAgImVudmlyb25tZW50IjogZW52aXJvbm1lbnRfbWFuaWZlc3QoKSwKICAgICAgICAicHJvdmVuYW5jZSI6IHByb3ZlbmFuY2UoKSwKICAgIH0pCgogICAgYnMgPSBtYXgoYXJncy5iYXRjaF9zaXplcykKICAgIGxpbmVzID0gWyJcXGJlZ2lue3RhYmxlfVt0XSIsICJcXGNlbnRlcmluZyIsCiAgICAgICAgICAgICBmIlxcY2FwdGlvbnt7RW5kLXRvLWVuZCBlZmZpY2llbmN5IG9uIHt0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKX0sIEZQMzIsIHJlYWwgRmxpY2tyOGsgaW5wdXRzLiAiCiAgICAgICAgICAgICBmIkxhdGVuY3k6IG1lZGlhbiBtcyBmb3IgYmF0Y2gge2JzfSBpbWFnZS0tY2FwdGlvbiBwYWlycyBpbmNsdWRpbmcgZnJvemVuIGJhY2tib25lcy4gIgogICAgICAgICAgICAgIkZMT1BzIGV4Y2x1ZGUgY3VzdG9tIE1hbWJhLTIga2VybmVscyAobG93ZXIgYm91bmQpLn0iLAogICAgICAgICAgICAgIlxcbGFiZWx7dGFiOmVmZmljaWVuY3l9IiwgIlxccmVzaXplYm94e1xcbGluZXdpZHRofXshfXslIiwgIlxcYmVnaW57dGFidWxhcn17bHJycnJycn0iLCAiXFxobGluZSIsCiAgICAgICAgICAgICAiTW9kZWwgJiBUcmFpbmFibGUgJiBUb3RhbCBwYXJhbXMgJiBIZWFkIG1zICYgRTJFIG1zICYgUGFpcnMvcyAmIFRyYWluIGNhcHQuL3MgXFxcXCIsICJcXGhsaW5lIl0KICAgIGZvciByIGluIHJlc3VsdHM6CiAgICAgICAgbmFtZSA9IHJbInZhcmlhbnQiXS5yZXBsYWNlKCJfIiwgIlxcXyIpCiAgICAgICAgbGluZXMuYXBwZW5kKGYie25hbWV9ICYge3JbJ3RyYWluYWJsZV9wYXJhbXMnXTosfSAmIHtyWyd0b3RhbF9wYXJhbXMnXSAvIDFlNjouMWZ9TSAmICIKICAgICAgICAgICAgICAgICAgICAgZiJ7cltmJ2Jze2JzfV9oZWFkX21lZGlhbl9tcyddOi4yZn0gJiB7cltmJ2Jze2JzfV9lMmVfbWVkaWFuX21zJ106LjFmfSAmICIKICAgICAgICAgICAgICAgICAgICAgZiJ7cltmJ2Jze2JzfV9lMmVfcGFpcnNfcGVyX3NlYyddOi4wZn0gJiB7clsndHJhaW5fY2FwdGlvbnNfcGVyX3NlYyddOi4wZn0gXFxcXCIpCiAgICBsaW5lcyArPSBbIlxcaGxpbmUiLCAiXFxlbmR7dGFidWxhcn19IiwgIlxcZW5ke3RhYmxlfSJdCiAgICAob3V0IC8gInRhYmxlX2VmZmljaWVuY3kudGV4Iikud3JpdGVfdGV4dCgiXG4iLmpvaW4obGluZXMpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHByaW50KCJFRkZJQ0lFTkNZX1BST0ZJTEU6IFBBU1MiKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"], "robustness.py": ["ba3d8f306f7268ed3811134e27d1206985449c93ff02999e23eab070fa45460f", "IiIiSDM6IHJvYnVzdG5lc3MgdG8gbm9pc3kgbXVsdGltb2RhbCBpbnB1dHMgb24gdGhlIEZsaWNrcjhrIHRlc3Qgc3BsaXQuCgogIGJ1aWxkICAgICByZWNvbXB1dGUgZnJvemVuIGZlYXR1cmVzIGZvciBjb3JydXB0ZWQgdGVzdCBpbnB1dHMgKGRldGVybWluaXN0aWMsIHNlZWQgMCk6CiAgICAgICAgICAgICAgaW1hZ2U6IGdhdXNzaWFuX25vaXNlX3swLjA4LDAuMTZ9IChwaXhlbCBzdGQgaW4gWzAsMV0pLCBibHVyX3sxLDN9IChHYXVzc2lhbiByYWRpdXMgcHgpLAogICAgICAgICAgICAgICAgICAgICBqcGVnX3szMCwxMH0gKHF1YWxpdHkpLCBvY2NsdXNpb25fezAuMjUsMC41fSAoZnJhY3Rpb24gb2YgMTZ4MTYgcGF0Y2hlcyBncmV5ZWQpCiAgICAgICAgICAgICAgdGV4dDogIHdvcmRfZHJvcG91dF97MC4xLDAuM30sIGNoYXJfdHlwb197MC4wNSwwLjE1fSAocGVyLWNoYXJhY3RlciBzd2FwL2RlbGV0ZS9pbnNlcnQpLAogICAgICAgICAgICAgICAgICAgICB3b3JkX3NodWZmbGVfezN9IChsb2NhbCBzaHVmZmxlcyB3aXRoaW4gd2luZG93cyBvZiAzIHdvcmRzKQogICAgICAgICAgICAtPiBGRUFUVVJFX0RJUi9yb2J1c3QvPG5hbWU+L3tpbWdfdGVzdC5ucHkgfCB0eHRfdGVzdC5ucHksIG1hc2tfdGVzdC5ucHl9CiAgZXZhbHVhdGUgIGV2ZXJ5IENPTVBMRVRFRCBydW4geCBldmVyeSBjb3JydXB0aW9uLCBzYW1lIG1ldHJpYyBjb2RlIGFzIHRoZSBjbGVhbiB0ZXN0CiAgICAgICAgICAgIC0+IHJlc3VsdHMvcm9idXN0bmVzcy5jc3Ygd2l0aCBtZWFuX3JlY2FsbCwgZHVhbF9tZWFuX3JlY2FsbCBhbmQgcmVsYXRpdmVfbXIgPSBNUl9jb3JydXB0IC8gTVJfY2xlYW4KClVzYWdlOiBweXRob24gcm9idXN0bmVzcy5weSBidWlsZCA7IHB5dGhvbiByb2J1c3RuZXNzLnB5IGV2YWx1YXRlIFstLXJ1bnMtZ2xvYiAiKiJdCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBpbwppbXBvcnQgb3MKaW1wb3J0IHJhbmRvbQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmZyb20gUElMIGltcG9ydCBJbWFnZSwgSW1hZ2VGaWx0ZXIKCmZyb20gYmFja2JvbmVzIGltcG9ydCBNQVhfVEVYVF9MRU4sIEZyb3plblJvQkVSVGEsIEZyb3plblZpVCwgbG9hZF90b2tlbml6ZXIsIHRva2VuaXplLCB2aXRfdHJhbnNmb3JtCmZyb20gY29tbW9uIGltcG9ydCBEQVRBX0RJUiwgRkVBVFVSRV9ESVIsIFJVTlNfRElSLCBiYW5uZXIsIHByb3ZlbmFuY2UsIHJlYWRfanNvbiwgc2hhMjU2X2ZpbGUsIHdyaXRlX2pzb24KZnJvbSBldmFsdWF0aW9uIGltcG9ydCBDYWNoZWRTcGxpdCwgZXZhbHVhdGUsIGxvYWRfcnVuX21vZGVsCgpST0JVU1RfRElSID0gRkVBVFVSRV9ESVIgLyAicm9idXN0IgpQQVRDSCA9IDE2CgoKZGVmIGdhdXNzaWFuX25vaXNlKHN0ZCk6CiAgICBkZWYgZihpbSwgcm5nKToKICAgICAgICBhID0gbnAuYXNhcnJheShpbSwgZHR5cGU9bnAuZmxvYXQzMikgLyAyNTUuMAogICAgICAgIGEgPSBucC5jbGlwKGEgKyBybmcubm9ybWFsKDAsIHN0ZCwgYS5zaGFwZSksIDAsIDEpCiAgICAgICAgcmV0dXJuIEltYWdlLmZyb21hcnJheSgoYSAqIDI1NSkucm91bmQoKS5hc3R5cGUobnAudWludDgpKQogICAgcmV0dXJuIGYKCgpkZWYgYmx1cihyYWRpdXMpOgogICAgcmV0dXJuIGxhbWJkYSBpbSwgcm5nOiBpbS5maWx0ZXIoSW1hZ2VGaWx0ZXIuR2F1c3NpYW5CbHVyKHJhZGl1cykpCgoKZGVmIGpwZWcocXVhbGl0eSk6CiAgICBkZWYgZihpbSwgcm5nKToKICAgICAgICBidWYgPSBpby5CeXRlc0lPKCkKICAgICAgICBpbS5zYXZlKGJ1ZiwgIkpQRUciLCBxdWFsaXR5PXF1YWxpdHkpCiAgICAgICAgYnVmLnNlZWsoMCkKICAgICAgICByZXR1cm4gSW1hZ2Uub3BlbihidWYpLmNvbnZlcnQoIlJHQiIpCiAgICByZXR1cm4gZgoKCmRlZiBvY2NsdXNpb24oZnJhYyk6CiAgICAiIiJBcHBsaWVkIGFmdGVyIHRoZSBvZmZpY2lhbCByZXNpemUvY3JvcCwgb24gdGhlIDIyNHgyMjQgZ3JpZCwgc28gb2NjbHVkZWQgYmxvY2tzIGFyZSBleGFjdCBWaVQgcGF0Y2hlcy4iIiIKICAgIGRlZiBmKGltLCBybmcpOgogICAgICAgIGEgPSBucC5hcnJheShpbSkKICAgICAgICBuID0gMTQgKiAxNAogICAgICAgIGZvciBrIGluIHJuZy5jaG9pY2UobiwgaW50KHJvdW5kKGZyYWMgKiBuKSksIHJlcGxhY2U9RmFsc2UpOgogICAgICAgICAgICByLCBjID0gZGl2bW9kKGludChrKSwgMTQpCiAgICAgICAgICAgIGFbciAqIFBBVENIOihyICsgMSkgKiBQQVRDSCwgYyAqIFBBVENIOihjICsgMSkgKiBQQVRDSF0gPSAxMjcKICAgICAgICByZXR1cm4gSW1hZ2UuZnJvbWFycmF5KGEpCiAgICByZXR1cm4gZgoKCmRlZiB3b3JkX2Ryb3BvdXQocCk6CiAgICBkZWYgZih0ZXh0LCBybmcpOgogICAgICAgIHdvcmRzID0gdGV4dC5zcGxpdCgpCiAgICAgICAga2VwdCA9IFt3IGZvciB3IGluIHdvcmRzIGlmIHJuZy5yYW5kb20oKSA+PSBwXQogICAgICAgIHJldHVybiAiICIuam9pbihrZXB0IG9yIHdvcmRzWzoxXSkKICAgIHJldHVybiBmCgoKZGVmIGNoYXJfdHlwbyhwKToKICAgIGxldHRlcnMgPSAiYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXoiCgogICAgZGVmIGYodGV4dCwgcm5nKToKICAgICAgICBvdXQsIGNoYXJzLCBpID0gW10sIGxpc3QodGV4dCksIDAKICAgICAgICB3aGlsZSBpIDwgbGVuKGNoYXJzKToKICAgICAgICAgICAgY2ggPSBjaGFyc1tpXQogICAgICAgICAgICBpZiBjaC5pc2FscGhhKCkgYW5kIHJuZy5yYW5kb20oKSA8IHA6CiAgICAgICAgICAgICAgICBvcCA9IHJuZy5yYW5kcmFuZ2UoMykKICAgICAgICAgICAgICAgIGlmIG9wID09IDAgYW5kIGkgKyAxIDwgbGVuKGNoYXJzKToKICAgICAgICAgICAgICAgICAgICBvdXQgKz0gW2NoYXJzW2kgKyAxXSwgY2hdCiAgICAgICAgICAgICAgICAgICAgaSArPSAyCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGlmIG9wID09IDE6CiAgICAgICAgICAgICAgICAgICAgaSArPSAxCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIG91dCArPSBbY2gsIHJuZy5jaG9pY2UobGV0dGVycyldCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKGNoKQogICAgICAgICAgICBpICs9IDEKICAgICAgICByZXR1cm4gIiIuam9pbihvdXQpCiAgICByZXR1cm4gZgoKCmRlZiB3b3JkX3NodWZmbGUod2luZG93KToKICAgIGRlZiBmKHRleHQsIHJuZyk6CiAgICAgICAgd29yZHMgPSB0ZXh0LnNwbGl0KCkKICAgICAgICBmb3IgcyBpbiByYW5nZSgwLCBsZW4od29yZHMpLCB3aW5kb3cpOgogICAgICAgICAgICBzZWcgPSB3b3Jkc1tzOnMgKyB3aW5kb3ddCiAgICAgICAgICAgIHJuZy5zaHVmZmxlKHNlZykKICAgICAgICAgICAgd29yZHNbczpzICsgd2luZG93XSA9IHNlZwogICAgICAgIHJldHVybiAiICIuam9pbih3b3JkcykKICAgIHJldHVybiBmCgoKSU1BR0VfQ09SUlVQVElPTlMgPSB7ImdhdXNzaWFuX25vaXNlXzAuMDgiOiBnYXVzc2lhbl9ub2lzZSgwLjA4KSwgImdhdXNzaWFuX25vaXNlXzAuMTYiOiBnYXVzc2lhbl9ub2lzZSgwLjE2KSwKICAgICAgICAgICAgICAgICAgICAgImJsdXJfMSI6IGJsdXIoMSksICJibHVyXzMiOiBibHVyKDMpLCAianBlZ18zMCI6IGpwZWcoMzApLCAianBlZ18xMCI6IGpwZWcoMTApLAogICAgICAgICAgICAgICAgICAgICAib2NjbHVzaW9uXzAuMjUiOiBvY2NsdXNpb24oMC4yNSksICJvY2NsdXNpb25fMC41Ijogb2NjbHVzaW9uKDAuNSl9ClRFWFRfQ09SUlVQVElPTlMgPSB7IndvcmRfZHJvcG91dF8wLjEiOiB3b3JkX2Ryb3BvdXQoMC4xKSwgIndvcmRfZHJvcG91dF8wLjMiOiB3b3JkX2Ryb3BvdXQoMC4zKSwKICAgICAgICAgICAgICAgICAgICAiY2hhcl90eXBvXzAuMDUiOiBjaGFyX3R5cG8oMC4wNSksICJjaGFyX3R5cG9fMC4xNSI6IGNoYXJfdHlwbygwLjE1KSwKICAgICAgICAgICAgICAgICAgICAid29yZF9zaHVmZmxlXzMiOiB3b3JkX3NodWZmbGUoMyl9CgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgYnVpbGQoZGV2aWNlKToKICAgIG1hbmlmZXN0ID0gcmVhZF9qc29uKERBVEFfRElSIC8gIm1hbmlmZXN0Lmpzb24iKQogICAgdGFibGUgPSBwZC5yZWFkX2NzdihEQVRBX0RJUiAvICJjYXB0aW9uc190ZXN0LmNzdiIsIGtlZXBfZGVmYXVsdF9uYT1GYWxzZSkKICAgIGltYWdlX2lkcyA9IHRhYmxlWyJpbWFnZV9pZCJdLmlsb2NbOjo1XS50b2xpc3QoKQogICAgdGYgPSB2aXRfdHJhbnNmb3JtKCkKICAgIHZpdCA9IEZyb3plblZpVCgpLnRvKGRldmljZSkKICAgIHJlY29yZCA9IHt9CiAgICBmb3IgbmFtZSwgZm4gaW4gSU1BR0VfQ09SUlVQVElPTlMuaXRlbXMoKToKICAgICAgICBvdXRfZGlyID0gUk9CVVNUX0RJUiAvIG5hbWUKICAgICAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMCkKICAgICAgICBmZWF0cyA9IFtdCiAgICAgICAgZm9yIHMgaW4gcmFuZ2UoMCwgbGVuKGltYWdlX2lkcyksIDY0KToKICAgICAgICAgICAgYmF0Y2ggPSBbXQogICAgICAgICAgICBmb3IgaWlkIGluIGltYWdlX2lkc1tzOnMgKyA2NF06CiAgICAgICAgICAgICAgICB3aXRoIEltYWdlLm9wZW4oZiJ7bWFuaWZlc3RbJ2ltYWdlX2RpciddfS97aWlkfSIpIGFzIGltOgogICAgICAgICAgICAgICAgICAgIGltID0gaW0uY29udmVydCgiUkdCIikKICAgICAgICAgICAgICAgIGlmIG5hbWUuc3RhcnRzd2l0aCgib2NjbHVzaW9uIik6CiAgICAgICAgICAgICAgICAgICAgIyBvZmZpY2lhbCByZXNpemUgKyBjZW50cmUgY3JvcCwgb2NjbHVkZSBvbiB0aGUgMjI0IGdyaWQsIHRoZW4gbm9ybWFsaXNlCiAgICAgICAgICAgICAgICAgICAgaW0gPSBJbWFnZS5mcm9tYXJyYXkoKF9jcm9wKGltLCB0ZikpKQogICAgICAgICAgICAgICAgICAgIGltID0gZm4oaW0sIHJuZykKICAgICAgICAgICAgICAgICAgICBiYXRjaC5hcHBlbmQoX25vcm1hbGlzZShpbSwgdGYpKQogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICBiYXRjaC5hcHBlbmQodGYoZm4oaW0sIHJuZykpKQogICAgICAgICAgICBmZWF0cy5hcHBlbmQodml0KHRvcmNoLnN0YWNrKGJhdGNoKS50byhkZXZpY2UpKS5mbG9hdCgpLmNwdSgpLm51bXB5KCkuYXN0eXBlKG5wLmZsb2F0MTYpKQogICAgICAgIG5wLnNhdmUob3V0X2RpciAvICJpbWdfdGVzdC5ucHkiLCBucC5jb25jYXRlbmF0ZShmZWF0cykpCiAgICAgICAgcmVjb3JkW25hbWVdID0geyJtb2RhbGl0eSI6ICJpbWFnZSIsICJzaGEyNTYiOiBzaGEyNTZfZmlsZShvdXRfZGlyIC8gImltZ190ZXN0Lm5weSIpfQogICAgICAgIHByaW50KCJidWlsdCIsIG5hbWUsIGZsdXNoPVRydWUpCiAgICBkZWwgdml0CgogICAgcm9iZXJ0YSwgdG9rZW5pemVyID0gRnJvemVuUm9CRVJUYSgpLnRvKGRldmljZSksIGxvYWRfdG9rZW5pemVyKCkKICAgIGZvciBuYW1lLCBmbiBpbiBURVhUX0NPUlJVUFRJT05TLml0ZW1zKCk6CiAgICAgICAgb3V0X2RpciA9IFJPQlVTVF9ESVIgLyBuYW1lCiAgICAgICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgcm5nID0gcmFuZG9tLlJhbmRvbSgwKQogICAgICAgIGNhcHRpb25zID0gW2ZuKGMsIHJuZykgZm9yIGMgaW4gdGFibGVbImNhcHRpb24iXS50b2xpc3QoKV0KICAgICAgICB0b2sgPSB0b2tlbml6ZSh0b2tlbml6ZXIsIGNhcHRpb25zKQogICAgICAgIGZlYXRzID0gW10KICAgICAgICBmb3IgcyBpbiByYW5nZSgwLCBsZW4oY2FwdGlvbnMpLCAyNTYpOgogICAgICAgICAgICBmZWF0cy5hcHBlbmQocm9iZXJ0YSh0b2tbImlucHV0X2lkcyJdW3M6cyArIDI1Nl0udG8oZGV2aWNlKSwgdG9rWyJhdHRlbnRpb25fbWFzayJdW3M6cyArIDI1Nl0udG8oZGV2aWNlKSkKICAgICAgICAgICAgICAgICAgICAgICAgIC5mbG9hdCgpLmNwdSgpLm51bXB5KCkuYXN0eXBlKG5wLmZsb2F0MTYpKQogICAgICAgIG5wLnNhdmUob3V0X2RpciAvICJ0eHRfdGVzdC5ucHkiLCBucC5jb25jYXRlbmF0ZShmZWF0cykpCiAgICAgICAgbnAuc2F2ZShvdXRfZGlyIC8gIm1hc2tfdGVzdC5ucHkiLCB0b2tbImF0dGVudGlvbl9tYXNrIl0ubnVtcHkoKS5hc3R5cGUobnAudWludDgpKQogICAgICAgIHBkLkRhdGFGcmFtZSh7ImNsZWFuIjogdGFibGVbImNhcHRpb24iXSwgImNvcnJ1cHRlZCI6IGNhcHRpb25zfSkuaGVhZCg1MCkudG9fY3N2KG91dF9kaXIgLyAiZXhhbXBsZXMuY3N2IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpbmRleD1GYWxzZSkKICAgICAgICByZWNvcmRbbmFtZV0gPSB7Im1vZGFsaXR5IjogInRleHQiLCAic2hhMjU2Ijogc2hhMjU2X2ZpbGUob3V0X2RpciAvICJ0eHRfdGVzdC5ucHkiKSwKICAgICAgICAgICAgICAgICAgICAgICAgIm1heF90ZXh0X2xlbiI6IE1BWF9URVhUX0xFTn0KICAgICAgICBwcmludCgiYnVpbHQiLCBuYW1lLCBmbHVzaD1UcnVlKQogICAgd3JpdGVfanNvbihST0JVU1RfRElSIC8gInJvYnVzdF9tYW5pZmVzdC5qc29uIiwgeyJjb3JydXB0aW9ucyI6IHJlY29yZCwgInByb3ZlbmFuY2UiOiBwcm92ZW5hbmNlKCl9KQogICAgcHJpbnQoIlJPQlVTVE5FU1NfQlVJTEQ6IFBBU1MiKQoKCmRlZiBfY3JvcChpbSwgdGYpOgogICAgIiIiUmVzaXplICsgY2VudHJlIGNyb3Agb2YgdGhlIG9mZmljaWFsIHRyYW5zZm9ybSwgcmV0dXJuZWQgYXMgdWludDggSHhXeDMuIiIiCiAgICBmcm9tIHRvcmNodmlzaW9uLnRyYW5zZm9ybXMgaW1wb3J0IGZ1bmN0aW9uYWwgYXMgVEYKICAgIGltID0gVEYucmVzaXplKGltLCB0Zi5yZXNpemVfc2l6ZSwgaW50ZXJwb2xhdGlvbj10Zi5pbnRlcnBvbGF0aW9uLCBhbnRpYWxpYXM9dGYuYW50aWFsaWFzKQogICAgcmV0dXJuIG5wLmFycmF5KFRGLmNlbnRlcl9jcm9wKGltLCB0Zi5jcm9wX3NpemUpKQoKCmRlZiBfbm9ybWFsaXNlKGltLCB0Zik6CiAgICBmcm9tIHRvcmNodmlzaW9uLnRyYW5zZm9ybXMgaW1wb3J0IGZ1bmN0aW9uYWwgYXMgVEYKICAgIHJldHVybiBURi5ub3JtYWxpemUoVEYudG9fdGVuc29yKGltKSwgbWVhbj10Zi5tZWFuLCBzdGQ9dGYuc3RkKQoKCmRlZiBldmFsdWF0ZV9hbGwoZGV2aWNlLCBydW5zX2dsb2IpOgogICAgc3BsaXRzID0geyJjbGVhbiI6IENhY2hlZFNwbGl0KCJ0ZXN0Iil9CiAgICBmb3IgbmFtZSBpbiBJTUFHRV9DT1JSVVBUSU9OUzoKICAgICAgICBzcGxpdHNbbmFtZV0gPSBDYWNoZWRTcGxpdCgidGVzdCIsIGltZ19wYXRoPVJPQlVTVF9ESVIgLyBuYW1lIC8gImltZ190ZXN0Lm5weSIpCiAgICBmb3IgbmFtZSBpbiBURVhUX0NPUlJVUFRJT05TOgogICAgICAgIHNwbGl0c1tuYW1lXSA9IENhY2hlZFNwbGl0KCJ0ZXN0IiwgdHh0X3BhdGg9Uk9CVVNUX0RJUiAvIG5hbWUgLyAidHh0X3Rlc3QubnB5IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXNrX3BhdGg9Uk9CVVNUX0RJUiAvIG5hbWUgLyAibWFza190ZXN0Lm5weSIpCiAgICByb3dzID0gW10KICAgIGZvciBzdW1tYXJ5X3BhdGggaW4gc29ydGVkKFJVTlNfRElSLmdsb2IoZiJ7cnVuc19nbG9ifS9ydW5fc3VtbWFyeS5qc29uIikpOgogICAgICAgIHMgPSByZWFkX2pzb24oc3VtbWFyeV9wYXRoKQogICAgICAgIGlmIHMuZ2V0KCJzdGF0dXMiKSAhPSAiQ09NUExFVEVEIjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBydW5fZGlyID0gc3VtbWFyeV9wYXRoLnBhcmVudAogICAgICAgIGNmZyA9IHJlYWRfanNvbihydW5fZGlyIC8gImNvbmZpZy5qc29uIikKICAgICAgICBtb2RlbCA9IGxvYWRfcnVuX21vZGVsKHJ1bl9kaXIsIGRldmljZSkKICAgICAgICByZXJhbmtfayA9IGNmZy5nZXQoInJlcmFua19rIiwgMTYpIGlmIG1vZGVsLmNmZy5leGNoYW5nZXMgZWxzZSAwCiAgICAgICAgY2xlYW4gPSBOb25lCiAgICAgICAgZm9yIG5hbWUsIGRhdGEgaW4gc3BsaXRzLml0ZW1zKCk6CiAgICAgICAgICAgIG0gPSBldmFsdWF0ZShtb2RlbCwgZGF0YSwgZGV2aWNlLCByZXJhbmtfaylbMF0KICAgICAgICAgICAgY2xlYW4gPSBjbGVhbiBvciBtCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsicnVuIjogcnVuX2Rpci5uYW1lLCAidmFyaWFudCI6IHNbInZhcmlhbnQiXSwgInNlZWQiOiBzWyJzZWVkIl0sICJrbF93ZWlnaHQiOiBzWyJrbF93ZWlnaHQiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICJodnNjX2NodW5rX3NpemUiOiBzWyJodnNjX2NodW5rX3NpemUiXSwgImNvcnJ1cHRpb24iOiBuYW1lLAogICAgICAgICAgICAgICAgICAgICAgICAgIm1vZGFsaXR5IjogIm5vbmUiIGlmIG5hbWUgPT0gImNsZWFuIiBlbHNlICgiaW1hZ2UiIGlmIG5hbWUgaW4gSU1BR0VfQ09SUlVQVElPTlMgZWxzZSAidGV4dCIpLAogICAgICAgICAgICAgICAgICAgICAgICAgIm1lYW5fcmVjYWxsIjogbVsibWVhbl9yZWNhbGwiXSwgImR1YWxfbWVhbl9yZWNhbGwiOiBtWyJkdWFsX21lYW5fcmVjYWxsIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAicmVsYXRpdmVfbXIiOiBtWyJtZWFuX3JlY2FsbCJdIC8gbWF4KGNsZWFuWyJtZWFuX3JlY2FsbCJdLCAxZS05KX0pCiAgICAgICAgcHJpbnQocnVuX2Rpci5uYW1lLCAiZG9uZSIsIGZsdXNoPVRydWUpCiAgICBvdXQgPSBSVU5TX0RJUi5wYXJlbnQgLyAicmVzdWx0cyIgLyAicm9idXN0bmVzcy5jc3YiCiAgICBvdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBkLkRhdGFGcmFtZShyb3dzKS50b19jc3Yob3V0LCBpbmRleD1GYWxzZSkKICAgIHByaW50KGYie2xlbihyb3dzKX0gcm93cyAtPiB7b3V0fSIpCiAgICBwcmludCgiUk9CVVNUTkVTU19FVkFMOiBQQVNTIikKCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoImFjdGlvbiIsIGNob2ljZXM9WyJidWlsZCIsICJldmFsdWF0ZSJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJ1bnMtZ2xvYiIsIGRlZmF1bHQ9IioiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKG9zLmVudmlyb24uZ2V0KCJIRURPX0RFVklDRSIsICJjdWRhIikpCiAgICBiYW5uZXIoZiJST0JVU1RORVNTIHthcmdzLmFjdGlvbi51cHBlcigpfSIpCiAgICBpZiBhcmdzLmFjdGlvbiA9PSAiYnVpbGQiOgogICAgICAgIGJ1aWxkKGRldmljZSkKICAgIGVsc2U6CiAgICAgICAgZXZhbHVhdGVfYWxsKGRldmljZSwgYXJncy5ydW5zX2dsb2IpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="], "run_matrix.py": ["345d2f34836948a7ccdde1d786e738d798d1accfc7a732d048ed7b860a48a8fd", "IiIiUnVuIGV4cGVyaW1lbnQgc3VpdGVzIGFuZCByZWJ1aWxkIHJlc3VsdHMvYWxsX3J1bnMuY3N2IGZyb20gcnVuX3N1bW1hcnkuanNvbiBmaWxlcy4KCkEgcnVuIGlzIHNraXBwZWQgb25seSBpZiBpdHMgcnVuX3N1bW1hcnkuanNvbiBzYXlzIENPTVBMRVRFRCAqYW5kKiBpdHMKY29kZV9zaGEyNTYsIGNvbmZpZ19zaGEyNTYgYW5kIGZlYXR1cmVfbWFuaWZlc3Rfc2hhMjU2IG1hdGNoIHdoYXQgdGhpcyBjb2RlCndvdWxkIHByb2R1Y2Ugbm93LiBBbnl0aGluZyBlbHNlIGlzIHJlLXJ1biBmcm9tIHNjcmF0Y2ggKC0tb3ZlcndyaXRlKS4KVGhlIENTViBpcyByZWdlbmVyYXRlZCBmcm9tIGRpc2sgZXZlcnkgdGltZTsgaXQgaXMgbmV2ZXIgYXBwZW5kZWQgdG8uCgpTdWl0ZXM6CiAgcHJvcG9zYWwgICAgICAgICAgICBiYXNlbGluZSwgaGVkb19lbmVyZ3ksIGh2c2NfeCwgZnVsbF94ICAocHJvcG9zZWQgMngyKQogIHByb3Bvc2FsX2FibGF0aW9ucyAgZnVsbF94X2FmZmluZSwgZnVsbF94X2NvbnN0ZGFtcCwgZnVsbF94X25vZXhjaGFuZ2UsIGZ1bGxfeF9ub3ByaW9yLAogICAgICAgICAgICAgICAgICAgICAgYmFzZWxpbmVfcGFyYW1fbWF0Y2hlZF94LCB0cmFuc2Zvcm1lcl9wYXJhbV9tYXRjaGVkLCB0cmFuc2Zvcm1lcl94LCBub19taXhlcl9tZWFucG9vbAogIHByb3Bvc2FsX2NodW5rcyAgICAgZnVsbF94IHdpdGggaHZzY19jaHVua19zaXplIGluIHs4LCAzMn0KICBjb3JlICAgICAgIGJhc2VsaW5lLCBoZWRvLCBodnNjLCBmdWxsICAgKGxlZ2FjeSkKICBhYmxhdGlvbnMgIGZ1bGxfbGluZWFyX29wZXJhdG9yLCBmdWxsX2h2c2NfZGV0ZXJtaW5pc3RpYywgYmFzZWxpbmVfcGFyYW1fbWF0Y2hlZCwKICAgICAgICAgICAgIG5vX21peGVyX21lYW5wb29sLCB0cmFuc2Zvcm1lcl9wYXJhbV9tYXRjaGVkCiAga2xzd2VlcCAgICBmdWxsIHdpdGgga2xfd2VpZ2h0IGluIHswLCAxZS0zLCAxZS0yfSAgICgxZS00IGlzIHRoZSBjb3JlIHJ1bikKICBjaHVua3MgICAgIGZ1bGwgd2l0aCBodnNjX2NodW5rX3NpemUgaW4gezgsIDMyfSAgICAgKDE2IGlzIHRoZSBjb3JlIHJ1bikKClVzYWdlOiBweXRob24gcnVuX21hdHJpeC5weSAtLXN1aXRlIHByb3Bvc2FsIC0tc2VlZHMgNDIgNDMgNDQgNDUgNDYgWy0tZXBvY2hzIDEwXQoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBjb21tb24gaW1wb3J0IEZFQVRVUkVfRElSLCBQQUNLQUdFX0RJUiwgUlVOU19ESVIsIGJhbm5lciwgY29kZV9zaGEyNTYsIHJlYWRfanNvbiwgc2hhMjU2X2ZpbGUKZnJvbSBtb2RlbHMgaW1wb3J0IFZBUklBTlRTCgpERUZBVUxUX0tMID0gMWUtNApERUZBVUxUX0NIVU5LID0gMTYKClNVSVRFUyA9IHsKICAgICMgcHJvcG9zZWQgbW9kZWw6IDJ4MiBmYWN0b3JpYWwgb2YgRW5lcmd5SEVETyB4IEV4Y2hhbmdlSFZTQwogICAgInByb3Bvc2FsIjogWyh2LCBERUZBVUxUX0tMLCBERUZBVUxUX0NIVU5LKSBmb3IgdiBpbiAoImJhc2VsaW5lIiwgImhlZG9fZW5lcmd5IiwgImh2c2NfeCIsICJmdWxsX3giKV0sCiAgICAicHJvcG9zYWxfYWJsYXRpb25zIjogWyh2LCBERUZBVUxUX0tMLCBERUZBVUxUX0NIVU5LKSBmb3IgdiBpbiAoCiAgICAgICAgImZ1bGxfeF9hZmZpbmUiLCAiZnVsbF94X2NvbnN0ZGFtcCIsICJmdWxsX3hfbm9leGNoYW5nZSIsICJmdWxsX3hfbm9wcmlvciIsICJiYXNlbGluZV9wYXJhbV9tYXRjaGVkX3giLAogICAgICAgICJ0cmFuc2Zvcm1lcl9wYXJhbV9tYXRjaGVkIiwgInRyYW5zZm9ybWVyX3giLCAibm9fbWl4ZXJfbWVhbnBvb2wiKV0sCiAgICAicHJvcG9zYWxfY2h1bmtzIjogWygiZnVsbF94IiwgREVGQVVMVF9LTCwgYykgZm9yIGMgaW4gKDgsIDMyKV0sCiAgICAjIGxlZ2FjeSAoYWZmaW5lIEhFRE8gLyBpbmRleC1LTCBIVlNDKQogICAgImNvcmUiOiBbKHYsIERFRkFVTFRfS0wsIERFRkFVTFRfQ0hVTkspIGZvciB2IGluICgiYmFzZWxpbmUiLCAiaGVkbyIsICJodnNjIiwgImZ1bGwiKV0sCiAgICAiYWJsYXRpb25zIjogWyh2LCBERUZBVUxUX0tMLCBERUZBVUxUX0NIVU5LKSBmb3IgdiBpbiAoCiAgICAgICAgImZ1bGxfbGluZWFyX29wZXJhdG9yIiwgImZ1bGxfaHZzY19kZXRlcm1pbmlzdGljIiwgImJhc2VsaW5lX3BhcmFtX21hdGNoZWQiLAogICAgICAgICJub19taXhlcl9tZWFucG9vbCIsICJ0cmFuc2Zvcm1lcl9wYXJhbV9tYXRjaGVkIildLAogICAgImtsc3dlZXAiOiBbKCJmdWxsIiwga2wsIERFRkFVTFRfQ0hVTkspIGZvciBrbCBpbiAoMC4wLCAxZS0zLCAxZS0yKV0sCiAgICAiY2h1bmtzIjogWygiZnVsbCIsIERFRkFVTFRfS0wsIGMpIGZvciBjIGluICg4LCAzMildLAp9CgoKZGVmIHJ1bl9uYW1lKHZhcmlhbnQsIGtsLCBjaHVuaywgc2VlZCk6CiAgICByZXR1cm4gZiJ7dmFyaWFudH1fX2tse2tsOmd9X19jaHVua3tjaHVua31fX3NlZWR7c2VlZH0iCgoKZGVmIGlzX2N1cnJlbnQocnVuX2RpciwgZXBvY2hzKToKICAgIHN1bW1hcnlfcGF0aCA9IHJ1bl9kaXIgLyAicnVuX3N1bW1hcnkuanNvbiIKICAgIGlmIG5vdCBzdW1tYXJ5X3BhdGguaXNfZmlsZSgpOgogICAgICAgIHJldHVybiBGYWxzZSwgIm5vIHN1bW1hcnkiCiAgICBzID0gcmVhZF9qc29uKHN1bW1hcnlfcGF0aCkKICAgIGlmIHMuZ2V0KCJzdGF0dXMiKSAhPSAiQ09NUExFVEVEIjoKICAgICAgICByZXR1cm4gRmFsc2UsIHMuZ2V0KCJzdGF0dXMiKQogICAgaWYgcy5nZXQoImNvZGVfc2hhMjU2IikgIT0gY29kZV9zaGEyNTYoKToKICAgICAgICByZXR1cm4gRmFsc2UsICJjb2RlIGNoYW5nZWQiCiAgICBpZiBzLmdldCgiZmVhdHVyZV9tYW5pZmVzdF9zaGEyNTYiKSAhPSBzaGEyNTZfZmlsZShGRUFUVVJFX0RJUiAvICJmZWF0dXJlX21hbmlmZXN0Lmpzb24iKToKICAgICAgICByZXR1cm4gRmFsc2UsICJmZWF0dXJlcyBjaGFuZ2VkIgogICAgY2ZnID0gcmVhZF9qc29uKHJ1bl9kaXIgLyAiY29uZmlnLmpzb24iKQogICAgaWYgY2ZnLmdldCgiZXBvY2hzIikgIT0gZXBvY2hzOgogICAgICAgIHJldHVybiBGYWxzZSwgImVwb2NoIGNvdW50IGRpZmZlcnMiCiAgICByZXN1bHRzID0gcmVhZF9qc29uKHJ1bl9kaXIgLyAidGVzdF9yZXN1bHRzLmpzb24iKQogICAgaWYgcmVzdWx0c1sicHJvdmVuYW5jZSJdWyJjb25maWdfc2hhMjU2Il0gIT0gcy5nZXQoImNvbmZpZ19zaGEyNTYiKToKICAgICAgICByZXR1cm4gRmFsc2UsICJjb25maWcgaGFzaCBtaXNtYXRjaCIKICAgIGlmIHNoYTI1Nl9maWxlKHJ1bl9kaXIgLyAidGVzdF9yZXN1bHRzLmpzb24iKSAhPSBzLmdldCgidGVzdF9yZXN1bHRzX3NoYTI1NiIpOgogICAgICAgIHJldHVybiBGYWxzZSwgInRlc3RfcmVzdWx0cy5qc29uIG1vZGlmaWVkIGFmdGVyIHJ1biIKICAgIHJldHVybiBUcnVlLCAiY3VycmVudCIKCgpkZWYgY29sbGVjdChyb290KToKICAgIHJvd3MgPSBbXQogICAgZm9yIHN1bW1hcnlfcGF0aCBpbiBzb3J0ZWQocm9vdC5nbG9iKCIqL3J1bl9zdW1tYXJ5Lmpzb24iKSk6CiAgICAgICAgcnVuX2RpciA9IHN1bW1hcnlfcGF0aC5wYXJlbnQKICAgICAgICBzID0gcmVhZF9qc29uKHN1bW1hcnlfcGF0aCkKICAgICAgICByb3cgPSB7InJ1biI6IHJ1bl9kaXIubmFtZSwgInZhcmlhbnQiOiBzLmdldCgidmFyaWFudCIpLCAic2VlZCI6IHMuZ2V0KCJzZWVkIiksCiAgICAgICAgICAgICAgICJrbF93ZWlnaHQiOiBzLmdldCgia2xfd2VpZ2h0IiksICJodnNjX2NodW5rX3NpemUiOiBzLmdldCgiaHZzY19jaHVua19zaXplIiksCiAgICAgICAgICAgICAgICJzdGF0dXMiOiBzLmdldCgic3RhdHVzIiksICJ0cmFpbmFibGVfcGFyYW1zIjogcy5nZXQoInRyYWluYWJsZV9wYXJhbXMiKX0KICAgICAgICByZXN1bHRzX3BhdGggPSBydW5fZGlyIC8gInRlc3RfcmVzdWx0cy5qc29uIgogICAgICAgIGlmIHMuZ2V0KCJzdGF0dXMiKSA9PSAiQ09NUExFVEVEIiBhbmQgcmVzdWx0c19wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgciA9IHJlYWRfanNvbihyZXN1bHRzX3BhdGgpCiAgICAgICAgICAgIGZvciBrIGluICgiaTJ0X3IxIiwgImkydF9yNSIsICJpMnRfcjEwIiwgImkydF9tZWRyIiwgInQyaV9yMSIsICJ0MmlfcjUiLCAidDJpX3IxMCIsICJ0MmlfbWVkciIsCiAgICAgICAgICAgICAgICAgICAgICAibWVhbl9yZWNhbGwiLCAiZHVhbF9tZWFuX3JlY2FsbCIsICJyZXJhbmtlZCIsICJiZXN0X2Vwb2NoIiwgImJlc3RfdmFsX21lYW5fcmVjYWxsIiwKICAgICAgICAgICAgICAgICAgICAgICJ0cmFpbl9zZWNvbmRzIiwgInBlYWtfdHJhaW5fbWVtb3J5X21iIik6CiAgICAgICAgICAgICAgICByb3dba10gPSByLmdldChrKQogICAgICAgICAgICByb3dbImNvZGVfc2hhMjU2Il0gPSByWyJwcm92ZW5hbmNlIl1bImNvZGVfc2hhMjU2Il0KICAgICAgICAgICAgcm93WyJjb25maWdfc2hhMjU2Il0gPSByWyJwcm92ZW5hbmNlIl1bImNvbmZpZ19zaGEyNTYiXQogICAgICAgICAgICByb3dbImN1cnJlbnRfY29kZSJdID0gcm93WyJjb2RlX3NoYTI1NiJdID09IGNvZGVfc2hhMjU2KCkKICAgICAgICByb3dzLmFwcGVuZChyb3cpCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXN1aXRlIiwgbmFyZ3M9IisiLCByZXF1aXJlZD1UcnVlLCBjaG9pY2VzPXNvcnRlZChTVUlURVMpKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlZWRzIiwgbmFyZ3M9IisiLCB0eXBlPWludCwgZGVmYXVsdD1bNDIsIDQzLCA0NCwgNDUsIDQ2XSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD0xMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zdG9wLW9uLWZhaWx1cmUiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQogICAgUlVOU19ESVIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIHBsYW4gPSBbKHYsIGtsLCBjLCBzKSBmb3Igc3VpdGUgaW4gYXJncy5zdWl0ZSBmb3IgKHYsIGtsLCBjKSBpbiBTVUlURVNbc3VpdGVdIGZvciBzIGluIGFyZ3Muc2VlZHNdCiAgICBhc3NlcnQgYWxsKHYgaW4gVkFSSUFOVFMgZm9yIHYsIF8sIF8sIF8gaW4gcGxhbikKICAgIGJhbm5lcihmIlJVTiBNQVRSSVg6IHtsZW4ocGxhbil9IHJ1bnMsIGNvZGUge2NvZGVfc2hhMjU2KClbOjEyXX0iKQoKICAgIGZhaWx1cmVzID0gW10KICAgIGZvciBpLCAodmFyaWFudCwga2wsIGNodW5rLCBzZWVkKSBpbiBlbnVtZXJhdGUocGxhbiwgMSk6CiAgICAgICAgcnVuX2RpciA9IFJVTlNfRElSIC8gcnVuX25hbWUodmFyaWFudCwga2wsIGNodW5rLCBzZWVkKQogICAgICAgIGN1cnJlbnQsIHJlYXNvbiA9IGlzX2N1cnJlbnQocnVuX2RpciwgYXJncy5lcG9jaHMpCiAgICAgICAgaWYgY3VycmVudDoKICAgICAgICAgICAgcHJpbnQoZiJbe2l9L3tsZW4ocGxhbil9XSBza2lwIHtydW5fZGlyLm5hbWV9ICh2ZXJpZmllZCBjdXJyZW50KSIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcHJpbnQoZiJbe2l9L3tsZW4ocGxhbil9XSBydW4gIHtydW5fZGlyLm5hbWV9ICh7cmVhc29ufSkiLCBmbHVzaD1UcnVlKQogICAgICAgIGNtZCA9IFtzeXMuZXhlY3V0YWJsZSwgIi11Iiwgc3RyKFBBQ0tBR0VfRElSIC8gInRyYWluLnB5IiksICItLXZhcmlhbnQiLCB2YXJpYW50LCAiLS1zZWVkIiwgc3RyKHNlZWQpLAogICAgICAgICAgICAgICAiLS1lcG9jaHMiLCBzdHIoYXJncy5lcG9jaHMpLCAiLS1rbC13ZWlnaHQiLCBzdHIoa2wpLCAiLS1odnNjLWNodW5rLXNpemUiLCBzdHIoY2h1bmspLAogICAgICAgICAgICAgICAiLS1vdXQiLCBzdHIocnVuX2RpciksICItLW92ZXJ3cml0ZSJdCiAgICAgICAgcmMgPSBzdWJwcm9jZXNzLnJ1bihjbWQpLnJldHVybmNvZGUKICAgICAgICBpZiByYyAhPSAwOgogICAgICAgICAgICBmYWlsdXJlcy5hcHBlbmQoKHJ1bl9kaXIubmFtZSwgcmMpKQogICAgICAgICAgICBwcmludChmIkZBSUxFRCB7cnVuX2Rpci5uYW1lfSBleGl0PXtyY30iLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBpZiBhcmdzLnN0b3Bfb25fZmFpbHVyZToKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgZGYgPSBjb2xsZWN0KFJVTlNfRElSKQogICAgb3V0X2NzdiA9IFJVTlNfRElSLnBhcmVudCAvICJyZXN1bHRzIiAvICJhbGxfcnVucy5jc3YiCiAgICBvdXRfY3N2LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBkZi50b19jc3Yob3V0X2NzdiwgaW5kZXg9RmFsc2UpCiAgICBiYW5uZXIoZiJ7bGVuKGRmKX0gcnVuIGRpcmVjdG9yaWVzLCB7aW50KChkZlsnc3RhdHVzJ10gPT0gJ0NPTVBMRVRFRCcpLnN1bSgpKSBpZiBsZW4oZGYpIGVsc2UgMH0gY29tcGxldGVkICIKICAgICAgICAgICBmIi0+IHtvdXRfY3N2fSIpCiAgICBpZiBmYWlsdXJlczoKICAgICAgICBwcmludCgiZmFpbHVyZXM6IiwgZmFpbHVyZXMpCiAgICAgICAgc3lzLmV4aXQoMSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="], "scaling.py": ["68738007231c6a448ec00e2cd932dd995750de446283d584589fabb783f4c810", "IiIiSDQ6IGNvbXB1dGF0aW9uYWwgb3ZlcmhlYWQgYW5kIHNjYWxpbmcgd2l0aCBpbWFnZSBzZXF1ZW5jZSBsZW5ndGguCgpGb3IgZWFjaCB2YXJpYW50LCB0aGUgdHJhaW5hYmxlIGhlYWQgKGZyb3plbiBiYWNrYm9uZXMgZXhjbHVkZWQ6IHRoZXkgYXJlIGlkZW50aWNhbCBhY3Jvc3MKdmFyaWFudHMgYW5kIGZpeGVkIGF0IDE5NiBwYXRjaGVzKSBpcyB0aW1lZCBhdCBpbWFnZSBsZW5ndGhzIEwgaW4gezE5NiwgNzg0LCAzMTM2LCAxMjU0NH0KKHBhdGNoIGdyaWRzIDE0XjIgLi4gMTEyXjIpIHdpdGggYSA2NC10b2tlbiBjYXB0aW9uOgogICogaW5mZXJlbmNlOiBpbWFnZSBlbmNvZGluZyAocGFzcyAxKSwgcGx1cyBvbmUgZXhjaGFuZ2UgcGFpciAocGFzcyAyKSBmb3IgZXhjaGFuZ2UgbW9kZWxzCiAgKiB0cmFpbmluZzogZm9yd2FyZCArIGJhY2t3YXJkIG9mIG9uZSA4LWltYWdlIHggNDAtY2FwdGlvbiBzdGVwCiAgKiBwZWFrIG1lbW9yeSBmb3IgYm90aApJbnB1dHMgYXJlIHJhbmRvbSB0ZW5zb3JzIHdpdGggdGhlIHNoYXBlIG9mIFZpVCB0b2tlbnMuIFRoaXMgaXMgZGVsaWJlcmF0ZTogdGhlIGV4cGVyaW1lbnQgbWVhc3VyZXMKY29tcHV0ZSBzY2FsaW5nIGF0IHJlc29sdXRpb25zIHRoZSBjYWNoZWQgVmlUIGZlYXR1cmVzIGRvIG5vdCBleGlzdCBmb3IsIGFuZCBpcyBsYWJlbGxlZCBhcyBzdWNoLgpUaGUgbG9nLWxvZyBzbG9wZSBvZiBtZWRpYW4gbGF0ZW5jeSB2cyBMIGlzIGZpdHRlZCBvbiBMID49IDc4NCAoc2xvcGUgfjEgPSBsaW5lYXIsIH4yID0gcXVhZHJhdGljKS4KT09NIGlzIHJlY29yZGVkLCBub3QgaGlkZGVuLgoKVXNhZ2U6IHB5dGhvbiBzY2FsaW5nLnB5IFstLXZhcmlhbnRzIGJhc2VsaW5lIGZ1bGxfeCB0cmFuc2Zvcm1lcl9wYXJhbV9tYXRjaGVkIC4uLl0KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IG9zCmltcG9ydCBzdGF0aXN0aWNzCmltcG9ydCB0aW1lCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKCmZyb20gY29tbW9uIGltcG9ydCBSRVBPUlRfRElSLCBiYW5uZXIsIHByb3ZlbmFuY2UsIHdyaXRlX2pzb24KZnJvbSBtb2RlbHMgaW1wb3J0IFJldHJpZXZhbE1vZGVsLCBidWlsZF9jb25maWcsIGNvdW50X3RyYWluYWJsZSwgdG90YWxfbG9zcwoKTEVOR1RIUyA9IFsxOTYsIDc4NCwgMzEzNiwgMTI1NDRdCkRFRkFVTFRfVkFSSUFOVFMgPSBbImJhc2VsaW5lIiwgImhlZG9fZW5lcmd5IiwgImh2c2NfeCIsICJmdWxsX3giLCAiZnVsbCIsICJ0cmFuc2Zvcm1lcl9wYXJhbV9tYXRjaGVkIiwgInRyYW5zZm9ybWVyX3giXQoKCmRlZiB0aW1lZChmbiwgd2FybXVwPTUsIGl0ZXJzPTIwKToKICAgIGZvciBfIGluIHJhbmdlKHdhcm11cCk6CiAgICAgICAgZm4oKQogICAgdCA9IFtdCiAgICBmb3IgXyBpbiByYW5nZShpdGVycyk6CiAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZm4oKQogICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgICAgIHQuYXBwZW5kKCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMCkKICAgIHJldHVybiBzdGF0aXN0aWNzLm1lZGlhbih0KQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS12YXJpYW50cyIsIG5hcmdzPSIrIiwgZGVmYXVsdD1ERUZBVUxUX1ZBUklBTlRTKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxlbmd0aHMiLCBuYXJncz0iKyIsIHR5cGU9aW50LCBkZWZhdWx0PUxFTkdUSFMpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2Uob3MuZW52aXJvbi5nZXQoIkhFRE9fREVWSUNFIiwgImN1ZGEiKSkKICAgIGJhbm5lcigiU0NBTElORyBXSVRIIFNFUVVFTkNFIExFTkdUSCAoSDQpIikKICAgIGcgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPWRldmljZSkubWFudWFsX3NlZWQoMCkKICAgIHJvd3MgPSBbXQogICAgZm9yIHZhcmlhbnQgaW4gYXJncy52YXJpYW50czoKICAgICAgICBmb3IgTCBpbiBhcmdzLmxlbmd0aHM6CiAgICAgICAgICAgIHJvdyA9IHsidmFyaWFudCI6IHZhcmlhbnQsICJpbWdfbGVuIjogTH0KICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgY2ZnID0gYnVpbGRfY29uZmlnKHZhcmlhbnQsIGltZ19sZW49TCkKICAgICAgICAgICAgICAgIG1vZGVsID0gUmV0cmlldmFsTW9kZWwoY2ZnKS50byhkZXZpY2UpCiAgICAgICAgICAgICAgICByb3dbInRyYWluYWJsZV9wYXJhbXMiXSA9IGNvdW50X3RyYWluYWJsZShtb2RlbCkKICAgICAgICAgICAgICAgIGltZzEgPSB0b3JjaC5yYW5kbigxLCBMLCA3NjgsIGRldmljZT1kZXZpY2UsIGdlbmVyYXRvcj1nKQogICAgICAgICAgICAgICAgaW1nOCA9IHRvcmNoLnJhbmRuKDgsIEwsIDc2OCwgZGV2aWNlPWRldmljZSwgZ2VuZXJhdG9yPWcpCiAgICAgICAgICAgICAgICB0eHQgPSB0b3JjaC5yYW5kbig0MCwgNjQsIDc2OCwgZGV2aWNlPWRldmljZSwgZ2VuZXJhdG9yPWcpCiAgICAgICAgICAgICAgICBtYXNrID0gdG9yY2gub25lcyg0MCwgNjQsIGR0eXBlPXRvcmNoLmJvb2wsIGRldmljZT1kZXZpY2UpCiAgICAgICAgICAgICAgICBwYWlyID0gdG9yY2guYXJhbmdlKDgsIGRldmljZT1kZXZpY2UpLnJlcGVhdF9pbnRlcmxlYXZlKDUpCgogICAgICAgICAgICAgICAgbW9kZWwuZXZhbCgpCgogICAgICAgICAgICAgICAgQHRvcmNoLm5vX2dyYWQoKQogICAgICAgICAgICAgICAgZGVmIGluZmVyKCk6CiAgICAgICAgICAgICAgICAgICAgXywgYWksIF8gPSBtb2RlbC5wYXNzMSgiaW1nIiwgaW1nMSwgTm9uZSwgc2FtcGxlPUZhbHNlKQogICAgICAgICAgICAgICAgICAgIGlmIGNmZy5leGNoYW5nZXM6CiAgICAgICAgICAgICAgICAgICAgICAgIF8sIGF0LCBfID0gbW9kZWwucGFzczEoInR4dCIsIHR4dFs6MV0sIG1hc2tbOjFdLCBzYW1wbGU9RmFsc2UpCiAgICAgICAgICAgICAgICAgICAgICAgIG1vZGVsLnBhaXJfc2ltaWxhcml0eShhaSwgYXQsIG1hc2tbOjFdKQoKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICAgICAgICAgICAgICByb3dbImluZmVyX2JzMV9tcyJdID0gdGltZWQoaW5mZXIpCiAgICAgICAgICAgICAgICByb3dbImluZmVyX3BlYWtfbWIiXSA9IHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSAvIDIqKjIwCgogICAgICAgICAgICAgICAgbW9kZWwudHJhaW4oKQogICAgICAgICAgICAgICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLCBscj0wLjApCgogICAgICAgICAgICAgICAgZGVmIHN0ZXAoKToKICAgICAgICAgICAgICAgICAgICBvcHQuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgICAgICAgICAgICAgb3V0ID0gbW9kZWwoaW1nOCwgdHh0LCBtYXNrLCBwYWlyKQogICAgICAgICAgICAgICAgICAgIHRvdGFsX2xvc3MobW9kZWwsIG91dCwgcGFpciwgMWUtNClbMF0uYmFja3dhcmQoKQogICAgICAgICAgICAgICAgICAgIG9wdC5zdGVwKCkKCiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgICAgICAgICAgICAgcm93WyJ0cmFpbl9zdGVwX21zIl0gPSB0aW1lZChzdGVwLCB3YXJtdXA9MywgaXRlcnM9MTApCiAgICAgICAgICAgICAgICByb3dbInRyYWluX3BlYWtfbWIiXSA9IHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSAvIDIqKjIwCiAgICAgICAgICAgICAgICByb3dbInN0YXR1cyJdID0gIm9rIgogICAgICAgICAgICBleGNlcHQgdG9yY2guY3VkYS5PdXRPZk1lbW9yeUVycm9yOgogICAgICAgICAgICAgICAgcm93WyJzdGF0dXMiXSA9ICJPT00iCiAgICAgICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgICAgICBtb2RlbCA9IG9wdCA9IGltZzEgPSBpbWc4ID0gdHh0ID0gTm9uZSAgIyBub3FhOiBGODQxICByZWxlYXNlIEdQVSBtZW1vcnkgYmVmb3JlIHRoZSBuZXh0IHNpemUKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQogICAgICAgICAgICByb3dzLmFwcGVuZChyb3cpCiAgICAgICAgICAgIHByaW50KHJvdywgZmx1c2g9VHJ1ZSkKCiAgICBkZiA9IHBkLkRhdGFGcmFtZShyb3dzKQogICAgc2xvcGVzID0ge30KICAgIGZvciB2YXJpYW50LCBncnAgaW4gZGZbZGZbInN0YXR1cyJdID09ICJvayJdLmdyb3VwYnkoInZhcmlhbnQiKToKICAgICAgICBncnAgPSBncnBbZ3JwWyJpbWdfbGVuIl0gPj0gNzg0XQogICAgICAgIGZvciBjb2wgaW4gKCJpbmZlcl9iczFfbXMiLCAidHJhaW5fc3RlcF9tcyIpOgogICAgICAgICAgICBpZiBsZW4oZ3JwKSA+PSAyOgogICAgICAgICAgICAgICAgc2xvcGVzW2Yie3ZhcmlhbnR9fHtjb2x9Il0gPSBmbG9hdChucC5wb2x5Zml0KG5wLmxvZyhncnBbImltZ19sZW4iXSksIG5wLmxvZyhncnBbY29sXSksIDEpWzBdKQogICAgYmFzZSA9IGRmWyhkZlsidmFyaWFudCJdID09ICJiYXNlbGluZSIpICYgKGRmWyJzdGF0dXMiXSA9PSAib2siKV0uc2V0X2luZGV4KCJpbWdfbGVuIikKICAgIGlmIG5vdCBiYXNlLmVtcHR5OgogICAgICAgIGZvciBjb2wgaW4gKCJpbmZlcl9iczFfbXMiLCAidHJhaW5fc3RlcF9tcyIsICJ0cmFpbl9wZWFrX21iIik6CiAgICAgICAgICAgIGRmW2Yie2NvbH1fdnNfYmFzZWxpbmUiXSA9IGRmLmFwcGx5KAogICAgICAgICAgICAgICAgbGFtYmRhIHI6IHJbY29sXSAvIGJhc2UubG9jW3JbImltZ19sZW4iXSwgY29sXSBpZiByWyJzdGF0dXMiXSA9PSAib2siIGFuZCByWyJpbWdfbGVuIl0gaW4gYmFzZS5pbmRleAogICAgICAgICAgICAgICAgZWxzZSBucC5uYW4sIGF4aXM9MSkKICAgIG91dCA9IFJFUE9SVF9ESVIgLyAic2NhbGluZyIKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBkZi50b19jc3Yob3V0IC8gInNjYWxpbmcuY3N2IiwgaW5kZXg9RmFsc2UpCiAgICB3cml0ZV9qc29uKG91dCAvICJzY2FsaW5nLmpzb24iLCB7InJvd3MiOiBkZi50b19kaWN0KCJyZWNvcmRzIiksICJsb2dsb2dfc2xvcGVzX0xfZ2VfNzg0Ijogc2xvcGVzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJncHUiOiB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiaW5wdXRzIjogInJhbmRvbSB0ZW5zb3JzIHdpdGggVmlUIHRva2VuIHNoYXBlIChjb21wdXRlIHNjYWxpbmcgb25seSkiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJwcm92ZW5hbmNlIjogcHJvdmVuYW5jZSgpfSkKICAgIHByaW50KCJsb2ctbG9nIHNsb3BlczoiLCB7azogcm91bmQodiwgMykgZm9yIGssIHYgaW4gc2xvcGVzLml0ZW1zKCl9KQogICAgcHJpbnQoIlNDQUxJTkc6IFBBU1MiKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"], "tests/test_cpu.py": ["fa7607e493987e3ec75257dad563958ce770bbf6188af93cb8181390f798eddf", "IiIiQ1BVIHVuaXQgdGVzdHMgKG5vIEdQVSwgbm8gbWFtYmFfc3NtKS4gUnVuOiBweXRob24gLW0gcHl0ZXN0IHRlc3RzIC1xICAob3IgcHl0aG9uIHRlc3RzL3Rlc3RfY3B1LnB5KQoKQSBzbWFsbCAqY2F1c2FsKiBzdGFuZC1pbiByZXBsYWNlcyBNYW1iYTIgaGVyZSBvbmx5OyB0cmFpbi5weSBhbmQgZ2F0ZS5weSByZWZ1c2UKYW55dGhpbmcgYnV0IHRoZSBvZmZpY2lhbCBtYW1iYV9zc20gY2xhc3MuCiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0pKQppbXBvcnQgbW9kZWxzICAjIG5vcWE6IEU0MDIKZnJvbSBtZXRyaWNzIGltcG9ydCByZXRyaWV2YWxfbWV0cmljcyAgIyBub3FhOiBFNDAyCgoKY2xhc3MgQ2F1c2FsU3RhbmRJbihubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWwsIGRfc3RhdGUsIGRfY29udiwgZXhwYW5kLCBoZWFkZGltKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnYgPSBubi5Db252MWQoZF9tb2RlbCwgZF9tb2RlbCwgZF9jb252LCBwYWRkaW5nPWRfY29udiAtIDEpCiAgICAgICAgc2VsZi5vdXQgPSBubi5MaW5lYXIoZF9tb2RlbCwgZF9tb2RlbCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBMID0geC5zaGFwZVsxXQogICAgICAgIHJldHVybiBzZWxmLm91dCh0b3JjaC50YW5oKHNlbGYuY29udih4LnRyYW5zcG9zZSgxLCAyKSlbLi4uLCA6TF0udHJhbnNwb3NlKDEsIDIpKSkgKyB4CgoKbW9kZWxzLk1BTUJBMl9GQUNUT1JZID0gQ2F1c2FsU3RhbmRJbgp0b3JjaC5tYW51YWxfc2VlZCgwKQoKCmRlZiBiYXRjaChVPTgsIGs9NSk6CiAgICBpbWcgPSB0b3JjaC5yYW5kbihVLCAxOTYsIDc2OCkKICAgIHR4dCA9IHRvcmNoLnJhbmRuKFUgKiBrLCA2NCwgNzY4KQogICAgbGVuZ3RocyA9IHRvcmNoLnJhbmRpbnQoNSwgNjQsIChVICogaywpKQogICAgbWFzayA9IHRvcmNoLmFyYW5nZSg2NClbTm9uZV0gPCBsZW5ndGhzWzosIE5vbmVdCiAgICBwYWlyID0gdG9yY2guYXJhbmdlKFUpLnJlcGVhdF9pbnRlcmxlYXZlKGspCiAgICByZXR1cm4gaW1nLCB0eHQsIG1hc2ssIHBhaXIKCgpkZWYgdGVzdF9hbGxfdmFyaWFudHNfZm9yd2FyZF9iYWNrd2FyZCgpOgogICAgaW1nLCB0eHQsIG1hc2ssIHBhaXIgPSBiYXRjaCgpCiAgICBmb3IgdiBpbiBtb2RlbHMuVkFSSUFOVFM6CiAgICAgICAgbSA9IG1vZGVscy5SZXRyaWV2YWxNb2RlbChtb2RlbHMuYnVpbGRfY29uZmlnKHYpKQogICAgICAgIG91dCA9IG0oaW1nLCB0eHQsIG1hc2ssIHBhaXIpCiAgICAgICAgbG9zcywgcGFydHMgPSBtb2RlbHMudG90YWxfbG9zcyhtLCBvdXQsIHBhaXIsIDFlLTQpCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgbWlzc2luZyA9IFtuIGZvciBuLCBwIGluIG0ubmFtZWRfcGFyYW1ldGVycygpIGlmIHAuZ3JhZCBpcyBOb25lXQogICAgICAgIGFzc2VydCB0b3JjaC5pc2Zpbml0ZShsb3NzKSBhbmQgbm90IG1pc3NpbmcsICh2LCBtaXNzaW5nKQogICAgICAgIGFzc2VydCBvdXRbInpfaW1nIl0uc2hhcGUgPT0gKDgsIDEyOCkgYW5kIG91dFsiel90eHQiXS5zaGFwZSA9PSAoNDAsIDEyOCkKICAgICAgICBpZiBtLmNmZy5leGNoYW5nZXM6CiAgICAgICAgICAgIGFzc2VydCBvdXRbInBhaXJfc2ltIl0uc2hhcGUgPT0gKDgsIDQwKSBhbmQgImluZm9uY2VfZXhjaGFuZ2UiIGluIHBhcnRzCgoKZGVmIHRlc3RfcGFyYW1fbWF0Y2hpbmcoKToKICAgIGNvdW50cyA9IHt2OiBtb2RlbHMuY291bnRfdHJhaW5hYmxlKG1vZGVscy5SZXRyaWV2YWxNb2RlbChtb2RlbHMuYnVpbGRfY29uZmlnKHYpKSkgZm9yIHYgaW4gbW9kZWxzLlZBUklBTlRTfQogICAgYXNzZXJ0IGNvdW50c1siZnVsbF9saW5lYXJfb3BlcmF0b3IiXSA9PSBjb3VudHNbImZ1bGwiXQogICAgYXNzZXJ0IGFicyhjb3VudHNbImJhc2VsaW5lX3BhcmFtX21hdGNoZWQiXSAtIGNvdW50c1siZnVsbCJdKSAvIGNvdW50c1siZnVsbCJdIDwgMC4wMSwgY291bnRzCiAgICBhc3NlcnQgYWJzKGNvdW50c1sidHJhbnNmb3JtZXJfcGFyYW1fbWF0Y2hlZCJdIC0gY291bnRzWyJiYXNlbGluZSJdKSAvIGNvdW50c1siYmFzZWxpbmUiXSA8IDAuMDEsIGNvdW50cwogICAgYXNzZXJ0IGFicyhjb3VudHNbImJhc2VsaW5lX3BhcmFtX21hdGNoZWRfeCJdIC0gY291bnRzWyJmdWxsX3giXSkgLyBjb3VudHNbImZ1bGxfeCJdIDwgMC4wMSwgY291bnRzCiAgICBoZWRvID0gbW9kZWxzLkhFRE8oMTI4KQogICAgYXNzZXJ0IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gaGVkby5wYXJhbWV0ZXJzKCkpID09IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWxzLkxpbmVhclJlc2lkdWFsU3RhY2soMTI4KS5wYXJhbWV0ZXJzKCkpCgoKZGVmIHRlc3RfaGVkb19hZmZpbmUoKToKICAgIGggPSBtb2RlbHMuSEVETygxMjgpLmRvdWJsZSgpCiAgICBhLCBiID0gdG9yY2gucmFuZG4oMiwgMywgMTI4LCBkdHlwZT10b3JjaC5mbG9hdDY0KSwgdG9yY2gucmFuZG4oMiwgMywgMTI4LCBkdHlwZT10b3JjaC5mbG9hdDY0KQogICAgYXNzZXJ0IChoKGEgKyBiKSAtIGgoYSkgLSBoKGIpICsgaCh0b3JjaC56ZXJvc19saWtlKGEpKSkuYWJzKCkubWF4KCkgPCAxZS0xMAoKCmRlZiB0ZXN0X2JvdW5kYXJpZXMoKToKICAgIGZvciBMLCB3YW50IGluIHsxOiBbMF0sIDE2OiBbMTVdLCAxNzogWzE1LCAxNl0sIDMzOiBbMTUsIDMxLCAzMl19Lml0ZW1zKCk6CiAgICAgICAgeSA9IHRvcmNoLmFyYW5nZShMLCBkdHlwZT10b3JjaC5mbG9hdDMyKS52aWV3KDEsIEwsIDEpCiAgICAgICAgcywgYyA9IG1vZGVscy5jaHVua19ib3VuZGFyaWVzKHksIHRvcmNoLm9uZXMoMSwgTCwgZHR5cGU9dG9yY2guYm9vbCksIDE2KQogICAgICAgIGFzc2VydCBzLnZpZXcoLTEpW2MudmlldygtMSkgPiAwXS5sb25nKCkudG9saXN0KCkgPT0gd2FudAoKCmRlZiB0ZXN0X3BhZGRpbmdfaW52YXJpYW5jZV9hbmRfZXZhbF9kZXRlcm1pbmlzbSgpOgogICAgbSA9IG1vZGVscy5SZXRyaWV2YWxNb2RlbChtb2RlbHMuYnVpbGRfY29uZmlnKCJmdWxsIikpLmV2YWwoKQogICAgXywgdHh0LCBtYXNrLCBfID0gYmF0Y2goKQogICAgTCA9IGludChtYXNrWzBdLnN1bSgpKQogICAgYSA9IG0uZW5jb2RlX3RleHQodHh0WzoxXSwgbWFza1s6MV0pCiAgICBiID0gbS5lbmNvZGVfdGV4dCh0eHRbOjEsIDpMXSwgbWFza1s6MSwgOkxdKQogICAgYXNzZXJ0IHRvcmNoLmFsbGNsb3NlKGEsIGIsIGF0b2w9MWUtNSkKICAgIGFzc2VydCB0b3JjaC5lcXVhbChtLmVuY29kZV90ZXh0KHR4dCwgbWFzayksIG0uZW5jb2RlX3RleHQodHh0LCBtYXNrKSkKCgpkZWYgdGVzdF9zbW9vdGhfYm91bmRzKCk6CiAgICBoID0gbW9kZWxzLkNodW5rV2lzZUhWU0MoMTI4LCA2NCkKICAgIG11LCBsdiA9IGgucG9zdGVyaW9yKCJpbWciLCB0b3JjaC5yYW5kbig0LCAzLCAxMjgpICogMWU0KQogICAgYXNzZXJ0IG11LmFicygpLm1heCgpIDw9IDEwIGFuZCBsdi5taW4oKSA+PSAtMS41IGFuZCBsdi5tYXgoKSA8PSAxLjUKICAgIHMgPSB0b3JjaC5yYW5kbigyLCAzLCAxMjgpCiAgICBtdSwgbHYgPSBoLnBvc3RlcmlvcigiaW1nIiwgcykKICAgIGFzc2VydCB0b3JjaC5hbGxjbG9zZShsdiwgdG9yY2guZnVsbF9saWtlKGx2LCAtMS4wKSwgYXRvbD0xZS01KQogICAga2wgPSBoLnN5bW1ldHJpY19rbChtdSwgbHYsIHRvcmNoLm9uZXMoMiwgMyksIG11LCBsdiwgdG9yY2gub25lcygyLCAzKSkKICAgIGFzc2VydCBhYnMoZmxvYXQoa2wpKSA8IDFlLTYKCgpkZWYgdGVzdF9zeW1tZXRyaWNfa2xfbWF0Y2hlc190ZXh0Ym9va19mb3JtdWxhKCk6CiAgICB0b3JjaC5tYW51YWxfc2VlZCgzKQogICAgbXVfaSwgbXVfdCA9IHRvcmNoLnJhbmRuKDMsIDQsIDY0LCBkdHlwZT10b3JjaC5mbG9hdDY0KSAqIDMsIHRvcmNoLnJhbmRuKDMsIDQsIDY0LCBkdHlwZT10b3JjaC5mbG9hdDY0KSAqIDMKICAgIGx2X2ksIGx2X3QgPSB0b3JjaC5yYW5kKDMsIDQsIDY0LCBkdHlwZT10b3JjaC5mbG9hdDY0KSAqIDMgLSAxLjUsIHRvcmNoLnJhbmQoMywgNCwgNjQsIGR0eXBlPXRvcmNoLmZsb2F0NjQpICogMyAtIDEuNQogICAgbSA9IHRvcmNoLm9uZXMoMywgNCwgZHR5cGU9dG9yY2guZmxvYXQ2NCkKICAgIHZpLCB2dCwgZDIgPSBsdl9pLmV4cCgpLCBsdl90LmV4cCgpLCAobXVfaSAtIG11X3QpICoqIDIKICAgIHJlZiA9IDAuNSAqICgwLjUgKiAobHZfdCAtIGx2X2kgKyAodmkgKyBkMikgLyB2dCAtIDEpICsgMC41ICogKGx2X2kgLSBsdl90ICsgKHZ0ICsgZDIpIC8gdmkgLSAxKSkubWVhbigtMSkubWVhbigpCiAgICBnb3QgPSBtb2RlbHMuQ2h1bmtXaXNlSFZTQy5zeW1tZXRyaWNfa2wobXVfaSwgbHZfaSwgbSwgbXVfdCwgbHZfdCwgbSkKICAgIGFzc2VydCB0b3JjaC5hbGxjbG9zZShnb3QuZG91YmxlKCksIHJlZiwgcnRvbD0xZS01KQoKCmRlZiB0ZXN0X2luZm9uY2VfY2hhbmNlX2FuZF9hbGlnbm1lbnQoKToKICAgIF8sIF8sIF8sIHBhaXIgPSBiYXRjaCgpCiAgICB6aSA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwubm9ybWFsaXplKHRvcmNoLnJhbmRuKDgsIDEyOCksIGRpbT0tMSkKICAgIHp0ID0gdG9yY2gubm4uZnVuY3Rpb25hbC5ub3JtYWxpemUodG9yY2gucmFuZG4oNDAsIDEyOCksIGRpbT0tMSkKICAgIGNoYW5jZSA9IDAuNSAqIChucC5sb2coNDApICsgbnAubG9nKDgpKQogICAgYXNzZXJ0IGFicyhmbG9hdChtb2RlbHMubXVsdGlwb3NpdGl2ZV9pbmZvbmNlKHppLCB6dCwgcGFpciwgdG9yY2gudGVuc29yKDFlLTYpKSkgLSBjaGFuY2UpIDwgMWUtMwogICAgIyA1IGVxdWFsIHBvc2l0aXZlcyBwZXIgaW1hZ2U6IGkydCBmbG9vciBpcyBsb2cgNSwgdDJpIGZsb29yIDAgLT4gMC41ICogbG9nIDUKICAgIGFzc2VydCBhYnMoZmxvYXQobW9kZWxzLm11bHRpcG9zaXRpdmVfaW5mb25jZSh6aSwgemlbcGFpcl0sIHBhaXIsIHRvcmNoLnRlbnNvcigxMDAuMCkpKSAtIDAuNSAqIG5wLmxvZyg1KSkgPCAxZS0yCgoKZGVmIHRlc3RfbWV0cmljc19hbmRfaW5kZXBlbmRlbnRfZXZhbHVhdG9yKCk6CiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMCkKICAgIGltZyA9IHJuZy5ub3JtYWwoc2l6ZT0oMTAwMCwgMTI4KSkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICB0eHQgPSAobnAucmVwZWF0KGltZywgNSwgYXhpcz0wKSArIHJuZy5ub3JtYWwoc2NhbGU9My4wLCBzaXplPSg1MDAwLCAxMjgpKSkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBpbWcgLz0gbnAubGluYWxnLm5vcm0oaW1nLCBheGlzPTEsIGtlZXBkaW1zPVRydWUpCiAgICB0eHQgLz0gbnAubGluYWxnLm5vcm0odHh0LCBheGlzPTEsIGtlZXBkaW1zPVRydWUpCiAgICBtID0gcmV0cmlldmFsX21ldHJpY3MoaW1nLCB0eHQpCiAgICBhc3NlcnQgMCA8IG1bIm1lYW5fcmVjYWxsIl0gPCAxMDAKICAgIHJhbmQgPSByZXRyaWV2YWxfbWV0cmljcyhpbWcsIHJuZy5wZXJtdXRhdGlvbih0eHQpKQogICAgYXNzZXJ0IHJhbmRbIm1lYW5fcmVjYWxsIl0gPCAyLjAKCiAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcDoKICAgICAgICB0bXAgPSBQYXRoKHRtcCkKICAgICAgICBkYXRhID0gdG1wIC8gImRhdGEiIC8gImZsaWNrcjhrIgogICAgICAgIGRhdGEubWtkaXIocGFyZW50cz1UcnVlKQogICAgICAgIGlkcyA9IFtmImltZ3tpOjA0ZH0uanBnIiBmb3IgaSBpbiByYW5nZSgxMDAwKV0KICAgICAgICB3aXRoIG9wZW4oZGF0YSAvICJjYXB0aW9uc190ZXN0LmNzdiIsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgZi53cml0ZSgiaW1hZ2VfaWQsY2FwX2lkeCxjYXB0aW9uXG4iKQogICAgICAgICAgICBmb3IgaSBpbiBpZHM6CiAgICAgICAgICAgICAgICBmb3IgayBpbiByYW5nZSg1KToKICAgICAgICAgICAgICAgICAgICBmLndyaXRlKGYie2l9LHtrfSxhIGNhcHRpb25cbiIpCiAgICAgICAgcnVuID0gdG1wIC8gInJ1biIKICAgICAgICBydW4ubWtkaXIoKQogICAgICAgIG5wLnNhdmUocnVuIC8gInRlc3RfaW1hZ2VfZW1iZWRkaW5ncy5ucHkiLCBpbWcpCiAgICAgICAgbnAuc2F2ZShydW4gLyAidGVzdF90ZXh0X2VtYmVkZGluZ3MubnB5IiwgdHh0KQogICAgICAgIGpzb24uZHVtcChpZHMsIG9wZW4ocnVuIC8gInRlc3RfaW1hZ2VfaWRzLmpzb24iLCAidyIpKQogICAgICAgIGpzb24uZHVtcChtLCBvcGVuKHJ1biAvICJ0ZXN0X3Jlc3VsdHMuanNvbiIsICJ3IikpCiAgICAgICAgc2NyaXB0ID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0gLyAiZXZhbHVhdGVfaW5kZXBlbmRlbnQucHkiCiAgICAgICAgb2sgPSBzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsIHN0cihzY3JpcHQpLCAiLS1ydW4iLCBzdHIocnVuKSwgIi0tZGF0YS1kaXIiLCBzdHIoZGF0YSldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKQogICAgICAgIGFzc2VydCBvay5yZXR1cm5jb2RlID09IDAsIG9rLnN0ZG91dCArIG9rLnN0ZGVycgogICAgICAgIGJhZCA9IGRpY3QobSwgbWVhbl9yZWNhbGw9bVsibWVhbl9yZWNhbGwiXSArIDAuMDEpCiAgICAgICAganNvbi5kdW1wKGJhZCwgb3BlbihydW4gLyAidGVzdF9yZXN1bHRzLmpzb24iLCAidyIpKQogICAgICAgIGZhaWwgPSBzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsIHN0cihzY3JpcHQpLCAiLS1ydW4iLCBzdHIocnVuKSwgIi0tZGF0YS1kaXIiLCBzdHIoZGF0YSldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpCiAgICAgICAgYXNzZXJ0IGZhaWwucmV0dXJuY29kZSA9PSAxCgoKZGVmIHRlc3RfZW5lcmd5X3RoZW9yZW1fYWR2ZXJzYXJpYWwoKToKICAgIHRvcmNoLm1hbnVhbF9zZWVkKDApCiAgICB4ID0gdG9yY2gucmFuZG4oNCwgNTAsIDEyOCwgZHR5cGU9dG9yY2guZmxvYXQ2NCkgKiAzCiAgICBmb3Igc2NhbGVfdSwgb21lZ2EsIHRoZXRhLCBkYW1wIGluICgoMSwgTm9uZSwgTm9uZSwgTm9uZSksICgzMCwgNS4wLCA1LjAsIC01LjApLCAoMywgMi4wLCA1LjAsIC0yMC4wKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICgxMCwgMS4wLCAwLjAsIDUuMCksICgxMDAsIDguMCwgMTAuMCwgLTMwLjApKToKICAgICAgICBvcCA9IG1vZGVscy5FbmVyZ3lIRURPKDEyOCwgNjQsIHN0ZXBzPTEwKS5kb3VibGUoKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBvcC5VLm11bF8oc2NhbGVfdSkKICAgICAgICAgICAgb3AuYi5ub3JtYWxfKCkKICAgICAgICAgICAgb3AucF9wcm9qLndlaWdodC5ub3JtYWxfKHN0ZD0xLjApCiAgICAgICAgICAgIGlmIG9tZWdhIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgb3Aub21lZ2EuZmlsbF8ob21lZ2EpCiAgICAgICAgICAgICAgICBvcC50aGV0YS5maWxsXyh0aGV0YSkKICAgICAgICAgICAgICAgIG9wLmRhbXBpbmcuYmlhcy5maWxsXyhkYW1wKQogICAgICAgIHQgPSBvcC50cmFqZWN0b3J5KHgpCiAgICAgICAgZSA9IHRbImVuZXJnaWVzIl0KICAgICAgICBhc3NlcnQgKChlWy4uLiwgMTpdIC0gZVsuLi4sIDotMV0pIC8gZVsuLi4sIDotMV0uYWJzKCkuY2xhbXAobWluPTEuMCkpLm1heCgpIDw9IDFlLTEwCiAgICAgICAgYXNzZXJ0IHRbImF0dGVudWF0aW9uIl0ubWF4KCkgPD0gMS4wIGFuZCB0WyJhdHRlbnVhdGlvbiJdLm1pbigpID4gMAogICAgICAgICMgdGhlIGJvdW5kIG9uIHRoZSBIZXNzaWFuIHJlYWxseSBob2xkczogY29tcGFyZSB3aXRoIHRoZSBleGFjdCBIZXNzaWFuIGF0IGEgcmFuZG9tIHBvaW50CiAgICAgICAgcSA9IHRvcmNoLnJhbmRuKDEyOCwgZHR5cGU9dG9yY2guZmxvYXQ2NCkKICAgICAgICBIID0gdG9yY2guYXV0b2dyYWQuZnVuY3Rpb25hbC5oZXNzaWFuKGxhbWJkYSB2OiBvcC5wb3RlbnRpYWwodiksIHEpCiAgICAgICAgYXNzZXJ0IHRvcmNoLmxpbmFsZy5laWd2YWxzaChIKS5tYXgoKSA8PSBvcC5zbW9vdGhuZXNzX2JvdW5kKCkgKiAoMSArIDFlLTkpCiAgICAgICAgYXNzZXJ0IHRvcmNoLmFsbGNsb3NlKHRvcmNoLmF1dG9ncmFkLmZ1bmN0aW9uYWwuamFjb2JpYW4ob3AucG90ZW50aWFsLCBxKSwgb3AuZ3JhZF9wb3RlbnRpYWwocSkpCgoKZGVmIHRlc3RfZXhjaGFuZ2Vfbm9vcF9hdF9pbml0X2FuZF9lZmZlY3Rfd2hlbl9nYXRlZCgpOgogICAgbSA9IG1vZGVscy5SZXRyaWV2YWxNb2RlbChtb2RlbHMuYnVpbGRfY29uZmlnKCJmdWxsX3giKSkuZXZhbCgpCiAgICBpbWcsIHR4dCwgbWFzaywgXyA9IGJhdGNoKDQsIDEpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICB6aSwgYWksIF8gPSBtLnBhc3MxKCJpbWciLCBpbWcsIE5vbmUsIHNhbXBsZT1GYWxzZSkKICAgICAgICB6dCwgYXQsIF8gPSBtLnBhc3MxKCJ0eHQiLCB0eHQsIG1hc2ssIHNhbXBsZT1GYWxzZSkKICAgICAgICBhc3NlcnQgdG9yY2guYWxsY2xvc2UobS5wYWlyX3NpbWlsYXJpdHkoYWksIGF0LCBtYXNrKSwgKHppICogenQpLnN1bSgtMSksIGF0b2w9MWUtNikKICAgICAgICBmb3IgayBpbiAoImltZyIsICJ0eHQiKToKICAgICAgICAgICAgbS54aHZzYy5nYXRlW2tdLmZpbGxfKDEuMCkKICAgICAgICBzX29uID0gbS5wYWlyX3NpbWlsYXJpdHkoYWksIGF0LCBtYXNrKQogICAgICAgIHNfb2ZmID0gbS5wYWlyX3NpbWlsYXJpdHkoYWksIGF0LCBtYXNrLCBtZXNzYWdlX3NjYWxlPSgwLjAsIDAuMCkpCiAgICAgICAgYXNzZXJ0IHRvcmNoLmFsbGNsb3NlKHNfb2ZmLCAoemkgKiB6dCkuc3VtKC0xKSwgYXRvbD0xZS02KQogICAgICAgIGFzc2VydCAoc19vbiAtIHNfb2ZmKS5hYnMoKS5tYXgoKSA+IDFlLTQKCgpkZWYgdGVzdF9yZXJhbmtfbWV0cmljc19hbmRfaW5kZXBlbmRlbnRfZXZhbHVhdG9yKCk6CiAgICBpbXBvcnQgbWV0cmljcwogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDEpCiAgICBpbWcgPSBybmcubm9ybWFsKHNpemU9KDIwMCwgMzIpKTsgaW1nIC89IG5wLmxpbmFsZy5ub3JtKGltZywgYXhpcz0xLCBrZWVwZGltcz1UcnVlKQogICAgdHh0ID0gbnAucmVwZWF0KGltZywgNSwgMCkgKyBybmcubm9ybWFsKHNjYWxlPTIuMCwgc2l6ZT0oMTAwMCwgMzIpKTsgdHh0IC89IG5wLmxpbmFsZy5ub3JtKHR4dCwgYXhpcz0xLCBrZWVwZGltcz1UcnVlKQogICAgaTJ0X2lkeCwgdDJpX2lkeCA9IG1ldHJpY3MudG9wa19jYW5kaWRhdGVzKGltZywgdHh0LCA4KQogICAgb3duZXIgPSBtZXRyaWNzLmNhcHRpb25fb3duZXIoMjAwKQogICAgIyBvcmFjbGUgZXhjaGFuZ2Ugc2NvcmU6IHBvc2l0aXZlcyBzY29yZSBoaWdoZXN0IC0+IGFueSBwb3NpdGl2ZSBpbnNpZGUgdGhlIHRvcC1LIG1vdmVzIHRvIHJhbmsgMAogICAgcnIgPSB7ImkydF9pZHgiOiBpMnRfaWR4LCAiaTJ0X3Njb3JlIjogKG93bmVyW2kydF9pZHhdID09IG5wLmFyYW5nZSgyMDApWzosIE5vbmVdKSArIHJuZy5yYW5kb20oaTJ0X2lkeC5zaGFwZSkgKiAwLjEsCiAgICAgICAgICAidDJpX2lkeCI6IHQyaV9pZHgsICJ0Mmlfc2NvcmUiOiAodDJpX2lkeCA9PSBvd25lcls6LCBOb25lXSkgKyBybmcucmFuZG9tKHQyaV9pZHguc2hhcGUpICogMC4xfQogICAgbSA9IG1ldHJpY3MucmV0cmlldmFsX21ldHJpY3MoaW1nLCB0eHQsIHJyKQogICAgZHVhbF9pMnQsIF8gPSBtZXRyaWNzLnJldHJpZXZhbF9yYW5rcyhpbWcsIHR4dCkKICAgIGFzc2VydCBtWyJpMnRfcjEiXSA9PSBucC5tZWFuKGR1YWxfaTJ0IDwgOCkgKiAxMDAKICAgIGFzc2VydCBtWyJtZWFuX3JlY2FsbCJdID49IG1bImR1YWxfbWVhbl9yZWNhbGwiXQogICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXA6CiAgICAgICAgdG1wID0gUGF0aCh0bXApCiAgICAgICAgZGF0YSA9IHRtcCAvICJkYXRhIgogICAgICAgIGRhdGEubWtkaXIoKQogICAgICAgIGlkcyA9IFtmIml7aX0uanBnIiBmb3IgaSBpbiByYW5nZSgyMDApXQogICAgICAgIHdpdGggb3BlbihkYXRhIC8gImNhcHRpb25zX3Rlc3QuY3N2IiwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBmLndyaXRlKCJpbWFnZV9pZCxjYXBfaWR4LGNhcHRpb25cbiIgKyAiIi5qb2luKGYie2l9LHtrfSxjXG4iIGZvciBpIGluIGlkcyBmb3IgayBpbiByYW5nZSg1KSkpCiAgICAgICAgcnVuID0gdG1wIC8gInJ1biIKICAgICAgICBydW4ubWtkaXIoKQogICAgICAgIG5wLnNhdmUocnVuIC8gInRlc3RfaW1hZ2VfZW1iZWRkaW5ncy5ucHkiLCBpbWcuYXN0eXBlKG5wLmZsb2F0MzIpKQogICAgICAgIG5wLnNhdmUocnVuIC8gInRlc3RfdGV4dF9lbWJlZGRpbmdzLm5weSIsIHR4dC5hc3R5cGUobnAuZmxvYXQzMikpCiAgICAgICAgbnAuc2F2ZXoocnVuIC8gInRlc3RfcmVyYW5rLm5weiIsICoqcnIpCiAgICAgICAganNvbi5kdW1wKGlkcywgb3BlbihydW4gLyAidGVzdF9pbWFnZV9pZHMuanNvbiIsICJ3IikpCiAgICAgICAgc3RvcmVkID0gbWV0cmljcy5yZXRyaWV2YWxfbWV0cmljcyhpbWcuYXN0eXBlKG5wLmZsb2F0MzIpLCB0eHQuYXN0eXBlKG5wLmZsb2F0MzIpLCBycikKICAgICAgICBqc29uLmR1bXAoc3RvcmVkLCBvcGVuKHJ1biAvICJ0ZXN0X3Jlc3VsdHMuanNvbiIsICJ3IikpCiAgICAgICAgc2NyaXB0ID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0gLyAiZXZhbHVhdGVfaW5kZXBlbmRlbnQucHkiCiAgICAgICAgb2sgPSBzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsIHN0cihzY3JpcHQpLCAiLS1ydW4iLCBzdHIocnVuKSwgIi0tZGF0YS1kaXIiLCBzdHIoZGF0YSldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKQogICAgICAgIGFzc2VydCBvay5yZXR1cm5jb2RlID09IDAsIG9rLnN0ZG91dCArIG9rLnN0ZGVycgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBmb3IgbmFtZSwgZm4gaW4gbGlzdChnbG9iYWxzKCkuaXRlbXMoKSk6CiAgICAgICAgaWYgbmFtZS5zdGFydHN3aXRoKCJ0ZXN0XyIpOgogICAgICAgICAgICBmbigpCiAgICAgICAgICAgIHByaW50KCJvayIsIG5hbWUpCg=="], "train.py": ["..."]}
if PKG.exists():
    shutil.rmtree(PKG)
for rel, (digest, b64) in EMBEDDED.items():
    data = base64.b64decode(b64)
    if hashlib.sha256(data).hexdigest() != digest:
        raise RuntimeError(f"embedded file corrupted: {rel}")
    path = PKG / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
got = hashlib.sha256(json.dumps({k: v[0] for k, v in EMBEDDED.items()}, sort_keys=True).encode()).hexdigest()
assert got == BUNDLE_SHA256, "bundle manifest mismatch"
print(f"wrote {len(EMBEDDED)} files to {PKG} | bundle sha256 {BUNDLE_SHA256}")

run([NATIVE_PYTHON, "-m", "pytest", "-q", "tests/test_cpu.py"], "unit_tests", cwd=PKG)
run([NATIVE_PYTHON, "-c", "from common import code_sha256; print('code_sha256', code_sha256())"], "code_hash", cwd=PKG)

In [ ]:
# CELL 3: FLICKR8K (download, SHA256, official splits, leakage report)
# First run records archive checksums in manifest.json. Save {"Flickr8k_Dataset.zip": sha, "Flickr8k_text.zip": sha}
# into PKG / "flickr8k_checksums.json" (or WORK) before re-running to enforce them.
CHECKSUMS = WORK / "flickr8k_checksums.json"
args = ["--expected-checksums", CHECKSUMS] if CHECKSUMS.is_file() else []
run([NATIVE_PYTHON, "data.py", *args], "data", cwd=PKG)
m = json.load(open(WORK / "data/flickr8k/manifest.json"))
print(json.dumps({k: m[k] for k in ("archive_sha256", "counts", "leakage")}, indent=2))

In [ ]:
# CELL 4: FROZEN BACKBONE FEATURE CACHE (about 6 GB; rebuilt automatically if feature code changed)
import shutil
# Official ViT-B/16 IMAGENET1K_V1 transforms, FP32 compute, FP16 storage with overflow check.
# RoBERTa commit hash and loading report are saved. Set HEDO_ROBERTA_REVISION to that hash for the paper runs.
code_now = subprocess.run([NATIVE_PYTHON, "-c", "from common import feature_code_sha256; print(feature_code_sha256())"],
                          cwd=PKG, capture_output=True, text=True, check=True).stdout.strip()
fm_path = WORK / "features/feature_manifest.json"
cached = json.load(open(fm_path)).get("feature_code_sha256") if fm_path.is_file() else None
if cached != code_now or not (WORK / "features/saliency_test.npy").is_file():
    print(f"feature cache {'missing' if cached is None else 'stale'} -> rebuilding (feature code {code_now[:12]})")
    shutil.rmtree(WORK / "features", ignore_errors=True)
    run([NATIVE_PYTHON, "features.py", "--batch-size", "64"], "features", cwd=PKG)
else:
    print(f"feature cache current (feature code {code_now[:12]})")
fm = json.load(open(WORK / "features/feature_manifest.json"))
print("RoBERTa:", fm["roberta"], "| absmax:", fm["absmax"])
print(open(WORK / "features/roberta_loading_info.json").read()[:800])

In [ ]:
# CELL 5: PRE-TRAINING GATE ON REAL DATA
run([NATIVE_PYTHON, "gate.py"], "gate", cwd=PKG)
print(open(WORK / "report/param_counts.json").read())

In [ ]:
# CELL 6: LEARNING SANITY (decision point before any matrix run)
# 2 epochs, per-batch JSONL, anomaly detection in epoch 1, exit 3 if epoch-1 val MR < 2.0 (chance ~0.53).
# Compare the proposed model with the no-mixer control. If ours is not clearly above chance, stop and debug.
SANITY = WORK / "sanity"
for variant in ("no_mixer_meanpool", "full_x"):
    out = SANITY / f"{variant}_seed42"
    run([NATIVE_PYTHON, "-u", "train.py", "--variant", variant, "--seed", 42, "--epochs", 2, "--detect-anomaly",
         "--out", out, "--overwrite"], f"sanity_{variant}", cwd=PKG, check=False)
    print(variant, json.load(open(out / "run_summary.json"))["status"])
    if (out / "training_history.csv").is_file():
        h = pd.read_csv(out / "training_history.csv")
        keep = ("epoch", "train_loss", "train_infonce", "train_infonce_exchange", "train_prior_kl",
                "val_mean_recall", "val_dual_mean_recall")
        print(h[[c for c in h.columns if c in keep]].to_string(index=False))
log = pd.read_json(SANITY / "full_x_seed42" / "batch_log.jsonl", lines=True)
print(log[[c for c in log.columns if "energy" in c or "gate" in c or c.startswith("grad_")]].describe().T.to_string())

# HARD STOP: the matrix costs many GPU hours; never start it on a model that does not train.
ALLOW_MATRIX_WITHOUT_SANITY = False
summary = json.load(open(SANITY / "full_x_seed42" / "run_summary.json"))
hist_path = SANITY / "full_x_seed42" / "training_history.csv"
val_mr = float(pd.read_csv(hist_path)["val_mean_recall"].max()) if hist_path.is_file() else float("nan")
sanity_ok = summary["status"] == "COMPLETED" and val_mr >= 5.0
print(f"SANITY: status={summary['status']} best val MR={val_mr:.3f} (chance ~0.53, required >= 5.0) ->",
      "PASS" if sanity_ok else "FAIL")
if not sanity_ok and not ALLOW_MATRIX_WITHOUT_SANITY:
    raise RuntimeError("Sanity run failed. Inspect WORK/logs/sanity_full_x.log and "
                       "WORK/sanity/full_x_seed42/failure_state.pt before running cells 7+.")

In [ ]:
# CELL 7: PROPOSED 2x2 FACTORIAL (baseline, +E-HEDO, +X-HVSC, +both) x CORE_SEEDS
# Safe to re-run after a disconnect: verified current runs are skipped, anything else restarts.
run([NATIVE_PYTHON, "-u", "run_matrix.py", "--suite", "proposal", "--epochs", EPOCHS, "--seeds", *CORE_SEEDS],
    "matrix_proposal", cwd=PKG, check=False)
print(pd.read_csv(WORK / "results/all_runs.csv")[["run", "status", "mean_recall", "dual_mean_recall"]].to_string(index=False))

In [ ]:
# CELL 8: ABLATIONS AND CONTROLS (identical protocol)
#   full_x_affine             E-HEDO replaced by the earlier affine HEDO
#   full_x_constdamp          damping not input-dependent (tests the background-selective mechanism)
#   full_x_noexchange         bottleneck only, no cross-modal messages (H2 reference)
#   full_x_noprior            beta = 0 (no information bottleneck)
#   baseline_param_matched_x  Mamba-2 with a wider head, #params matched to ours
#   transformer_param_matched Transformer mixer, #params matched to baseline
#   transformer_x             our modules on a Transformer mixer
#   no_mixer_meanpool         frozen features + mean pool
#   proposal_chunks           chunk size 8 and 32
RUN_LEGACY = False   # True also reruns the earlier affine-HEDO / index-KL study
suites = ["proposal_ablations", "proposal_chunks"] + (["core", "ablations", "klsweep", "chunks"] if RUN_LEGACY else [])
run([NATIVE_PYTHON, "-u", "run_matrix.py", "--suite", *suites, "--epochs", EPOCHS, "--seeds", *ABLATION_SEEDS],
    "matrix_ablations", cwd=PKG, check=False)
df = pd.read_csv(WORK / "results/all_runs.csv")
print(df.groupby(["variant", "hvsc_chunk_size", "status"]).size().to_string())

In [ ]:
# CELL 9: H1 / H2 MECHANISM PROBES (energy vs saliency, occlusion, gradient balance, message reliance)
run([NATIVE_PYTHON, "-u", "probe.py"], "probes", cwd=PKG)
pr = pd.read_csv(WORK / "results/probes.csv")
cols = ["attenuation_bg", "attenuation_fg", "attenuation_spearman_saliency", "energy_max_increase_float64",
        "occlude_bg_dual_mean_recall", "occlude_fg_dual_mean_recall", "grad_abs_log10_ratio", "dominance_index"]
print(pr.groupby("variant")[[c for c in cols if c in pr]].mean().round(4).to_string())

In [ ]:
# CELL 10: H3 CORRUPTION BENCHMARK (image noise/blur/JPEG/occlusion; caption dropout/typos/shuffle)
if not (WORK / "features/robust/robust_manifest.json").is_file():
    run([NATIVE_PYTHON, "-u", "robustness.py", "build"], "robust_build", cwd=PKG)
run([NATIVE_PYTHON, "-u", "robustness.py", "evaluate"], "robust_eval", cwd=PKG)
rb = pd.read_csv(WORK / "results/robustness.csv")
print(rb.groupby(["variant", "corruption"])["relative_mr"].mean().unstack().round(3).to_string())

In [ ]:
# CELL 11: H4 SCALING WITH SEQUENCE LENGTH + END-TO-END EFFICIENCY (real inputs, backbones included)
run([NATIVE_PYTHON, "-u", "scaling.py"], "scaling", cwd=PKG)
print(pd.read_csv(WORK / "report/scaling/scaling.csv").to_string(index=False))
run([NATIVE_PYTHON, "-u", "profile_efficiency.py", "--batch-sizes", 1, 32], "efficiency", cwd=PKG)
print(pd.read_csv(WORK / "report/efficiency/efficiency_profile.csv").T.to_string())

In [ ]:
# CELL 12: TABLES, STATISTICS, FIGURES, HYPOTHESIS VERDICTS (only verified, current-code runs)
from IPython.display import Image, display
run([NATIVE_PYTHON, "-u", "analyze.py", "--bootstrap", 2000], "analyze", cwd=PKG)
run([NATIVE_PYTHON, "-u", "hypotheses.py"], "hypotheses", cwd=PKG)
for name in ("table_main.tex", "table_ablations.tex", "table_stats.tex", "table_hypotheses.tex"):
    print(f"--- {name}\n" + (WORK / "report" / name).read_text())
for fig in ("fig_mean_recall.png", "fig_training_curves.png"):
    display(Image(str(WORK / "report" / fig)))

In [ ]:
# CELL 13: BUNDLE (small artifacts + SHA256SUMS; checkpoints and features excluded by default)
import hashlib, zipfile
INCLUDE_CHECKPOINTS = False
bundle = WORK / "hedo_hvsc_bundle.zip"
files = [p for p in WORK.rglob("*") if p.is_file() and p != bundle and "features" not in p.parts
         and "extracted" not in p.parts and p.suffix not in {".zip", ".part"}
         and (INCLUDE_CHECKPOINTS or p.suffix != ".pt")]
sums = "".join(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(WORK)}\n" for p in sorted(files))
(WORK / "SHA256SUMS").write_text(sums)
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(files) + [WORK / "SHA256SUMS"]:
        z.write(p, p.relative_to(WORK))
print(bundle, f"{bundle.stat().st_size / 2**20:.1f} MB", len(files), "files")